# CALM-AH Classification Pipeline


# 1. Environment and shared configuration


In [ ]:
from pathlib import Path
import os
import sys
import random
import warnings

# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------
RANDOM_SEED = int(os.getenv("RANDOM_SEED", "42"))
random.seed(RANDOM_SEED)

# ---------------------------------------------------------------------
# Portable project configuration
# Override these values through environment variables where appropriate.
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", "/workspace/project")
).expanduser().resolve()

OUTPUT_ROOT = Path(
    os.getenv("OUTPUT_ROOT", "/workspace/outputs")
).expanduser().resolve()

HF_CACHE_ROOT = Path(
    os.getenv(
        "HF_CACHE_ROOT",
        "/workspace/cache/huggingface/hub",
    )
).expanduser().resolve()

FEATURE_PYTHON = Path(
    os.getenv(
        "FEATURE_PYTHON",
        "/opt/conda/envs/CALMAH/bin/python",
    )
).expanduser()

FEATURE_GPU = os.getenv("FEATURE_GPU", "0")

# Keras must be configured before importing TensorFlow/Keras.
os.environ.pop("TF_USE_LEGACY_KERAS", None)
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ.setdefault("CUDA_VISIBLE_DEVICES", FEATURE_GPU)
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Feature Python:", FEATURE_PYTHON)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

In [ ]:
import itertools
import json
import math
import os
import pickle
import re
import subprocess
import sys
from pathlib import Path

import joblib
import librosa
import numpy as np
from tqdm.auto import tqdm

from sklearn.metrics import f1_score, log_loss

from bah_metrics import bah_perfs


def zscore(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    return (values - values.mean()) / (values.std() + 1e-12)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / (denominator + 1e-12))


def softmax(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    shifted = values - np.max(values)
    exp_values = np.exp(shifted)
    return exp_values / (exp_values.sum() + 1e-12)


def cosine_to_unit(values: np.ndarray) -> np.ndarray:
    return (np.asarray(values) + 1.0) / 2.0


def max_abs_norm(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    return values / (np.max(np.abs(values)) + 1e-12)


def linalg_array(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    return values / (
        np.linalg.norm(values, axis=1, keepdims=True) + 1e-12
    )


def linalg_vec(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    return values / (np.linalg.norm(values) + 1e-12)


def stats_pool(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    return np.array(
        [
            np.nanmin(values),
            np.nanmax(values),
            np.nanmean(values),
            np.nanstd(values),
        ]
    )


def clean_text(text: str) -> str:
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def get_best_f1_threshold(
    probabilities: np.ndarray,
    labels: np.ndarray,
    step: float = 0.01,
) -> tuple[float, float]:
    best_threshold = 0.5
    best_score = -np.inf

    for threshold in np.arange(step, 1.0, step):
        predictions = (
            np.asarray(probabilities) >= threshold
        ).astype(int)
        score = f1_score(
            labels,
            predictions,
            average="macro",
        )
        if score > best_score:
            best_threshold = float(threshold)
            best_score = float(score)

    return best_threshold, best_score

In [ ]:
import keras
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

physical_gpus = tf.config.list_physical_devices("GPU")
print("Visible GPUs:", physical_gpus)

for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        # Memory growth cannot be changed after GPU initialization.
        pass

if physical_gpus:
    with tf.device("/GPU:0"):
        left = tf.random.normal((512, 512))
        right = tf.random.normal((512, 512))
        result = tf.matmul(left, right)
    print("TensorFlow test device:", result.device)
else:
    print("No TensorFlow GPU detected; CPU execution remains available.")

In [ ]:
# Discover locally cached Hugging Face snapshots without embedding a
# user-specific home directory in the notebook.

CACHE_ROOT_CANDIDATES = [
    HF_CACHE_ROOT,
    Path.home() / ".cache" / "huggingface" / "hub",
    PROJECT_ROOT / ".cache" / "huggingface" / "hub",
]

REPO_CACHE_NAMES = {
    "siglip2": "models--google--siglip2-base-patch16-naflex",
    "hubert": "models--facebook--hubert-large-ll60k",
    "f2llm": "models--codefuse-ai--F2LLM-1.7B",
}


def find_complete_snapshots(repo_cache_name: str) -> list[dict]:
    candidates = []

    for cache_root in CACHE_ROOT_CANDIDATES:
        snapshot_root = cache_root / repo_cache_name / "snapshots"
        if not snapshot_root.exists():
            continue

        for snapshot in sorted(snapshot_root.iterdir()):
            if not snapshot.is_dir():
                continue

            files = [path for path in snapshot.rglob("*") if path.is_file()]
            if not files:
                continue

            candidates.append(
                {
                    "snapshot": snapshot.resolve(),
                    "file_count": len(files),
                    "total_bytes": sum(
                        path.stat().st_size for path in files
                    ),
                    "has_config": (snapshot / "config.json").exists(),
                }
            )

    return candidates


LOCAL_MODEL_PATHS = {}

for modality, repo_cache_name in REPO_CACHE_NAMES.items():
    snapshots = find_complete_snapshots(repo_cache_name)
    snapshots.sort(
        key=lambda item: (
            item["has_config"],
            item["file_count"],
            item["total_bytes"],
        ),
        reverse=True,
    )
    LOCAL_MODEL_PATHS[modality] = (
        snapshots[0]["snapshot"] if snapshots else None
    )

print("Selected local model snapshots:")
for modality, model_path in LOCAL_MODEL_PATHS.items():
    print(f"{modality:8s}: {model_path}")

# 2. Optional foundation-model feature extraction


In [ ]:
from transformers import AutoProcessor, AutoModel
from transformers import HubertModel, Wav2Vec2FeatureExtractor
from sentence_transformers import SentenceTransformer
from PIL import Image
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class SigLip2():
    def __init__(self):
        self.ckpt = "google/siglip2-base-patch16-naflex"
        try:
            self.model = SIGLIP2_MODEL.model
            self.processor = SIGLIP2_MODEL.processor
        except:
            self.model = AutoModel.from_pretrained(self.ckpt, device_map="auto").to(DEVICE).eval()
            self.processor = AutoProcessor.from_pretrained(self.ckpt)

    def embed_images(self, images: "list[Image.Image]") -> np.ndarray:
        images = images if isinstance(images, list) else [images]
        inputs = self.processor(images=images, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            embeddings = self.model.get_image_features(**inputs).pooler_output.cpu().numpy()
        return np.array([e for e in embeddings])

SIGLIP2_MODEL = SigLip2()

In [ ]:
class HuBERT():
    def __init__(self):
        try:
            self.model = HUBERT_MODEL.model
            self.feature_extractor = HUBERT_MODEL.feature_extractor
        except:
            self.model = HubertModel.from_pretrained("facebook/hubert-large-ll60k").to(DEVICE)
            self.feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-large-ll60k")

    def embed_audios(self, audios: list) -> np.ndarray:
        audios = audios if isinstance(audios, list) else [audios]
        inputs = self.feature_extractor(audios, return_tensors="pt", padding=True, sampling_rate=16000).to(DEVICE)
        with torch.no_grad():
            embeddings = self.model(**inputs, output_hidden_states=True).last_hidden_state.cpu().numpy().mean(axis=1)
        return np.array([e for e in embeddings])

HUBERT_MODEL = HuBERT()

In [ ]:
class F2LLM():
    def __init__(self):
        try: self.model = F2LLM_MODEL.model
        except: self.model = SentenceTransformer("codefuse-ai/F2LLM-1.7B", model_kwargs={"torch_dtype": "bfloat16"}, device=DEVICE)

    def embed_queries(self, queries: "list[str]") -> np.ndarray:
        embeddings = self.model.encode_query(queries if isinstance(queries, list) else [queries])
        return np.array([e for e in embeddings])

    def embed_documents(self, documents: "list[str]") -> np.ndarray:
        embeddings = self.model.encode_document(documents if isinstance(documents, list) else [documents])
        return np.array([e for e in embeddings])

F2LLM_MODEL = F2LLM()

# 3. Modality feature extraction

Definitions for face-aligned video features, audio features, and linguistic/statistical text features.

In [ ]:
from retinaface import RetinaFace
from skimage import transform as trans
import cv2

In [ ]:
def crop_align_face(img: Image.Image, dims=(256,256)) -> Image.Image:
    img_np = np.array(img.convert('RGB'))
    img_cv2 = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

    resp = RetinaFace.detect_faces(img_cv2, threshold=0.5)
    if not isinstance(resp, dict) or len(resp) == 0:
        return img.resize(dims)

    face_key = list(resp.keys())[0]
    lm = resp[face_key]['landmarks']

    eyes = np.array([lm['left_eye'], lm['right_eye']])
    eyes = eyes[np.argsort(eyes[:, 0])]
    mouths = np.array([lm['mouth_left'], lm['mouth_right']])
    mouths = mouths[np.argsort(mouths[:, 0])]
    nose = np.array(lm['nose'])

    src_pts = np.array([eyes[0], eyes[1], nose, mouths[0], mouths[1]], dtype=np.float32)
    dst_pts = np.array([[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366], [41.5493, 92.3655], [70.7299, 92.2041]], dtype=np.float32) * dims[0] / 112

    tform = trans.SimilarityTransform()
    tform.estimate(src_pts, dst_pts)

    M = tform.params[0:2, :]
    aligned_face = cv2.warpAffine(img_cv2, M, dims, borderValue=0.0)
    return Image.fromarray(cv2.cvtColor(aligned_face, cv2.COLOR_BGR2RGB))

In [ ]:
def extract_video_features(video, fps: int=24, extract_faces=True) -> dict:
    # Reading the video
    if isinstance(video, str):
        cap = cv2.VideoCapture(video)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        cap.release()
    elif isinstance(video, list):
        frames = video
    else:
        raise ValueError("Invalid video input. It must be either a path to a video or a list of images")

    # Extracting the faces
    if extract_faces:
        frames = [crop_align_face(f) for f in tqdm(frames, desc="Extracting faces", position=0, leave=True)]

    embed_image = lambda i : SIGLIP2_MODEL.embed_images(i)[0] if isinstance(i, Image.Image) else None
    vid = np.array([embed_image(i) for i in tqdm(frames, desc="Embedding with SigLip2", position=0, leave=True)])

    # Normal + 1st derivative + 2nd derivative
    vnorm = linalg_array(vid)
    diff1 = linalg_array(np.pad(np.diff(vnorm, n=1, axis=0), ((0,1),(0,0))))
    diff2 = linalg_array(np.pad(np.diff(vnorm, n=2, axis=0), ((0,2),(0,0))))
    video_emb = np.concatenate([vnorm.mean(axis=0), diff1.mean(axis=0), diff2.mean(axis=0)])

    # Similarities
    sim_matrix = vnorm @ vnorm.T
    np.fill_diagonal(sim_matrix, np.nan)
    sims = np.nanmean(sim_matrix, axis=1)

    # Outliers
    med = np.median(sims)
    mad = np.median(np.abs(sims - med))
    threshold = float(np.clip(med - 50 * mad, 0.0, 1.0))
    mask = sims >= threshold

    # Chunking
    chunk_idxes = [np.arange(i, min(i + fps, vid.shape[0])) for i in range(0, vid.shape[0], fps)]

    stats = []

    for c in chunk_idxes:
        chunk_sims = sims[c]
        valid = mask[c]
        stats.append([1, valid.mean(), chunk_sims[valid].mean()] if valid.any() else [0, 0, 0])

    stats = np.array(stats)
    stat_names = ["valid_chunk", "valid_ratio", "sim_mean"]
    return {"SigLip2": video_emb, **{name: stats_pool(stats[:, i]) for i, name in enumerate(stat_names)}}

In [ ]:
def extract_audio_features(video_path, sr=16000) -> dict:
    full_audio, _ = librosa.load(video_path, sr=sr, mono=True)
    hubert_emb = HUBERT_MODEL.embed_audios(full_audio)[0]
    aud_secs = len(full_audio) / sr
    full_audio = librosa.util.fix_length(full_audio, size=round(aud_secs * sr))
    n_chunks = math.ceil(aud_secs)
    audio_ref = np.max(np.abs(full_audio))

    stats = []
    for i in range(n_chunks):
        chunk = full_audio[i*sr:min((i+1)*sr, len(full_audio))]
        f0 = librosa.yin(chunk, fmin=50, fmax=500)

        stats.append([
            librosa.feature.rms(y=chunk).mean(),
            librosa.feature.spectral_centroid(y=chunk, sr=sr).mean(),
            librosa.feature.spectral_bandwidth(y=chunk, sr=sr).mean(),
            librosa.feature.spectral_rolloff(y=chunk, sr=sr).mean(),
            librosa.feature.zero_crossing_rate(chunk).mean(),
            np.mean(librosa.amplitude_to_db(np.abs(chunk), ref=audio_ref) < -30),
            np.nanmean(f0),
            np.nanstd(f0)
        ])

    stats = np.array(stats)
    stat_names = ["rms", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "zero_crossing_rate", "silence_ratio", "pitch_mean", "pitch_std"]
    return {"HuBERT": hubert_emb, **{name: stats_pool(stats[:, i]) for i, name in enumerate(stat_names)}}

In [ ]:
def get_ambivalent_cues(embeddings: np.ndarray) -> np.ndarray:
    all_cues = []
    for e in embeddings:
        cue_vec = []
        for group in AMBIVALENT_CUES:
            group_vec = np.array([cosine_similarity(e, ce) for ce in np.array(list(group["answer embeddings"].values()))])
            group_vec = softmax(max_abs_norm(group_vec) * 2)
            cue_vec.append(group_vec)
        all_cues.append(np.concatenate(cue_vec))
    return np.array(all_cues)

def get_hesitant_cues(embeddings: np.ndarray) -> np.ndarray:
    all_cues = []
    for e in embeddings:
        cue_vec = []
        for group in HESITANT_CUES:
            sims = np.array([cosine_similarity(e, ce) for ce in np.array(list(group["answer embeddings"].values()))])
            sims = softmax(max_abs_norm(sims) * np.sqrt(len(group)))
            cue_vec += [sims.max() - sims.mean()]
        all_cues.append(np.array(cue_vec))
    return np.array(all_cues)

def extract_text_features(text: str) -> dict:
    text = text.strip()

    # Global embedding
    full_embedding = F2LLM_MODEL.embed_documents([text])[0]

    # Tokenization
    clean = clean_text(text).lower()
    words = clean.split()
    n_words = max(len(words), 1)
    word_set = set(words)

    # Global statistics
    global_stats = {
        "word_count": len(words),
        "short_pause_count": text.count(","),
        "long_pause_count": sum(text.count(c) for c in ".!?"),
        "repetition_count": sum(words.count(w) for w in word_set if words.count(w) > 1),
        "adjacent_repetition_count": sum(words[i] == words[i+1] for i in range(len(words)-1)),
        "lexical_diversity": len(word_set) / n_words
    }

    # Ambivalent cues
    ambivalent = get_ambivalent_cues([full_embedding])[0]
    chunks = [c.strip() for c in text.replace(",", ".").replace("!", ".").replace("?", ".").split(".") if c.strip()]
    chunk_embeddings = []
    chunk_stats = []

    for c in chunks:
        c_words = clean_text(c).lower().split()
        emb = F2LLM_MODEL.embed_documents([c])[0]
        chunk_embeddings.append(emb)
        chunk_stats.append([len(c_words), len(set(c_words)) / max(len(c_words), 1), cosine_similarity(emb, full_embedding)])

    chunk_embeddings = np.array(chunk_embeddings) if len(chunk_embeddings) > 0 else np.zeros((1, full_embedding.shape[0]))
    chunk_stats = np.array(chunk_stats) if len(chunk_stats) > 0 else np.zeros((1,3))

    # Hesitation cues
    hesitation = get_hesitant_cues(chunk_embeddings)

    stat_names = ["chunk_word_count", "chunk_lexical_diversity", "chunk_similarity"]
    hesitation_names = [f"hesitation_{i}" for i in range(hesitation.shape[1])] if hesitation.ndim > 1 else ["hesitation"]
    return {"F2LLM": full_embedding, **global_stats, **{name: stats_pool(chunk_stats[:, i]) for i, name in enumerate(stat_names)}, **{name: stats_pool(hesitation[:, i]) for i, name in enumerate(hesitation_names)}, "ambivalent": ambivalent}

# 4. Load embeddings and assemble dataset matrices

This is the normal entry point when modality embeddings have already been extracted.

In [ ]:
with open("embeddings/text_embeddings.pkl", "rb") as f:
    TEXT_EMBEDDINGS : dict = pickle.load(f)

with open("embeddings/video_embeddings.pkl", "rb") as f:
    VIDEO_EMBEDDINGS : dict = pickle.load(f)

with open("embeddings/audio_embeddings.pkl", "rb") as f:
    AUDIO_EMBEDDINGS : dict = pickle.load(f)

In [ ]:
FULL_EMBEDDINGS = {}

for set_name in ["train", "val", "test", "unlabeled"]:
    with open(f"labels/{set_name}.txt", "r") as f:
        lines = [l.split(",")[:2] for l in f.readlines()]
        keys, ys = [l[0] for l in lines], [int(l[1]) for l in lines]

    for k,y in zip(keys, ys):
        new_data = {**TEXT_EMBEDDINGS[k], **VIDEO_EMBEDDINGS[k], **AUDIO_EMBEDDINGS[k]}
        new_data = {a:b if isinstance(b, np.ndarray) else np.array([b]) for a,b in new_data.items()}

        new_data = {
            "F2LLM": new_data["F2LLM"],
            "SigLip2": new_data["SigLip2"],
            "HuBERT": new_data["HuBERT"],
            "statistics": np.concatenate([new_data[a] for a in new_data.keys() if a not in ["F2LLM", "SigLip2", "HuBERT", "set", "y"]], axis=0),
            "set": set_name,
            "y": y if set_name != "unlabeled" else None
        }
        new_data = dict(sorted(new_data.items()))
        FULL_EMBEDDINGS[k] = new_data

In [ ]:
X_text = {"train": [], "val": [], "test": [], "unlabeled": []}
X_audio = {"train": [], "val": [], "test": [], "unlabeled": []}
X_video = {"train": [], "val": [], "test": [], "unlabeled": []}
X_stats = {"train": [], "val": [], "test": [], "unlabeled": []}
y = {"train": [], "val": [], "test": [], "unlabeled": []}

for v in FULL_EMBEDDINGS.values():
    X_text[v["set"]].append(v["F2LLM"])
    X_audio[v["set"]].append(v["HuBERT"])
    X_video[v["set"]].append(v["SigLip2"])
    X_stats[v["set"]].append(v["statistics"])
    y[v["set"]].append(v["y"])

for arr in [X_text, X_audio, X_video, X_stats, y]:
    for k, v in arr.items():
        arr[k] = np.array(v)

X_text = {k: linalg_array(v) for k,v in X_text.items()}
X_audio = {k: linalg_array(v) for k,v in X_audio.items()}
X_stats = {k: np.nan_to_num(v, 0.0) for k,v in X_stats.items()}

# 5. CALMAH pipeline


In [ ]:
# import joblib
# import itertools
# import tensorflow as tf
# from tensorflow.keras.layers import Input, Dense, Dropout, GaussianNoise, BatchNormalization, Activation
# from tensorflow.keras.models import Model, load_model
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# from lightgbm import LGBMClassifier
# ============================================================
# K1: Keras 3 imports for CALMAH prediction
# ============================================================

import os
import itertools
import joblib
import numpy as np

import tensorflow as tf
import keras

from keras.layers import (
    Input,
    Dense,
    Dropout,
    GaussianNoise,
    BatchNormalization,
    Activation,
)

from keras.models import Model, load_model
from keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from lightgbm import LGBMClassifier


print("TensorFlow version:", tf.__version__)
print("Standalone Keras version:", keras.__version__)

print("keras.Model module:")
print(keras.Model.__module__)

print("tf.keras.Model module:")
print(tf.keras.Model.__module__)

assert int(keras.__version__.split(".")[0]) >= 3, (
    "CALMAH .keras models require standalone Keras 3."
)

assert not tf.keras.Model.__module__.startswith("tf_keras"), (
    "tf.keras is still routed to legacy tf_keras. "
    "Restart the kernel and run K0 before importing TensorFlow."
)

print("\nPASS: Keras 3 is active with the TensorFlow backend.")

In [ ]:
class CALMAH_pipeline:
    def __init__(self):
        self.stats_scaler = None
        self.video_scaler = None
        self.video_pca = None
        self.models = {}
        self.model_types = {}
        self.combinations = []
        self.thresholds = {}
        self.weights = None

    def save(self, save_dir):
        os.makedirs(save_dir, exist_ok=True)
        for name, model in self.models.items():
            if self.model_types[name] == "mlp":
                model.save(os.path.join(save_dir, f"model_{name}.keras"))
            else:
                joblib.dump(model, os.path.join(save_dir, f"model_{name}.joblib"))

        meta_data = {
            "stats_scaler": self.stats_scaler,
            "video_scaler": self.video_scaler,
            "video_pca": self.video_pca,
            "combinations": self.combinations,
            "model_types": self.model_types,
            "thresholds": self.thresholds,
            "weights": self.weights
        }
        joblib.dump(meta_data, os.path.join(save_dir, "pipeline_meta.joblib"))
        print(f"Pipeline successfully saved to {save_dir}/")

    @classmethod
    def load(cls, load_dir):
        pipeline = cls()
        meta_data = joblib.load(os.path.join(load_dir, "pipeline_meta.joblib"))
        pipeline.stats_scaler = meta_data["stats_scaler"]
        pipeline.video_scaler = meta_data.get("video_scaler")
        pipeline.video_pca = meta_data.get("video_pca")
        pipeline.combinations = meta_data["combinations"]
        pipeline.model_types = meta_data.get("model_types", {})
        pipeline.thresholds = meta_data["thresholds"]
        pipeline.weights = meta_data["weights"]

        # Load models dynamically based on their saved type
        for combo_name in pipeline.combinations:
            m_type = pipeline.model_types.get(combo_name, "mlp")
            if m_type == "mlp":
                model_path = os.path.join(load_dir, f"model_{combo_name}.keras")
                if os.path.exists(model_path):
                    pipeline.models[combo_name] = load_model(model_path)
            else:
                model_path = os.path.join(load_dir, f"model_{combo_name}.joblib")
                if os.path.exists(model_path):
                    pipeline.models[combo_name] = joblib.load(model_path)

        print(f"Pipeline successfully loaded from {load_dir}/")
        return pipeline

    def predict(self, X_text, X_audio, X_video, X_stats, custom_weights=None):
        X_stats_scaled = self.stats_scaler.transform(X_stats)
        X_video_scaled = self.video_scaler.transform(X_video)
        X_video_pca = self.video_pca.transform(X_video_scaled)
        raw_inputs = {"Text": X_text, "Audio": X_audio, "Video": X_video_pca, "Stats": X_stats_scaled}

        weights = custom_weights if custom_weights is not None else self.weights
        if weights is None:
            weights = np.ones(len(self.combinations)) / len(self.combinations)
        weighted_sum = np.zeros(X_text.shape[0])

        for idx, combo_name in enumerate(self.combinations):
            modality_names = combo_name.split("_")
            X_combo = np.hstack([raw_inputs[mod] for mod in modality_names])

            # Predict differently depending on the winning algorithm type
            if self.model_types[combo_name] == "mlp":
                pred_prob = self.models[combo_name].predict(X_combo, verbose=0).flatten()
            else:
                pred_prob = self.models[combo_name].predict_proba(X_combo)[:, 1]

            threshold = self.thresholds.get(combo_name, 0.5)
            pred_binary = (pred_prob >= threshold).astype(int)
            weighted_sum += pred_binary * weights[idx]

        final_preds = (weighted_sum >= (np.sum(weights) / 2.0)).astype(int)
        return final_preds

# 7. Current dataset reconstruction


In [ ]:
import sys
print(sys.executable)

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import hashlib
import json
import os
import pickle
import re
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")

if "CALMAH_pipeline" not in globals():
    raise RuntimeError(
        "CALMAH_pipeline is not defined. Run the original Run.ipynb imports and "
        "the Section 5 cell that defines class CALMAH_pipeline, then start this append block."
    )


def locate_repo_root():
    candidates = [
        Path.cwd(),
        Path("/workspace/project"),
        Path.cwd() / "CALMAH_ABAW11_AH",
        Path.cwd().parent / "CALMAH_ABAW11_AH",
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (
            (candidate / "CALMAH_v1" / "pipeline_meta.joblib").exists()
            and (candidate / "embeddings").exists()
            and (candidate / "labels").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the CALMAH_ABAW11_AH repository root.")


REPO_DIR = locate_repo_root()
NEW_DATA_ROOT = REPO_DIR / "data"
OLD_LABEL_DIR = REPO_DIR / "labels"
EMBEDDING_DIR = REPO_DIR / "embeddings"
PIPELINE_DIR = REPO_DIR / "CALMAH_v1"
OUTPUT_DIR = REPO_DIR / "abaw11_append_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_COUNTS = {
    "train": 778,
    "val": 124,
    "test": 525,
    "private": 152,
}

assert NEW_DATA_ROOT.exists(), NEW_DATA_ROOT
assert (NEW_DATA_ROOT / "Videos").exists(), NEW_DATA_ROOT / "Videos"
assert (NEW_DATA_ROOT / "split").exists(), NEW_DATA_ROOT / "split"

# Reload from disk deliberately. This removes any w4/subset-weight changes
# that may have been made in earlier notebook cells.
pipeline = CALMAH_pipeline.load(str(PIPELINE_DIR))
disk_meta = joblib.load(PIPELINE_DIR / "pipeline_meta.joblib")


def normalize_combo_name(combo):
    if isinstance(combo, (list, tuple)):
        return "_".join(map(str, combo))
    return str(combo)


COMBO_NAMES = [normalize_combo_name(c) for c in disk_meta["combinations"]]
ORIGINAL_WEIGHTS = np.asarray(disk_meta["weights"], dtype=np.float64).copy()
ORIGINAL_THRESHOLDS = np.asarray(
    [disk_meta["thresholds"][name] for name in COMBO_NAMES],
    dtype=np.float64,
)

assert len(COMBO_NAMES) == 15
assert ORIGINAL_WEIGHTS.shape == (15,)
assert ORIGINAL_THRESHOLDS.shape == (15,)
assert set(COMBO_NAMES) == set(pipeline.models.keys())

# Restore the exact released state in memory as well.
pipeline.weights = ORIGINAL_WEIGHTS.copy()
pipeline.thresholds = dict(disk_meta["thresholds"])

print("Repository:", REPO_DIR)
print("Current dataset:", NEW_DATA_ROOT)
print("Output directory:", OUTPUT_DIR)
print("Model order:", COMBO_NAMES)
print("Released weight sum:", ORIGINAL_WEIGHTS.sum())
print("PASS: released CALMAH weights and thresholds were restored from disk.")

In [ ]:
# Run this after the original append Cell A1 has loaded the released CALMAH pipeline.

from pathlib import Path

NEW_DATA_ROOT = REPO_DIR / "data"
CURRENT_UNLABEL_ROOT = REPO_DIR / "unlabel" / "data"

assert NEW_DATA_ROOT.exists(), NEW_DATA_ROOT
assert (NEW_DATA_ROOT / "Videos").exists(), NEW_DATA_ROOT / "Videos"
assert (NEW_DATA_ROOT / "split").exists(), NEW_DATA_ROOT / "split"

assert CURRENT_UNLABEL_ROOT.exists(), CURRENT_UNLABEL_ROOT

print("Current labeled root :", NEW_DATA_ROOT)
print("Current unlabel root :", CURRENT_UNLABEL_ROOT)

# Legacy recovery outputs are intentionally quarantined.
LEGACY_PRIVATE_WORK_DIR = Path(
    "/workspace/outputs/official_unlabeled_work"
)
print("Legacy private cache (will NOT be loaded):", LEGACY_PRIVATE_WORK_DIR)

In [ ]:
from collections import defaultdict
from pathlib import Path
import re

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}


def norm_path(value):
    value = str(value).strip().strip('"').strip("'").replace("\\", "/")
    value = re.sub(r"^\./", "", value)
    value = re.sub(r"/+", "/", value)
    return value


def basename_id(value):
    return Path(norm_path(value)).name.lower()


def parse_video_line(line):
    """Return the first video token and an optional binary label."""
    line = str(line).strip()
    if not line:
        return None, None

    match = re.search(
        r"([^,\t\r\n]*?\.(?:mp4|avi|mov|mkv|webm|m4v))",
        line,
        flags=re.I,
    )

    if match:
        raw_video_id = match.group(1).strip().strip('"').strip("'")
        tail = line[match.end():]
    else:
        fields = [x.strip() for x in re.split(r"[,\t]", line) if x.strip()]
        raw_video_id = fields[0] if fields else None
        tail = line[len(raw_video_id):] if raw_video_id else ""

    label = None
    for token in re.split(r"[,\t\s]+", tail.strip(" ,\t")):
        if token in {"0", "1", "0.0", "1.0"}:
            label = int(float(token))
            break

    return norm_path(raw_video_id) if raw_video_id else None, label


# ------------------------------------------------------------------
# A. Current labeled train/val/test
# ------------------------------------------------------------------

LABELED_VIDEOS_ROOT = NEW_DATA_ROOT / "Videos"

labeled_video_paths = sorted(
    path for path in LABELED_VIDEOS_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
)

assert len(labeled_video_paths) == (
    EXPECTED_COUNTS["train"]
    + EXPECTED_COUNTS["val"]
    + EXPECTED_COUNTS["test"]
), len(labeled_video_paths)

labeled_by_basename = defaultdict(list)
labeled_by_relative = {}

for path in labeled_video_paths:
    relative = norm_path(path.relative_to(NEW_DATA_ROOT))
    labeled_by_relative[relative.lower()] = relative
    labeled_by_basename[path.name.lower()].append(relative)

duplicate_labeled_basenames = {
    key: values
    for key, values in labeled_by_basename.items()
    if len(values) != 1
}
assert not duplicate_labeled_basenames, list(duplicate_labeled_basenames.items())[:5]


def resolve_labeled_video(raw_id):
    raw_id = norm_path(raw_id)

    candidates = [
        raw_id,
        raw_id.removeprefix("data/"),
        f"Videos/{raw_id}" if not raw_id.lower().startswith("videos/") else raw_id,
    ]

    for candidate in candidates:
        candidate = norm_path(candidate).lower()
        if candidate in labeled_by_relative:
            return labeled_by_relative[candidate]

    matches = labeled_by_basename.get(Path(raw_id).name.lower(), [])

    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous labeled video: {raw_id} -> {matches}")

    raise KeyError(f"Cannot resolve labeled split entry: {raw_id}")


new_split_rows = []

for split_name in ["train", "val", "test"]:
    split_path = NEW_DATA_ROOT / "split" / f"{split_name}.txt"
    assert split_path.exists(), split_path

    with split_path.open("r", encoding="utf-8", errors="replace") as handle:
        for order, line in enumerate(handle):
            raw_id, label = parse_video_line(line)

            if raw_id is None:
                continue

            video_id = resolve_labeled_video(raw_id)
            absolute_path = NEW_DATA_ROOT / video_id

            new_split_rows.append({
                "split": split_name,
                "order": order,
                "raw_id": raw_id,
                "video_id": video_id,
                "video_path": str(absolute_path),
                "basename": basename_id(video_id),
                "label": label,
            })

new_split_df = pd.DataFrame(new_split_rows)
assert not new_split_df.empty


# Recover labels from bah-video.csv when split files do not include them.
metadata_path = NEW_DATA_ROOT / "bah-video.csv"
metadata_df = pd.read_csv(metadata_path) if metadata_path.exists() else None
CSV_VIDEO_COL = None
CSV_LABEL_COL = None

if metadata_df is not None:
    basename_universe = set(labeled_by_basename)

    video_candidates = []
    for column in metadata_df.columns:
        sample = metadata_df[column].dropna().astype(str)
        if sample.empty:
            continue

        matched = sample.map(lambda value: basename_id(value) in basename_universe).sum()
        name_bonus = (
            1000
            if any(token in column.lower() for token in ["video", "file", "path"])
            else 0
        )

        if matched:
            video_candidates.append((matched + name_bonus, matched, column))

    if video_candidates:
        CSV_VIDEO_COL = max(video_candidates)[2]

    label_candidates = []
    for column in metadata_df.columns:
        values = metadata_df[column].dropna()

        if values.empty:
            continue

        normalized_values = set(
            values.astype(str).str.strip().str.lower().unique()
        )

        if normalized_values and normalized_values.issubset(
            {"0", "1", "0.0", "1.0", "false", "true"}
        ):
            name_bonus = (
                1000
                if any(
                    token in column.lower()
                    for token in [
                        "label", "target", "ambivalence",
                        "hesitancy", "class",
                    ]
                )
                else 0
            )
            label_candidates.append((len(values) + name_bonus, column))

    if label_candidates:
        CSV_LABEL_COL = max(label_candidates)[1]

print("Auto-detected labeled CSV video column:", CSV_VIDEO_COL)
print("Auto-detected labeled CSV label column:", CSV_LABEL_COL)

if (
    new_split_df["label"].isna().any()
    and metadata_df is not None
    and CSV_VIDEO_COL
    and CSV_LABEL_COL
):
    csv_label_map = {}

    for _, row in metadata_df[[CSV_VIDEO_COL, CSV_LABEL_COL]].dropna().iterrows():
        raw_value = str(row[CSV_LABEL_COL]).strip().lower()

        if raw_value in {"0", "0.0", "false"}:
            csv_label_map[basename_id(row[CSV_VIDEO_COL])] = 0
        elif raw_value in {"1", "1.0", "true"}:
            csv_label_map[basename_id(row[CSV_VIDEO_COL])] = 1

    missing_mask = new_split_df["label"].isna()
    new_split_df.loc[missing_mask, "label"] = (
        new_split_df.loc[missing_mask, "basename"].map(csv_label_map)
    )

split_counts = new_split_df.groupby("split").size().to_dict()

for split_name in ["train", "val", "test"]:
    assert split_counts.get(split_name) == EXPECTED_COUNTS[split_name], (
        split_name,
        split_counts.get(split_name),
        EXPECTED_COUNTS[split_name],
    )

for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    left_ids = set(new_split_df.loc[new_split_df.split == left, "basename"])
    right_ids = set(new_split_df.loc[new_split_df.split == right, "basename"])
    assert not (left_ids & right_ids), f"{left}/{right} overlap detected."


# ------------------------------------------------------------------
# B. Current 152 competition-private videos
# ------------------------------------------------------------------

private_video_paths = sorted(
    path for path in CURRENT_UNLABEL_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
)

print("Current unlabel videos found:", len(private_video_paths))
assert len(private_video_paths) == EXPECTED_COUNTS["private"], (
    len(private_video_paths),
    EXPECTED_COUNTS["private"],
)

private_disk_by_basename = defaultdict(list)

for path in private_video_paths:
    private_disk_by_basename[path.name.lower()].append(path)

duplicate_private_basenames = {
    key: values
    for key, values in private_disk_by_basename.items()
    if len(values) != 1
}
assert not duplicate_private_basenames, list(duplicate_private_basenames.items())[:5]

private_basename_set = set(private_disk_by_basename)


def normalize_submission_id(raw_value):
    """Keep an official relative ID when available; remove machine-local prefixes."""
    raw_value = norm_path(raw_value)

    current_root_text = norm_path(CURRENT_UNLABEL_ROOT)
    if raw_value.startswith(current_root_text + "/"):
        raw_value = raw_value[len(current_root_text) + 1:]

    if raw_value.startswith("/"):
        raw_value = Path(raw_value).name

    return raw_value


def order_candidate_from_csv(path):
    try:
        frame = pd.read_csv(path)
    except Exception:
        return None

    best = None

    for column in frame.columns:
        values = frame[column].dropna().astype(str).tolist()
        matched = [
            value for value in values
            if basename_id(value) in private_basename_set
        ]

        matched_basenames = [basename_id(value) for value in matched]

        if (
            len(matched_basenames) == EXPECTED_COUNTS["private"]
            and len(set(matched_basenames)) == EXPECTED_COUNTS["private"]
            and set(matched_basenames) == private_basename_set
        ):
            score = 0
            lower_name = path.name.lower()
            lower_column = column.lower()

            if any(token in lower_name for token in [
                "submission", "template", "private", "unlabel", "test", "bah-video"
            ]):
                score += 100

            if any(token in lower_column for token in [
                "video", "file", "path", "id"
            ]):
                score += 50

            candidate = {
                "score": score,
                "source": f"{path}::{column}",
                "raw_ids": matched,
            }

            if best is None or candidate["score"] > best["score"]:
                best = candidate

    return best


def order_candidate_from_text(path):
    if path.stat().st_size > 20 * 1024 * 1024:
        return None

    raw_ids = []

    try:
        with path.open("r", encoding="utf-8", errors="replace") as handle:
            for line in handle:
                video_id, _ = parse_video_line(line)
                if video_id is not None and basename_id(video_id) in private_basename_set:
                    raw_ids.append(video_id)
    except Exception:
        return None

    basenames = [basename_id(value) for value in raw_ids]

    if not (
        len(basenames) == EXPECTED_COUNTS["private"]
        and len(set(basenames)) == EXPECTED_COUNTS["private"]
        and set(basenames) == private_basename_set
    ):
        return None

    score = 0
    lower_name = path.name.lower()

    if any(token in lower_name for token in [
        "submission", "template", "private", "unlabel", "test", "list"
    ]):
        score += 100

    return {
        "score": score,
        "source": str(path),
        "raw_ids": raw_ids,
    }


order_candidates = []

for path in CURRENT_UNLABEL_ROOT.rglob("*.csv"):
    candidate = order_candidate_from_csv(path)
    if candidate is not None:
        order_candidates.append(candidate)

for suffix in ["*.txt", "*.tsv"]:
    for path in CURRENT_UNLABEL_ROOT.rglob(suffix):
        candidate = order_candidate_from_text(path)
        if candidate is not None:
            order_candidates.append(candidate)

if order_candidates:
    selected_order = max(order_candidates, key=lambda item: item["score"])
    official_raw_ids = selected_order["raw_ids"]
    PRIVATE_ORDER_SOURCE = selected_order["source"]
else:
    # Deterministic fallback. The output includes IDs, so the order is usually
    # not semantically required, but the challenge template should still be
    # checked before submission.
    official_raw_ids = [
        norm_path(path.relative_to(CURRENT_UNLABEL_ROOT))
        for path in private_video_paths
    ]
    PRIVATE_ORDER_SOURCE = "sorted_disk_fallback"

ordered_private_paths = []

for raw_id in official_raw_ids:
    basename = basename_id(raw_id)
    matches = private_disk_by_basename.get(basename, [])

    assert len(matches) == 1, (raw_id, matches)
    ordered_private_paths.append(matches[0])

private_rows = []

for private_order, (raw_id, path) in enumerate(
    zip(official_raw_ids, ordered_private_paths)
):
    relative_id = norm_path(path.relative_to(CURRENT_UNLABEL_ROOT))

    private_rows.append({
        "private_order": private_order,
        "video_id": relative_id,
        "video_path": str(path),
        "basename": path.name.lower(),
        "submission_id": normalize_submission_id(raw_id),
        "order_source": PRIVATE_ORDER_SOURCE,
        "file_size_bytes": path.stat().st_size,
    })

private_df = pd.DataFrame(private_rows)

assert len(private_df) == EXPECTED_COUNTS["private"]
assert private_df["basename"].is_unique
assert private_df["submission_id"].is_unique
assert all(Path(path).exists() for path in private_df["video_path"])

labeled_private_overlap = (
    set(new_split_df["basename"])
    & set(private_df["basename"])
)
assert not labeled_private_overlap, sorted(labeled_private_overlap)[:20]

new_split_df.to_csv(
    OUTPUT_DIR / "current_labelled_manifest.csv",
    index=False,
)

private_df.to_csv(
    OUTPUT_DIR / "current_private_manifest.csv",
    index=False,
)

print("\nCurrent labelled split counts:", split_counts)
print("Current private count:", len(private_df))
print("Private order source:", PRIVATE_ORDER_SOURCE)

display(
    private_df[[
        "private_order",
        "submission_id",
        "video_id",
        "video_path",
    ]].head(10)
)

if PRIVATE_ORDER_SOURCE == "sorted_disk_fallback":
    print(
        "\nWARNING: no 152-entry template/list was found. "
        "The manifest uses deterministic disk order. "
        "Check the challenge sample-submission format before upload."
    )

print("\nPASS: current private IDs are derived from the real ABAW11 unlabel directory.")
print("PASS: no legacy official_template_manifest.json was used.")

In [ ]:
def participant_question(video_name):
    name = Path(norm_path(video_name)).name
    participant_match = re.match(r"(\d+)_", name)
    question_match = re.search(r"_Question_(\d+)_", name, flags=re.I)
    return (
        participant_match.group(1) if participant_match else None,
        int(question_match.group(1)) if question_match else None,
    )


def read_old_label_file(split_name):
    path = OLD_LABEL_DIR / f"{split_name}.txt"
    assert path.exists(), path
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for order, line in enumerate(handle):
            raw_id, label = parse_video_line(line)
            if raw_id is None:
                continue
            participant, question = participant_question(raw_id)
            rows.append({
                "old_split": split_name,
                "old_order": order,
                "old_raw_id": raw_id,
                "basename": basename_id(raw_id),
                "old_label": label,
                "participant": participant,
                "question": question,
            })
    return rows


old_rows = []
for split_name in ["train", "val", "test", "unlabeled"]:
    old_rows.extend(read_old_label_file(split_name))
old_df = pd.DataFrame(old_rows)
assert old_df["basename"].is_unique, "Old label files contain duplicate video basenames."

current_compare_rows = []
for _, row in new_split_df.iterrows():
    participant, question = participant_question(row.video_id)
    current_compare_rows.append({
        "new_split": row.split,
        "new_order": row.order,
        "new_raw_id": row.raw_id,
        "basename": row.basename,
        "new_label": row.label,
        "participant": participant,
        "question": question,
    })
for _, row in private_df.iterrows():
    participant, question = participant_question(row.video_id)
    current_compare_rows.append({
        "new_split": "private",
        "new_order": row.private_order,
        "new_raw_id": row.submission_id,
        "basename": row.basename,
        "new_label": None,
        "participant": participant,
        "question": question,
    })
current_compare_df = pd.DataFrame(current_compare_rows)
assert current_compare_df["basename"].is_unique

comparison_df = old_df.merge(
    current_compare_df,
    on="basename",
    how="outer",
    suffixes=("_old", "_new"),
)

comparison_df["participant"] = comparison_df["participant_new"].fillna(comparison_df["participant_old"])
comparison_df["question"] = comparison_df["question_new"].fillna(comparison_df["question_old"])


def comparison_status(row):
    if pd.isna(row.old_split):
        return "added_in_current"
    if pd.isna(row.new_split):
        return "removed_from_current"
    old_normalized = "private" if row.old_split == "unlabeled" else row.old_split
    if old_normalized != row.new_split:
        return "moved_split"
    old_label = row.old_label
    new_label = row.new_label
    if pd.notna(old_label) and pd.notna(new_label) and int(old_label) != int(new_label):
        return "label_changed"
    return "unchanged"


comparison_df["status"] = comparison_df.apply(comparison_status, axis=1)
comparison_df["label_changed"] = (
    comparison_df["old_label"].notna()
    & comparison_df["new_label"].notna()
    & (comparison_df["old_label"].astype("Int64") != comparison_df["new_label"].astype("Int64"))
)

comparison_columns = [
    "status", "participant", "question", "basename",
    "old_split", "new_split", "old_label", "new_label",
    "old_raw_id", "new_raw_id",
]
comparison_df[comparison_columns].sort_values(
    ["status", "participant", "question", "basename"],
    na_position="last",
).to_csv(OUTPUT_DIR / "old_vs_current_dataset.csv", index=False)

old_private = set(old_df.loc[old_df.old_split == "unlabeled", "basename"])
new_private = set(private_df["basename"])
private_overlap = old_private & new_private
old_private_only = old_private - new_private
new_private_only = new_private - old_private

print("Overall status counts:")
print(comparison_df["status"].value_counts())
print("\nOld private count:", len(old_private))
print("Current private count:", len(new_private))
print("Private overlap:", len(private_overlap))
print("Removed from old private:", len(old_private_only))
print("Added to current private:", len(new_private_only))

private_diff_df = comparison_df[
    comparison_df["basename"].isin(old_private_only | new_private_only)
][comparison_columns].sort_values(["status", "participant", "question", "basename"])
private_diff_df.to_csv(OUTPUT_DIR / "private_set_difference.csv", index=False)

display(private_diff_df[[
    "status", "participant", "question", "basename", "old_split", "new_split"
]])

moved_or_changed_df = comparison_df[
    comparison_df.status.isin(["moved_split", "label_changed"])
][comparison_columns].sort_values(["status", "participant", "question", "basename"])
moved_or_changed_df.to_csv(OUTPUT_DIR / "moved_splits_or_changed_labels.csv", index=False)

if len(old_private) == 161 and len(new_private) == 152 and len(private_overlap) == 137:
    assert len(old_private_only) == 24
    assert len(new_private_only) == 15
    print("\nImportant: 161 -> 152 with overlap 137 means 24 old-private videos were removed and 15 new-private videos were added.")

print("\nSaved:")
print(OUTPUT_DIR / "old_vs_current_dataset.csv")
print(OUTPUT_DIR / "private_set_difference.csv")
print(OUTPUT_DIR / "moved_splits_or_changed_labels.csv")

# 8. Offline feature completion and audit

Runs missing-modality extraction in an isolated PyTorch environment, validates generated dictionaries, merges legacy/current features, and builds verified arrays.

In [ ]:
# ============================================================
# Patch P5 (replacement):
# Offline feature extraction in a separate PyTorch environment
# ============================================================
#
# Run this cell only after P4 reports missing required features.
#
# This TensorFlow notebook will:
#   1. Prepare transcripts and a missing-feature manifest.
#   2. Generate one standalone PyTorch extraction script.
#   3. Launch separate OFFLINE subprocesses for:
#        text  -> local F2LLM snapshot
#        audio -> local HuBERT snapshot
#        video -> local SigLip2 snapshot
#   4. Save resumable pickle files under CURRENT_EXTRACTION_DIR.
#
# No Hugging Face network request is allowed.

from pathlib import Path
from collections import defaultdict
import json
import os
import pickle
import re
import subprocess
import sys

import pandas as pd
import yaml


# ------------------------------------------------------------
# 0. Preconditions and local model paths
# ------------------------------------------------------------

if "missing_required_df" not in globals():
    raise RuntimeError("Run Patch P4 first.")

targets_df = missing_required_df.copy()

if targets_df.empty:
    print("No extraction is required. Rerun P4 and continue to A5.")
else:
    print("Videos requiring at least one modality:", len(targets_df))
    display(
        targets_df[[
            "split",
            "video_id",
            "video_path",
            "text_available",
            "audio_available",
            "video_available",
        ]]
    )

    assert len(targets_df) <= 50, (
        "More than 50 required videos are missing features. "
        "This is more likely a feature-key mapping problem than a "
        "legitimate extraction requirement.",
        len(targets_df),
    )


LOCAL_MODEL_PATHS = {
    "siglip2": Path(
        "/workspace/cache/huggingface/hub/"
        "models--google--siglip2-base-patch16-naflex/"
        "snapshots/b53b807d3a2d5e2b3911292f2d69e5341cdc064c"
    ),
    "hubert": Path(
        "/workspace/cache/huggingface/hub/"
        "models--facebook--hubert-large-ll60k/"
        "snapshots/ff022d095678a2995f3c49bab18a96a9e553f782"
    ),
    "f2llm": Path(
        "/workspace/cache/huggingface/hub/"
        "models--codefuse-ai--F2LLM-1.7B/"
        "snapshots/cb8d3c8032406948af52b960ebb05b9e7830ac3a"
    ),
}

PYTORCH_PYTHON_CANDIDATES = [
    Path("/opt/conda/envs/CALMAH/bin/python"),
    Path("/opt/conda/envs/CALMAH/bin/python"),
]

PYTORCH_PYTHON = next(
    (
        path.resolve()
        for path in PYTORCH_PYTHON_CANDIDATES
        if path.exists()
    ),
    None,
)

assert PYTORCH_PYTHON is not None, (
    "Could not locate the PyTorch CALMAH environment."
)

for model_name, model_path in LOCAL_MODEL_PATHS.items():
    assert model_path.exists(), (model_name, model_path)

assert CURRENT_EXTRACTION_DIR.exists(), CURRENT_EXTRACTION_DIR
assert REPO_DIR.exists(), REPO_DIR

FEATURE_GPU = str(globals().get("FEATURE_GPU", "5"))

print("\nTensorFlow notebook Python:", sys.executable)
print("PyTorch extraction Python:", PYTORCH_PYTHON)
print("Feature extraction GPU:", FEATURE_GPU)

for model_name, model_path in LOCAL_MODEL_PATHS.items():
    print(f"{model_name:8s}: {model_path}")


# ------------------------------------------------------------
# 1. Safe BAH YAML loader
# ------------------------------------------------------------

class BAHSafeLoader(yaml.SafeLoader):
    """SafeLoader plus the tuple tag used in the BAH transcription files."""
    pass


def construct_python_tuple(loader, node):
    return tuple(
        loader.construct_sequence(
            node,
            deep=True,
        )
    )


BAHSafeLoader.add_constructor(
    "tag:yaml.org,2002:python/tuple",
    construct_python_tuple,
)


# ------------------------------------------------------------
# 2. Transcript discovery
# ------------------------------------------------------------

TRANSCRIPT_FIELD_NAMES = {
    "text",
    "transcript",
    "transcription",
    "utterance",
    "sentence",
    "answer",
    "content",
}

SEGMENT_FIELD_NAMES = {
    "segments",
    "segment",
    "chunks",
    "chunk",
    "utterances",
    "sentences",
}


def collect_all_strings(value):
    strings = []

    if isinstance(value, str):
        cleaned = value.strip()
        if cleaned:
            strings.append(cleaned)

    elif isinstance(value, dict):
        for nested_value in value.values():
            strings.extend(collect_all_strings(nested_value))

    elif isinstance(value, (list, tuple)):
        for nested_value in value:
            strings.extend(collect_all_strings(nested_value))

    return strings


def clean_transcript_fragments(fragments):
    cleaned = []

    for fragment in fragments:
        fragment = str(fragment).strip()

        if not fragment:
            continue

        lower_fragment = fragment.lower()

        if lower_fragment.endswith(
            tuple(extension.lower() for extension in VIDEO_EXTENSIONS)
        ):
            continue

        if (
            ("/" in fragment or "\\" in fragment)
            and any(
                extension.lower() in lower_fragment
                for extension in VIDEO_EXTENSIONS
            )
        ):
            continue

        if lower_fragment in {
            "en",
            "english",
            "none",
            "null",
            "true",
            "false",
        }:
            continue

        if cleaned and fragment == cleaned[-1]:
            continue

        cleaned.append(fragment)

    return cleaned


def extract_transcript_from_yaml_object(obj):
    transcript_fragments = []
    text_fragments = []
    segment_fragments = []

    def walk(value, inside_segment=False):
        if isinstance(value, dict):
            for key, nested_value in value.items():
                key_lower = str(key).strip().lower()

                if (
                    "transcript" in key_lower
                    or "transcription" in key_lower
                ):
                    transcript_fragments.extend(
                        collect_all_strings(nested_value)
                    )
                    continue

                if (
                    key_lower in TRANSCRIPT_FIELD_NAMES
                    or key_lower.endswith("_text")
                ):
                    text_fragments.extend(
                        collect_all_strings(nested_value)
                    )
                    continue

                is_segment = (
                    inside_segment
                    or key_lower in SEGMENT_FIELD_NAMES
                    or "segment" in key_lower
                    or "chunk" in key_lower
                    or "utterance" in key_lower
                )

                if is_segment:
                    segment_fragments.extend(
                        collect_all_strings(nested_value)
                    )
                else:
                    walk(nested_value, inside_segment=False)

        elif isinstance(value, (list, tuple)):
            for nested_value in value:
                walk(nested_value, inside_segment=inside_segment)

    walk(obj)

    for fragments in [
        transcript_fragments,
        text_fragments,
        segment_fragments,
    ]:
        fragments = clean_transcript_fragments(fragments)

        if fragments:
            return " ".join(fragments).strip()

    return None


def infer_video_basename_from_yaml_path(yaml_path):
    yaml_path = Path(yaml_path)

    for parent in yaml_path.parents:
        if parent.name.lower().endswith(
            tuple(extension.lower() for extension in VIDEO_EXTENSIONS)
        ):
            return parent.name.lower()

    return None


def collect_transcripts_from_object(obj, target_basenames):
    transcript_map = {}

    video_hints = {
        "video",
        "video_path",
        "video-path",
        "file",
        "filename",
        "path",
        "id",
    }

    text_hints = {
        "transcript",
        "transcription",
        "text",
        "utterance",
        "answer",
        "sentence",
    }

    def add_candidate(video_value, text_value):
        basename = basename_id(video_value)

        if basename not in target_basenames:
            return

        fragments = clean_transcript_fragments(
            collect_all_strings(text_value)
        )

        if not fragments:
            return

        candidate = " ".join(fragments).strip()
        previous = transcript_map.get(basename)

        if previous is None or len(candidate) > len(previous):
            transcript_map[basename] = candidate

    def walk(value, forced_video_id=None):
        if isinstance(value, dict):
            local_video_ids = []

            if forced_video_id is not None:
                local_video_ids.append(forced_video_id)

            for key, nested_value in value.items():
                key_text = str(key)
                lower_key = key_text.lower()

                if Path(key_text).suffix.lower() in VIDEO_EXTENSIONS:
                    local_video_ids.append(key_text)
                    walk(nested_value, forced_video_id=key_text)
                    continue

                if (
                    lower_key in video_hints
                    or any(
                        hint in lower_key
                        for hint in ["video", "file", "path"]
                    )
                ) and isinstance(nested_value, str):
                    if Path(nested_value).suffix.lower() in VIDEO_EXTENSIONS:
                        local_video_ids.append(nested_value)

            text_values = []

            for key, nested_value in value.items():
                lower_key = str(key).lower()

                if (
                    lower_key in text_hints
                    or any(
                        hint in lower_key
                        for hint in [
                            "transcript",
                            "transcription",
                            "utterance",
                            "answer",
                        ]
                    )
                ):
                    text_values.append(nested_value)

            for video_id in local_video_ids:
                for text_value in text_values:
                    add_candidate(video_id, text_value)

            for key, nested_value in value.items():
                if not (
                    isinstance(key, str)
                    and Path(key).suffix.lower() in VIDEO_EXTENSIONS
                ):
                    walk(
                        nested_value,
                        forced_video_id=forced_video_id,
                    )

        elif isinstance(value, (list, tuple)):
            for nested_value in value:
                walk(
                    nested_value,
                    forced_video_id=forced_video_id,
                )

    walk(obj)
    return transcript_map


def load_transcript_map(target_df):
    target_basenames = set(
        target_df.loc[
            ~target_df["text_available"],
            "basename",
        ].astype(str).str.lower()
    )

    if not target_basenames:
        return {}

    transcript_map = {}
    roots = [
        CURRENT_UNLABEL_ROOT,
        NEW_DATA_ROOT,
    ]

    for root in roots:
        for csv_path in root.rglob("*.csv"):
            try:
                frame = pd.read_csv(csv_path)
            except Exception:
                continue

            video_columns = [
                column
                for column in frame.columns
                if any(
                    token in column.lower()
                    for token in ["video", "file", "path", "id"]
                )
            ]

            text_columns = [
                column
                for column in frame.columns
                if any(
                    token in column.lower()
                    for token in [
                        "transcript",
                        "transcription",
                        "utterance",
                        "answer",
                        "text",
                    ]
                )
            ]

            for video_column in video_columns:
                for text_column in text_columns:
                    subset = frame[
                        [video_column, text_column]
                    ].dropna()

                    for _, row in subset.iterrows():
                        basename = basename_id(row[video_column])

                        if basename not in target_basenames:
                            continue

                        text = str(row[text_column]).strip()

                        if text and (
                            basename not in transcript_map
                            or len(text) > len(transcript_map[basename])
                        ):
                            transcript_map[basename] = text

    yaml_paths = []

    for root in roots:
        yaml_paths.extend(root.rglob("*.yaml"))
        yaml_paths.extend(root.rglob("*.yml"))

    yaml_paths = sorted(set(yaml_paths))
    per_video_candidates = []
    aggregate_candidates = []

    target_stems = {
        Path(basename).stem.lower()
        for basename in target_basenames
    }

    for yaml_path in yaml_paths:
        inferred_basename = infer_video_basename_from_yaml_path(
            yaml_path
        )

        path_text = str(yaml_path).lower()
        filename_stem = yaml_path.stem.lower()

        if (
            inferred_basename in target_basenames
            or filename_stem in target_stems
            or any(stem in path_text for stem in target_stems)
        ):
            per_video_candidates.append(yaml_path)
        elif any(
            token in yaml_path.name.lower()
            for token in [
                "annotation",
                "transcript",
                "transcription",
                "metadata",
            ]
        ):
            aggregate_candidates.append(yaml_path)

    yaml_errors = []

    for yaml_path in per_video_candidates:
        unresolved = target_basenames - set(transcript_map)

        if not unresolved:
            break

        inferred_basename = infer_video_basename_from_yaml_path(
            yaml_path
        )

        try:
            with yaml_path.open(
                "r",
                encoding="utf-8",
                errors="replace",
            ) as handle:
                obj = yaml.load(
                    handle,
                    Loader=BAHSafeLoader,
                )

            transcript = extract_transcript_from_yaml_object(
                obj
            )

            if transcript:
                if inferred_basename in unresolved:
                    transcript_map[inferred_basename] = transcript
                else:
                    matching_basenames = [
                        basename
                        for basename in unresolved
                        if Path(basename).stem.lower()
                        == yaml_path.stem.lower()
                    ]

                    if len(matching_basenames) == 1:
                        transcript_map[
                            matching_basenames[0]
                        ] = transcript

        except Exception as error:
            yaml_errors.append({
                "yaml_path": str(yaml_path),
                "error_type": type(error).__name__,
                "error": str(error),
            })

    unresolved = target_basenames - set(transcript_map)

    for yaml_path in aggregate_candidates:
        if not unresolved:
            break

        if yaml_path in per_video_candidates:
            continue

        try:
            with yaml_path.open(
                "r",
                encoding="utf-8",
                errors="replace",
            ) as handle:
                obj = yaml.load(
                    handle,
                    Loader=BAHSafeLoader,
                )

            discovered = collect_transcripts_from_object(
                obj,
                unresolved,
            )

            transcript_map.update(discovered)
            unresolved = target_basenames - set(transcript_map)

        except Exception as error:
            yaml_errors.append({
                "yaml_path": str(yaml_path),
                "error_type": type(error).__name__,
                "error": str(error),
            })

    if yaml_errors:
        yaml_error_path = (
            CURRENT_EXTRACTION_DIR
            / "yaml_loading_errors.csv"
        )

        pd.DataFrame(yaml_errors).to_csv(
            yaml_error_path,
            index=False,
        )

        print(
            f"Warning: {len(yaml_errors)} YAML files could "
            f"not be parsed. Details: {yaml_error_path}"
        )

    return transcript_map


transcript_map = load_transcript_map(targets_df)

missing_text_targets = (
    targets_df.loc[
        ~targets_df["text_available"],
        "basename",
    ]
    .astype(str)
    .str.lower()
    .tolist()
)

unresolved_transcripts = [
    basename
    for basename in missing_text_targets
    if basename not in transcript_map
]

if unresolved_transcripts:
    unresolved_path = (
        CURRENT_EXTRACTION_DIR
        / "unresolved_transcripts.txt"
    )

    unresolved_path.write_text(
        "\n".join(unresolved_transcripts),
        encoding="utf-8",
    )

    raise RuntimeError(
        "Could not locate transcripts for these videos:\n"
        + "\n".join(unresolved_transcripts)
        + f"\nSaved to: {unresolved_path}"
    )

if transcript_map:
    transcript_audit_df = pd.DataFrame([
        {
            "basename": basename,
            "length": len(text),
            "preview": text[:240],
        }
        for basename, text in sorted(transcript_map.items())
    ])

    transcript_audit_path = (
        CURRENT_EXTRACTION_DIR
        / "current_transcript_audit.csv"
    )

    transcript_audit_df.to_csv(
        transcript_audit_path,
        index=False,
    )

    display(transcript_audit_df)


# ------------------------------------------------------------
# 3. Write an extraction manifest for the child processes
# ------------------------------------------------------------

manifest_records = []

for _, row in targets_df.sort_values(
    ["split", "order"]
).iterrows():
    video_id = str(row["video_id"])
    video_path = Path(row["video_path"]).resolve()
    basename = str(row["basename"]).lower()

    assert video_path.exists(), video_path

    need_text = not bool(row["text_available"])
    need_audio = not bool(row["audio_available"])
    need_video = not bool(row["video_available"])

    manifest_records.append({
        "split": str(row["split"]),
        "order": int(row["order"]),
        "video_id": video_id,
        "video_path": str(video_path),
        "basename": basename,
        "need_text": need_text,
        "need_audio": need_audio,
        "need_video": need_video,
        "transcript": (
            transcript_map[basename]
            if need_text
            else None
        ),
    })

EXTRACTION_MANIFEST_PATH = (
    CURRENT_EXTRACTION_DIR
    / "current_missing_extraction_manifest.json"
)

with EXTRACTION_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest_records,
        handle,
        ensure_ascii=False,
        indent=2,
    )

print("\nExtraction manifest:", EXTRACTION_MANIFEST_PATH)
print("Manifest records:", len(manifest_records))


# ------------------------------------------------------------
# 4. Generate the standalone PyTorch extraction script
# ------------------------------------------------------------

PYTORCH_EXTRACTION_SCRIPT = (
    CURRENT_EXTRACTION_DIR
    / "extract_current_missing_offline.py"
)

child_script = r"""
from pathlib import Path
import argparse
import json
import math
import os
import pickle
import re
import warnings

warnings.filterwarnings("ignore")

import numpy as np


def atomic_pickle_dump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary_path.open("wb") as handle:
        pickle.dump(obj, handle)

    os.replace(temporary_path, path)


def load_optional_pickle(path):
    path = Path(path)

    if not path.exists():
        return {}

    with path.open("rb") as handle:
        obj = pickle.load(handle)

    if not isinstance(obj, dict):
        raise TypeError(
            f"{path} contains {type(obj)}, expected dict."
        )

    return obj


def clean_text(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def cosine_similarity(a, b):
    denominator = (
        np.linalg.norm(a)
        * np.linalg.norm(b)
        + 1e-12
    )
    return np.dot(a, b) / denominator


def softmax(x):
    shifted = x - np.max(x)
    exp_x = np.exp(shifted)
    return exp_x / (exp_x.sum() + 1e-12)


def max_abs_norm(x):
    return x / (
        np.max(np.abs(x))
        + 1e-12
    )


def linalg_array(array):
    return array / (
        np.linalg.norm(
            array,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )


def stats_pool(x):
    return np.array([
        np.nanmin(x),
        np.nanmax(x),
        np.nanmean(x),
        np.nanstd(x),
    ])


def load_manifest(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        records = json.load(handle)

    if not isinstance(records, list):
        raise TypeError("Extraction manifest must be a list.")

    return records


def extract_text(records, output_path, repo_dir, model_dir):
    import torch
    from sentence_transformers import SentenceTransformer

    model_dir = Path(model_dir)
    repo_dir = Path(repo_dir)

    for path in [
        model_dir,
        repo_dir / "cues" / "ambivalent_cues.json",
        repo_dir / "cues" / "hesitant_cues.json",
    ]:
        if not path.exists():
            raise FileNotFoundError(path)

    with (
        repo_dir / "cues" / "ambivalent_cues.json"
    ).open("r", encoding="utf-8") as handle:
        ambivalent_cues = json.load(handle)

    with (
        repo_dir / "cues" / "hesitant_cues.json"
    ).open("r", encoding="utf-8") as handle:
        hesitant_cues = json.load(handle)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Text device:", device, flush=True)

    kwargs = {
        "device": device,
        "model_kwargs": {
            "torch_dtype": torch.bfloat16,
        },
    }

    try:
        model = SentenceTransformer(
            str(model_dir),
            local_files_only=True,
            trust_remote_code=True,
            **kwargs,
        )
    except TypeError:
        model = SentenceTransformer(
            str(model_dir),
            **kwargs,
        )

    def embed_documents(documents):
        if hasattr(model, "encode_document"):
            embeddings = model.encode_document(documents)
        else:
            print(
                "Warning: encode_document unavailable; using encode.",
                flush=True,
            )
            embeddings = model.encode(documents)

        return np.asarray(embeddings)

    def get_ambivalent_cues(embeddings):
        all_cues = []

        for embedding in embeddings:
            cue_vector = []

            for group in ambivalent_cues:
                answer_embeddings = np.array(
                    list(group["answer embeddings"].values())
                )

                group_vector = np.array([
                    cosine_similarity(embedding, cue_embedding)
                    for cue_embedding in answer_embeddings
                ])

                group_vector = softmax(
                    max_abs_norm(group_vector) * 2
                )
                cue_vector.append(group_vector)

            all_cues.append(np.concatenate(cue_vector))

        return np.array(all_cues)

    def get_hesitant_cues(embeddings):
        all_cues = []

        for embedding in embeddings:
            cue_vector = []

            for group in hesitant_cues:
                answer_embeddings = np.array(
                    list(group["answer embeddings"].values())
                )

                similarities = np.array([
                    cosine_similarity(embedding, cue_embedding)
                    for cue_embedding in answer_embeddings
                ])

                similarities = softmax(
                    max_abs_norm(similarities)
                    * np.sqrt(len(group))
                )

                cue_vector.append(
                    similarities.max() - similarities.mean()
                )

            all_cues.append(np.array(cue_vector))

        return np.array(all_cues)

    def extract_text_features(text):
        text = text.strip()
        full_embedding = embed_documents([text])[0]

        cleaned = clean_text(text).lower()
        words = cleaned.split()
        number_of_words = max(len(words), 1)
        word_set = set(words)

        global_stats = {
            "word_count": len(words),
            "short_pause_count": text.count(","),
            "long_pause_count": sum(
                text.count(character)
                for character in ".!?"
            ),
            "repetition_count": sum(
                words.count(word)
                for word in word_set
                if words.count(word) > 1
            ),
            "adjacent_repetition_count": sum(
                words[index] == words[index + 1]
                for index in range(len(words) - 1)
            ),
            "lexical_diversity": (
                len(word_set) / number_of_words
            ),
        }

        ambivalent = get_ambivalent_cues(
            [full_embedding]
        )[0]

        chunks = [
            chunk.strip()
            for chunk in (
                text
                .replace(",", ".")
                .replace("!", ".")
                .replace("?", ".")
                .split(".")
            )
            if chunk.strip()
        ]

        chunk_embeddings = []
        chunk_stats = []

        for chunk in chunks:
            chunk_words = clean_text(chunk).lower().split()
            embedding = embed_documents([chunk])[0]

            chunk_embeddings.append(embedding)
            chunk_stats.append([
                len(chunk_words),
                len(set(chunk_words))
                / max(len(chunk_words), 1),
                cosine_similarity(
                    embedding,
                    full_embedding,
                ),
            ])

        if chunk_embeddings:
            chunk_embeddings = np.array(chunk_embeddings)
            chunk_stats = np.array(chunk_stats)
        else:
            chunk_embeddings = np.zeros(
                (1, full_embedding.shape[0])
            )
            chunk_stats = np.zeros((1, 3))

        hesitation = get_hesitant_cues(
            chunk_embeddings
        )

        stat_names = [
            "chunk_word_count",
            "chunk_lexical_diversity",
            "chunk_similarity",
        ]

        hesitation_names = (
            [
                f"hesitation_{index}"
                for index in range(hesitation.shape[1])
            ]
            if hesitation.ndim > 1
            else ["hesitation"]
        )

        return {
            "F2LLM": full_embedding,
            **global_stats,
            **{
                name: stats_pool(chunk_stats[:, index])
                for index, name in enumerate(stat_names)
            },
            **{
                name: stats_pool(hesitation[:, index])
                for index, name in enumerate(hesitation_names)
            },
            "ambivalent": ambivalent,
        }

    output = load_optional_pickle(output_path)
    selected_records = [
        record for record in records
        if record["need_text"]
    ]

    print("Text records:", len(selected_records), flush=True)

    for index, record in enumerate(selected_records, start=1):
        video_id = record["video_id"]

        if video_id in output:
            print(
                f"[{index}/{len(selected_records)}] "
                f"SKIP text: {video_id}",
                flush=True,
            )
            continue

        transcript = record.get("transcript")

        if not transcript:
            raise ValueError(
                f"Missing transcript for {video_id}"
            )

        print(
            f"[{index}/{len(selected_records)}] "
            f"Extract text: {video_id}",
            flush=True,
        )

        output[video_id] = extract_text_features(transcript)
        atomic_pickle_dump(output, output_path)

    print("Text output entries:", len(output), flush=True)


def extract_audio(records, output_path, model_dir):
    import librosa
    import torch
    from transformers import (
        HubertModel,
        Wav2Vec2FeatureExtractor,
    )

    model_dir = Path(model_dir)
    if not model_dir.exists():
        raise FileNotFoundError(model_dir)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print("Audio device:", device, flush=True)

    model = HubertModel.from_pretrained(
        str(model_dir),
        local_files_only=True,
    ).to(device).eval()

    feature_extractor = (
        Wav2Vec2FeatureExtractor.from_pretrained(
            str(model_dir),
            local_files_only=True,
        )
    )

    def embed_audio(audio):
        inputs = feature_extractor(
            [audio],
            return_tensors="pt",
            padding=True,
            sampling_rate=16000,
        ).to(device)

        with torch.no_grad():
            embedding = (
                model(
                    **inputs,
                    output_hidden_states=True,
                )
                .last_hidden_state
                .cpu()
                .numpy()
                .mean(axis=1)
            )

        return embedding[0]

    def extract_audio_features(
        video_path,
        sampling_rate=16000,
    ):
        full_audio, _ = librosa.load(
            video_path,
            sr=sampling_rate,
            mono=True,
        )

        hubert_embedding = embed_audio(full_audio)
        audio_seconds = len(full_audio) / sampling_rate

        full_audio = librosa.util.fix_length(
            full_audio,
            size=round(audio_seconds * sampling_rate),
        )

        number_of_chunks = math.ceil(audio_seconds)

        if number_of_chunks <= 0:
            raise ValueError(f"Empty audio: {video_path}")

        audio_reference = np.max(np.abs(full_audio))
        stats = []

        for index in range(number_of_chunks):
            chunk = full_audio[
                index * sampling_rate:
                min(
                    (index + 1) * sampling_rate,
                    len(full_audio),
                )
            ]

            fundamental_frequency = librosa.yin(
                chunk,
                fmin=50,
                fmax=500,
            )

            stats.append([
                librosa.feature.rms(y=chunk).mean(),
                librosa.feature.spectral_centroid(
                    y=chunk,
                    sr=sampling_rate,
                ).mean(),
                librosa.feature.spectral_bandwidth(
                    y=chunk,
                    sr=sampling_rate,
                ).mean(),
                librosa.feature.spectral_rolloff(
                    y=chunk,
                    sr=sampling_rate,
                ).mean(),
                librosa.feature.zero_crossing_rate(
                    chunk
                ).mean(),
                np.mean(
                    librosa.amplitude_to_db(
                        np.abs(chunk),
                        ref=audio_reference,
                    ) < -30
                ),
                np.nanmean(fundamental_frequency),
                np.nanstd(fundamental_frequency),
            ])

        stats = np.array(stats)

        stat_names = [
            "rms",
            "spectral_centroid",
            "spectral_bandwidth",
            "spectral_rolloff",
            "zero_crossing_rate",
            "silence_ratio",
            "pitch_mean",
            "pitch_std",
        ]

        return {
            "HuBERT": hubert_embedding,
            **{
                name: stats_pool(stats[:, index])
                for index, name in enumerate(stat_names)
            },
        }

    output = load_optional_pickle(output_path)
    selected_records = [
        record for record in records
        if record["need_audio"]
    ]

    print("Audio records:", len(selected_records), flush=True)

    for index, record in enumerate(selected_records, start=1):
        video_id = record["video_id"]

        if video_id in output:
            print(
                f"[{index}/{len(selected_records)}] "
                f"SKIP audio: {video_id}",
                flush=True,
            )
            continue

        video_path = Path(record["video_path"])

        if not video_path.exists():
            raise FileNotFoundError(video_path)

        print(
            f"[{index}/{len(selected_records)}] "
            f"Extract audio: {video_id}",
            flush=True,
        )

        output[video_id] = extract_audio_features(
            str(video_path)
        )
        atomic_pickle_dump(output, output_path)

    print("Audio output entries:", len(output), flush=True)


def extract_video(records, output_path, model_dir):
    try:
        import tensorflow as tf
        tf.config.set_visible_devices([], "GPU")
        print(
            "TensorFlow GPU disabled for RetinaFace.",
            flush=True,
        )
    except Exception as error:
        print(
            "TensorFlow GPU-disable note:",
            repr(error),
            flush=True,
        )

    import cv2
    import torch
    from PIL import Image
    from retinaface import RetinaFace
    from skimage import transform as trans
    from tqdm import tqdm
    from transformers import AutoModel, AutoProcessor

    model_dir = Path(model_dir)
    if not model_dir.exists():
        raise FileNotFoundError(model_dir)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    print("Video device:", device, flush=True)

    model = AutoModel.from_pretrained(
        str(model_dir),
        local_files_only=True,
    ).to(device).eval()

    processor = AutoProcessor.from_pretrained(
        str(model_dir),
        local_files_only=True,
    )

    def crop_align_face(image, dimensions=(256, 256)):
        image_numpy = np.array(image.convert("RGB"))
        image_cv2 = cv2.cvtColor(
            image_numpy,
            cv2.COLOR_RGB2BGR,
        )

        response = RetinaFace.detect_faces(
            image_cv2,
            threshold=0.5,
        )

        if (
            not isinstance(response, dict)
            or len(response) == 0
        ):
            return image.resize(dimensions)

        face_key = list(response.keys())[0]
        landmarks = response[face_key]["landmarks"]

        eyes = np.array([
            landmarks["left_eye"],
            landmarks["right_eye"],
        ])
        eyes = eyes[np.argsort(eyes[:, 0])]

        mouths = np.array([
            landmarks["mouth_left"],
            landmarks["mouth_right"],
        ])
        mouths = mouths[np.argsort(mouths[:, 0])]
        nose = np.array(landmarks["nose"])

        source_points = np.array(
            [
                eyes[0],
                eyes[1],
                nose,
                mouths[0],
                mouths[1],
            ],
            dtype=np.float32,
        )

        destination_points = (
            np.array(
                [
                    [38.2946, 51.6963],
                    [73.5318, 51.5014],
                    [56.0252, 71.7366],
                    [41.5493, 92.3655],
                    [70.7299, 92.2041],
                ],
                dtype=np.float32,
            )
            * dimensions[0]
            / 112
        )

        similarity_transform = trans.SimilarityTransform()
        similarity_transform.estimate(
            source_points,
            destination_points,
        )

        matrix = similarity_transform.params[0:2, :]

        aligned_face = cv2.warpAffine(
            image_cv2,
            matrix,
            dimensions,
            borderValue=0.0,
        )

        return Image.fromarray(
            cv2.cvtColor(
                aligned_face,
                cv2.COLOR_BGR2RGB,
            )
        )

    def embed_image(image):
        inputs = processor(
            images=[image],
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            features = model.get_image_features(**inputs)

            if hasattr(features, "pooler_output"):
                features = features.pooler_output

            features = (
                features.detach().cpu().numpy()
            )

        return features[0]

    def extract_video_features(
        video_path,
        frames_per_chunk=24,
    ):
        capture = cv2.VideoCapture(str(video_path))

        if not capture.isOpened():
            raise RuntimeError(
                f"Cannot open video: {video_path}"
            )

        frames = []

        while True:
            success, frame = capture.read()

            if not success:
                break

            frames.append(
                Image.fromarray(
                    cv2.cvtColor(
                        frame,
                        cv2.COLOR_BGR2RGB,
                    )
                )
            )

        capture.release()

        if not frames:
            raise ValueError(
                f"No frames decoded: {video_path}"
            )

        frames = [
            crop_align_face(frame)
            for frame in tqdm(
                frames,
                desc="Extracting faces",
                leave=True,
            )
        ]

        video_embeddings = np.array([
            embed_image(frame)
            for frame in tqdm(
                frames,
                desc="Embedding with SigLip2",
                leave=True,
            )
        ])

        normalized = linalg_array(video_embeddings)

        first_difference = linalg_array(
            np.pad(
                np.diff(
                    normalized,
                    n=1,
                    axis=0,
                ),
                ((0, 1), (0, 0)),
            )
        )

        second_difference = linalg_array(
            np.pad(
                np.diff(
                    normalized,
                    n=2,
                    axis=0,
                ),
                ((0, 2), (0, 0)),
            )
        )

        video_embedding = np.concatenate([
            normalized.mean(axis=0),
            first_difference.mean(axis=0),
            second_difference.mean(axis=0),
        ])

        similarity_matrix = normalized @ normalized.T
        np.fill_diagonal(similarity_matrix, np.nan)

        similarities = np.nanmean(
            similarity_matrix,
            axis=1,
        )

        median = np.median(similarities)
        median_absolute_deviation = np.median(
            np.abs(similarities - median)
        )

        threshold = float(
            np.clip(
                median - 50 * median_absolute_deviation,
                0.0,
                1.0,
            )
        )

        mask = similarities >= threshold

        chunk_indices = [
            np.arange(
                index,
                min(
                    index + frames_per_chunk,
                    video_embeddings.shape[0],
                ),
            )
            for index in range(
                0,
                video_embeddings.shape[0],
                frames_per_chunk,
            )
        ]

        stats = []

        for indices in chunk_indices:
            chunk_similarities = similarities[indices]
            valid = mask[indices]

            if valid.any():
                stats.append([
                    1,
                    valid.mean(),
                    chunk_similarities[valid].mean(),
                ])
            else:
                stats.append([0, 0, 0])

        stats = np.array(stats)

        stat_names = [
            "valid_chunk",
            "valid_ratio",
            "sim_mean",
        ]

        return {
            "SigLip2": video_embedding,
            **{
                name: stats_pool(stats[:, index])
                for index, name in enumerate(stat_names)
            },
        }

    output = load_optional_pickle(output_path)
    selected_records = [
        record for record in records
        if record["need_video"]
    ]

    print("Video records:", len(selected_records), flush=True)

    for index, record in enumerate(selected_records, start=1):
        video_id = record["video_id"]

        if video_id in output:
            print(
                f"[{index}/{len(selected_records)}] "
                f"SKIP video: {video_id}",
                flush=True,
            )
            continue

        video_path = Path(record["video_path"])

        if not video_path.exists():
            raise FileNotFoundError(video_path)

        print(
            f"[{index}/{len(selected_records)}] "
            f"Extract video: {video_id}",
            flush=True,
        )

        output[video_id] = extract_video_features(video_path)
        atomic_pickle_dump(output, output_path)

    print("Video output entries:", len(output), flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--modality",
        required=True,
        choices=["text", "audio", "video"],
    )
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--repo-dir", required=True)
    parser.add_argument("--model-dir", required=True)
    arguments = parser.parse_args()

    records = load_manifest(arguments.manifest)

    if arguments.modality == "text":
        extract_text(
            records=records,
            output_path=arguments.output,
            repo_dir=arguments.repo_dir,
            model_dir=arguments.model_dir,
        )
    elif arguments.modality == "audio":
        extract_audio(
            records=records,
            output_path=arguments.output,
            model_dir=arguments.model_dir,
        )
    else:
        extract_video(
            records=records,
            output_path=arguments.output,
            model_dir=arguments.model_dir,
        )


if __name__ == "__main__":
    main()
"""

PYTORCH_EXTRACTION_SCRIPT.write_text(
    child_script,
    encoding="utf-8",
)

print("PyTorch extraction script:", PYTORCH_EXTRACTION_SCRIPT)


# ------------------------------------------------------------
# 5. Offline subprocess configuration
# ------------------------------------------------------------

def build_offline_pytorch_env():
    environment = os.environ.copy()

    environment.update({
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "HF_DATASETS_OFFLINE": "1",
        "HF_HUB_DISABLE_XET": "1",
        "HF_HUB_DISABLE_TELEMETRY": "1",
        "PYTHONNOUSERSITE": "1",
        "TOKENIZERS_PARALLELISM": "false",
        "TF_CPP_MIN_LOG_LEVEL": "2",
        "CUDA_VISIBLE_DEVICES": FEATURE_GPU,
    })

    environment.pop("HF_ENDPOINT", None)
    if modality == "video":
        environment["TF_USE_LEGACY_KERAS"] = "1"
        environment.pop("KERAS_BACKEND", None)
    else:
        environment.pop("TF_USE_LEGACY_KERAS", None)
    return environment


def stream_subprocess(command, environment):
    print("\nRunning command:")
    print(" ".join(str(item) for item in command))
    print("-" * 90)

    process = subprocess.Popen(
        [str(item) for item in command],
        env=environment,
        cwd=str(REPO_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    output_lines = []

    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    return_code = process.wait()

    if return_code != 0:
        log_path = (
            CURRENT_EXTRACTION_DIR
            / "last_failed_subprocess.log"
        )

        log_path.write_text(
            "".join(output_lines),
            encoding="utf-8",
        )

        raise RuntimeError(
            f"Extraction subprocess failed with exit code "
            f"{return_code}. Log: {log_path}"
        )


# ------------------------------------------------------------
# 6. Run only modalities that are missing
# ------------------------------------------------------------

modality_jobs = {
    "text": {
        "needed": any(
            record["need_text"]
            for record in manifest_records
        ),
        "output": CURRENT_EXTRA_PATHS["text"],
        "model": LOCAL_MODEL_PATHS["f2llm"],
    },
    "audio": {
        "needed": any(
            record["need_audio"]
            for record in manifest_records
        ),
        "output": CURRENT_EXTRA_PATHS["audio"],
        "model": LOCAL_MODEL_PATHS["hubert"],
    },
    "video": {
        "needed": any(
            record["need_video"]
            for record in manifest_records
        ),
        "output": CURRENT_EXTRA_PATHS["video"],
        "model": LOCAL_MODEL_PATHS["siglip2"],
    },
}

for modality in ["text", "audio", "video"]:
    job = modality_jobs[modality]

    if not job["needed"]:
        print(
            f"\nSKIP {modality}: no missing "
            "features for this modality."
        )
        continue

    command = [
        PYTORCH_PYTHON,
        "-u",
        PYTORCH_EXTRACTION_SCRIPT,
        "--modality",
        modality,
        "--manifest",
        EXTRACTION_MANIFEST_PATH,
        "--output",
        job["output"],
        "--repo-dir",
        REPO_DIR,
        "--model-dir",
        job["model"],
    ]

    stream_subprocess(
        command=command,
        environment=build_offline_pytorch_env(
        modality=modality,
    ),
    )


# ------------------------------------------------------------
# 7. Final audit
# ------------------------------------------------------------

audit_rows = []

for modality, path in CURRENT_EXTRA_PATHS.items():
    if path.exists():
        with path.open("rb") as handle:
            dictionary = pickle.load(handle)

        entry_count = len(dictionary)
        keys = set(dictionary)
    else:
        entry_count = 0
        keys = set()

    required_keys = {
        record["video_id"]
        for record in manifest_records
        if record[f"need_{modality}"]
    }

    missing_keys = sorted(required_keys - keys)

    audit_rows.append({
        "modality": modality,
        "output_path": str(path),
        "entries": entry_count,
        "required_entries": len(required_keys),
        "missing_after_extraction": len(missing_keys),
    })

    if missing_keys:
        raise RuntimeError(
            f"{modality} extraction finished but these keys "
            f"are still missing: {missing_keys}"
        )

extraction_summary_df = pd.DataFrame(audit_rows)
display(extraction_summary_df)

extraction_summary_df.to_csv(
    CURRENT_EXTRACTION_DIR
    / "offline_extraction_summary.csv",
    index=False,
)

print(
    "\nPASS: all requested features were extracted "
    "using local models in offline PyTorch subprocesses."
)
print("Now rerun Patch P4 from the beginning.")

In [ ]:
# ============================================================
# Retry only the missing video features with legacy tf_keras
# ============================================================

from pathlib import Path
import pickle


def load_pickle_keys(path):
    path = Path(path)

    if not path.exists():
        return set()

    with path.open("rb") as handle:
        obj = pickle.load(handle)

    assert isinstance(obj, dict), type(obj)
    return set(obj)


required_video_keys = {
    record["video_id"]
    for record in manifest_records
    if record["need_video"]
}

existing_video_keys = load_pickle_keys(
    CURRENT_EXTRA_PATHS["video"]
)

still_missing_video_keys = (
    required_video_keys
    - existing_video_keys
)

print("Required video features:", len(required_video_keys))
print("Already extracted:", len(required_video_keys & existing_video_keys))
print("Still missing:", len(still_missing_video_keys))

if not still_missing_video_keys:
    print("All requested video features already exist.")

else:
    video_environment = build_offline_pytorch_env()

    # RetinaFace compatibility: force Keras 2 only in this child process.
    video_environment["TF_USE_LEGACY_KERAS"] = "1"

    # Avoid inheriting Keras 3 configuration from the prediction notebook.
    video_environment.pop("KERAS_BACKEND", None)

    command = [
        PYTORCH_PYTHON,
        "-u",
        PYTORCH_EXTRACTION_SCRIPT,

        "--modality",
        "video",

        "--manifest",
        EXTRACTION_MANIFEST_PATH,

        "--output",
        CURRENT_EXTRA_PATHS["video"],

        "--repo-dir",
        REPO_DIR,

        "--model-dir",
        LOCAL_MODEL_PATHS["siglip2"],
    ]

    stream_subprocess(
        command=command,
        environment=video_environment,
    )


# Final video-only audit
final_video_keys = load_pickle_keys(
    CURRENT_EXTRA_PATHS["video"]
)

still_missing_video_keys = (
    required_video_keys
    - final_video_keys
)

assert not still_missing_video_keys, (
    "Video extraction remains incomplete:",
    sorted(still_missing_video_keys),
)

print(
    "\nPASS: all requested video features "
    "were extracted successfully."
)

In [ ]:
# ============================================================
# Compare legacy and newly verified features
# ============================================================

from pathlib import Path
from collections import defaultdict
import pickle
import numpy as np
import pandas as pd


LEGACY_DIR = Path(
    "/workspace/outputs/"
    "official_unlabeled_work/extracted_missing"
)

CURRENT_DIR = Path(
    "/workspace/project/"
    "abaw11_append_results/"
    "current_unlabel_extracted_missing"
)

FEATURE_FILES = {
    "text": {
        "old": LEGACY_DIR / "new_text_missing.pkl",
        "new": CURRENT_DIR / "text_missing_current.pkl",
        "primary": "F2LLM",
    },
    "audio": {
        "old": LEGACY_DIR / "new_audio_missing.pkl",
        "new": CURRENT_DIR / "audio_missing_current.pkl",
        "primary": "HuBERT",
    },
    "video": {
        "old": LEGACY_DIR / "new_video_missing.pkl",
        "new": CURRENT_DIR / "video_missing_current.pkl",
        "primary": "SigLip2",
    },
}


def load_pickle(path):
    if not path.exists():
        return None

    with path.open("rb") as handle:
        obj = pickle.load(handle)

    assert isinstance(obj, dict), (path, type(obj))
    return obj


def basename_key(value):
    return Path(
        str(value).replace("\\", "/")
    ).name.lower()


def build_unique_basename_map(dictionary):
    mapping = defaultdict(list)

    for key in dictionary:
        mapping[basename_key(key)].append(key)

    duplicates = {
        basename: keys
        for basename, keys in mapping.items()
        if len(keys) > 1
    }

    if duplicates:
        print(
            "Warning: duplicate basenames:",
            list(duplicates.items())[:5],
        )

    return {
        basename: keys[0]
        for basename, keys in mapping.items()
        if len(keys) == 1
    }


comparison_rows = []

for modality, config in FEATURE_FILES.items():
    old_dict = load_pickle(config["old"])
    new_dict = load_pickle(config["new"])

    print("\n" + "=" * 80)
    print(modality.upper())
    print("=" * 80)
    print("Old file:", config["old"], config["old"].exists())
    print("New file:", config["new"], config["new"].exists())

    if old_dict is None or new_dict is None:
        continue

    old_map = build_unique_basename_map(old_dict)
    new_map = build_unique_basename_map(new_dict)

    old_basenames = set(old_map)
    new_basenames = set(new_map)
    overlap = old_basenames & new_basenames

    print("Old entries:", len(old_dict))
    print("New entries:", len(new_dict))
    print("Basename overlap:", len(overlap))
    print("Only old:", len(old_basenames - new_basenames))
    print("Only new:", len(new_basenames - old_basenames))

    for basename in sorted(overlap):
        old_key = old_map[basename]
        new_key = new_map[basename]

        old_record = old_dict[old_key]
        new_record = new_dict[new_key]

        primary = config["primary"]

        old_array = np.asarray(
            old_record[primary]
        ).reshape(-1)

        new_array = np.asarray(
            new_record[primary]
        ).reshape(-1)

        same_shape = old_array.shape == new_array.shape

        if same_shape:
            difference = old_array - new_array

            max_abs_difference = float(
                np.max(np.abs(difference))
            )

            mean_abs_difference = float(
                np.mean(np.abs(difference))
            )

            cosine_similarity = float(
                np.dot(old_array, new_array)
                / (
                    np.linalg.norm(old_array)
                    * np.linalg.norm(new_array)
                    + 1e-12
                )
            )

            allclose = bool(
                np.allclose(
                    old_array,
                    new_array,
                    rtol=1e-5,
                    atol=1e-6,
                )
            )
        else:
            max_abs_difference = np.nan
            mean_abs_difference = np.nan
            cosine_similarity = np.nan
            allclose = False

        comparison_rows.append({
            "modality": modality,
            "basename": basename,
            "old_key": old_key,
            "new_key": new_key,
            "same_shape": same_shape,
            "old_dimension": old_array.size,
            "new_dimension": new_array.size,
            "allclose": allclose,
            "cosine_similarity": cosine_similarity,
            "max_abs_difference": max_abs_difference,
            "mean_abs_difference": mean_abs_difference,
        })


comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

comparison_path = (
    CURRENT_DIR
    / "legacy_vs_current_feature_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

print("\nSaved comparison to:")
print(comparison_path)

if not comparison_df.empty:
    print("\nSummary:")
    display(
        comparison_df.groupby("modality").agg(
            compared=("basename", "count"),
            identical_or_close=("allclose", "sum"),
            mean_cosine=("cosine_similarity", "mean"),
            minimum_cosine=("cosine_similarity", "min"),
            maximum_difference=(
                "max_abs_difference",
                "max",
            ),
        )
    )

In [ ]:
# ============================================================
# Check legacy Keras compatibility in the PyTorch environment
# ============================================================

import os
import subprocess

legacy_test_env = build_offline_pytorch_env()

# Must be present before the child process imports TensorFlow.
legacy_test_env["TF_USE_LEGACY_KERAS"] = "1"

# Do not inherit the main notebook's Keras 3 configuration.
legacy_test_env.pop("KERAS_BACKEND", None)

# This is only a CPU compatibility test.
legacy_test_env["CUDA_VISIBLE_DEVICES"] = ""

legacy_test_code = r'''
import os
from importlib.metadata import version, PackageNotFoundError

print("TF_USE_LEGACY_KERAS:", os.environ.get("TF_USE_LEGACY_KERAS"))

for package in [
    "tensorflow",
    "keras",
    "tf-keras",
    "retina-face",
]:
    try:
        print(package, version(package))
    except PackageNotFoundError:
        print(package, "NOT INSTALLED")

import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("tf.keras.Model module:", tf.keras.Model.__module__)

assert tf.keras.Model.__module__.startswith("tf_keras"), (
    "TensorFlow is not using legacy tf_keras: "
    + tf.keras.Model.__module__
)

from retinaface import RetinaFace

print("Building RetinaFace model...")
model = RetinaFace.build_model()

print("RetinaFace model type:", type(model))
print("PASS: RetinaFace works with legacy tf_keras.")
'''

result = subprocess.run(
    [
        str(PYTORCH_PYTHON),
        "-u",
        "-c",
        legacy_test_code,
    ],
    env=legacy_test_env,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

assert result.returncode == 0, (
    "Legacy RetinaFace compatibility test failed."
)

In [ ]:
# This cell deliberately does NOT load:
#
# /workspace/outputs/official_unlabeled_work/
#     official_template_manifest.json
#     merged_embeddings/*
#     extracted_missing/*
#
# Those files were created before the actual ABAW11 unlabel videos were
# available and are treated as untrusted.

def load_pickle_dictionary(path):
    path = Path(path)
    assert path.exists(), path

    with path.open("rb") as handle:
        obj = pickle.load(handle)

    assert isinstance(obj, dict), f"{path} contains {type(obj)}, not dict."
    return obj


BASE_FEATURE_PATHS = {
    "text": EMBEDDING_DIR / "text_embeddings.pkl",
    "audio": EMBEDDING_DIR / "audio_embeddings.pkl",
    "video": EMBEDDING_DIR / "video_embeddings.pkl",
}

CURRENT_EXTRACTION_DIR = (
    OUTPUT_DIR / "current_unlabel_extracted_missing"
)
CURRENT_EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)

CURRENT_EXTRA_PATHS = {
    "text": CURRENT_EXTRACTION_DIR / "text_missing_current.pkl",
    "audio": CURRENT_EXTRACTION_DIR / "audio_missing_current.pkl",
    "video": CURRENT_EXTRACTION_DIR / "video_missing_current.pkl",
}

PRIMARY_FEATURE_NAME = {
    "text": "F2LLM",
    "audio": "HuBERT",
    "video": "SigLip2",
}


def load_optional_dictionary(path):
    if not Path(path).exists():
        return {}

    return load_pickle_dictionary(path)


feature_dicts = {}
feature_sources = {}

for modality in ["text", "audio", "video"]:
    base_dict = load_pickle_dictionary(BASE_FEATURE_PATHS[modality])
    current_extra = load_optional_dictionary(CURRENT_EXTRA_PATHS[modality])

    merged = dict(base_dict)
    source_map = {
        key: "released_base_embeddings"
        for key in base_dict
    }

    for key, value in current_extra.items():
        merged[key] = value
        source_map[key] = str(CURRENT_EXTRA_PATHS[modality])

    feature_dicts[modality] = merged
    feature_sources[modality] = source_map

    print(
        f"{modality:5s}: "
        f"base={len(base_dict)}, "
        f"current-extra={len(current_extra)}, "
        f"merged={len(merged)}"
    )


def make_feature_resolver(dictionary, primary_name):
    normalized_map = {}
    basename_map = defaultdict(list)

    for key in dictionary:
        normalized_map[norm_path(key).lower()] = key
        basename_map[basename_id(key)].append(key)

    def resolve(video_id):
        video_id = norm_path(video_id)

        candidates = [
            video_id,
            video_id.removeprefix("data/"),
            f"data/{video_id.removeprefix('data/')}",
            f"Videos/{Path(video_id).name}",
            Path(video_id).name,
        ]

        for candidate in candidates:
            lowered = norm_path(candidate).lower()
            if lowered in normalized_map:
                return normalized_map[lowered]

        matches = basename_map.get(basename_id(video_id), [])

        if len(matches) == 1:
            return matches[0]

        if len(matches) > 1:
            reference = np.asarray(
                dictionary[matches[0]][primary_name]
            ).reshape(-1)

            for other_key in matches[1:]:
                other = np.asarray(
                    dictionary[other_key][primary_name]
                ).reshape(-1)

                if (
                    reference.shape != other.shape
                    or not np.allclose(
                        reference,
                        other,
                        atol=1e-7,
                        rtol=1e-6,
                    )
                ):
                    raise ValueError(
                        f"Conflicting {primary_name} features for "
                        f"{video_id}: {matches}"
                    )

            return matches[0]

        return None

    return resolve


feature_resolvers = {
    modality: make_feature_resolver(
        feature_dicts[modality],
        PRIMARY_FEATURE_NAME[modality],
    )
    for modality in ["text", "audio", "video"]
}

audit_input_rows = []

for _, row in new_split_df.iterrows():
    audit_input_rows.append({
        "split": row["split"],
        "order": row["order"],
        "video_id": row["video_id"],
        "video_path": row["video_path"],
        "basename": row["basename"],
        "required_for_experiment": row["split"] in {"val", "test"},
    })

for _, row in private_df.iterrows():
    audit_input_rows.append({
        "split": "private",
        "order": row["private_order"],
        "video_id": row["video_id"],
        "video_path": row["video_path"],
        "basename": row["basename"],
        "required_for_experiment": True,
    })

audit_rows = []

for item in audit_input_rows:
    audit_row = dict(item)

    for modality in ["text", "audio", "video"]:
        key = feature_resolvers[modality](item["video_id"])

        audit_row[f"{modality}_key"] = key
        audit_row[f"{modality}_available"] = key is not None
        audit_row[f"{modality}_source"] = (
            feature_sources[modality].get(key)
            if key is not None
            else None
        )

    audit_rows.append(audit_row)

feature_coverage_df = pd.DataFrame(audit_rows)
feature_coverage_df.to_csv(
    OUTPUT_DIR / "current_feature_coverage.csv",
    index=False,
)

print("\nFeature coverage by split:")

coverage_summary = []

for split_name, group in feature_coverage_df.groupby("split"):
    coverage_summary.append({
        "split": split_name,
        "videos": len(group),
        "text": int(group["text_available"].sum()),
        "audio": int(group["audio_available"].sum()),
        "video": int(group["video_available"].sum()),
    })

display(pd.DataFrame(coverage_summary))

feature_coverage_df["all_modalities_available"] = (
    feature_coverage_df[[
        "text_available",
        "audio_available",
        "video_available",
    ]].all(axis=1)
)

missing_all_df = feature_coverage_df[
    ~feature_coverage_df["all_modalities_available"]
].copy()

missing_required_df = missing_all_df[
    missing_all_df["required_for_experiment"]
].copy()

missing_private_df = missing_all_df[
    missing_all_df["split"] == "private"
].copy()

missing_all_df.to_csv(
    OUTPUT_DIR / "missing_current_features_all_splits.csv",
    index=False,
)

missing_private_df.to_csv(
    OUTPUT_DIR / "private_missing_for_extraction.csv",
    index=False,
)

print("Missing across all current splits:", len(missing_all_df))
print("Missing in val/test/private:", len(missing_required_df))
print("Missing in current private:", len(missing_private_df))

if not missing_private_df.empty:
    display(
        missing_private_df[[
            "split",
            "video_id",
            "video_path",
            "text_available",
            "audio_available",
            "video_available",
        ]]
    )

if not missing_required_df.empty:
    raise RuntimeError(
        "Required current features are incomplete. "
        "Do not load the legacy new_*_missing.pkl files. "
        "Run Patch P5 to extract only the rows listed in "
        "private_missing_for_extraction.csv, then rerun P4."
    )


# ------------------------------------------------------------------
# Feature schema validation. These variables are required by A5.
# ------------------------------------------------------------------

schema_video_id = (
    feature_coverage_df.loc[
        feature_coverage_df["split"] == "val",
        "video_id",
    ].iloc[0]
)

schema_keys = {
    modality: feature_resolvers[modality](schema_video_id)
    for modality in ["text", "audio", "video"]
}

schema_combined = {
    **feature_dicts["text"][schema_keys["text"]],
    **feature_dicts["video"][schema_keys["video"]],
    **feature_dicts["audio"][schema_keys["audio"]],
}

PRIMARY_KEYS = {
    "F2LLM",
    "HuBERT",
    "SigLip2",
    "set",
    "y",
}

STAT_KEYS = tuple(
    key
    for key in schema_combined.keys()
    if key not in PRIMARY_KEYS
)

EXPECTED_DIMS = {
    "text": int(
        np.asarray(schema_combined["F2LLM"]).size
    ),
    "audio": int(
        np.asarray(schema_combined["HuBERT"]).size
    ),
    "video": int(
        getattr(
            pipeline.video_scaler,
            "n_features_in_",
            np.asarray(schema_combined["SigLip2"]).size,
        )
    ),
    "stats": int(
        getattr(
            pipeline.stats_scaler,
            "n_features_in_",
            102,
        )
    ),
}

schema_errors = []
provenance_rows = []

required_complete_df = feature_coverage_df[
    feature_coverage_df["required_for_experiment"]
].copy()

for _, row in required_complete_df.iterrows():
    video_id = row["video_id"]

    resolved = {
        modality: feature_resolvers[modality](video_id)
        for modality in ["text", "audio", "video"]
    }

    combined = {
        **feature_dicts["text"][resolved["text"]],
        **feature_dicts["video"][resolved["video"]],
        **feature_dicts["audio"][resolved["audio"]],
    }

    missing_stat_keys = [
        key for key in STAT_KEYS
        if key not in combined
    ]

    extra_stat_keys = [
        key for key in combined
        if key not in PRIMARY_KEYS
        and key not in STAT_KEYS
    ]

    dims = {
        "text": np.asarray(
            combined["F2LLM"]
        ).reshape(-1).size,
        "audio": np.asarray(
            combined["HuBERT"]
        ).reshape(-1).size,
        "video": np.asarray(
            combined["SigLip2"]
        ).reshape(-1).size,
        "stats": sum(
            np.asarray(combined[key]).reshape(-1).size
            for key in STAT_KEYS
            if key in combined
        ),
    }

    if (
        missing_stat_keys
        or extra_stat_keys
        or dims != EXPECTED_DIMS
    ):
        schema_errors.append({
            "video_id": video_id,
            "missing_stat_keys": missing_stat_keys,
            "extra_stat_keys": extra_stat_keys,
            "dims": dims,
        })

    provenance_rows.append({
        "split": row["split"],
        "video_id": video_id,
        "basename": row["basename"],
        **{
            f"{modality}_source": (
                feature_sources[modality][resolved[modality]]
            )
            for modality in ["text", "audio", "video"]
        },
    })

if schema_errors:
    pd.DataFrame(schema_errors).to_csv(
        OUTPUT_DIR / "feature_schema_errors.csv",
        index=False,
    )
    raise RuntimeError(
        "Feature dimensions/statistics schema are inconsistent. "
        "See feature_schema_errors.csv."
    )

feature_provenance_df = pd.DataFrame(provenance_rows)
feature_provenance_df.to_csv(
    OUTPUT_DIR / "feature_provenance.csv",
    index=False,
)

print("\nExpected dimensions:", EXPECTED_DIMS)
print("Statistics fields:", len(STAT_KEYS))
print("PASS: val/test/current-private features are complete and compatible.")
print("PASS: no legacy private cache was loaded.")

In [ ]:
def build_feature_arrays(video_ids):
    text_rows, audio_rows, video_rows, stats_rows = [], [], [], []

    for video_id in video_ids:
        text_key = feature_resolvers["text"](video_id)
        audio_key = feature_resolvers["audio"](video_id)
        video_key = feature_resolvers["video"](video_id)
        assert text_key is not None and audio_key is not None and video_key is not None

        combined = {
            **feature_dicts["text"][text_key],
            **feature_dicts["video"][video_key],
            **feature_dicts["audio"][audio_key],
        }

        text_rows.append(np.asarray(combined["F2LLM"]).reshape(-1))
        audio_rows.append(np.asarray(combined["HuBERT"]).reshape(-1))
        video_rows.append(np.asarray(combined["SigLip2"]).reshape(-1))
        stats_rows.append(np.concatenate([
            np.asarray(combined[key]).reshape(-1) for key in STAT_KEYS
        ]))

    X_text_local = np.vstack(text_rows).astype(np.float32)
    X_audio_local = np.vstack(audio_rows).astype(np.float32)
    X_video_local = np.vstack(video_rows).astype(np.float32)
    X_stats_local = np.nan_to_num(
        np.vstack(stats_rows).astype(np.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    # Same normalization as the original Run.ipynb.
    X_text_local /= np.linalg.norm(X_text_local, axis=1, keepdims=True) + 1e-12
    X_audio_local /= np.linalg.norm(X_audio_local, axis=1, keepdims=True) + 1e-12

    assert X_text_local.shape[1] == EXPECTED_DIMS["text"]
    assert X_audio_local.shape[1] == EXPECTED_DIMS["audio"]
    assert X_video_local.shape[1] == EXPECTED_DIMS["video"]
    assert X_stats_local.shape[1] == EXPECTED_DIMS["stats"]
    assert all(np.isfinite(x).all() for x in [
        X_text_local, X_audio_local, X_video_local, X_stats_local
    ])

    return X_text_local, X_audio_local, X_video_local, X_stats_local


current_ids = {
    split: new_split_df.loc[new_split_df.split == split]
        .sort_values("order")["video_id"].tolist()
    for split in ["val", "test"]
}
current_ids["private"] = private_df.sort_values("private_order")["video_id"].tolist()

current_y = {
    split: new_split_df.loc[new_split_df.split == split]
        .sort_values("order")["label"].to_numpy()
    for split in ["val", "test"]
}

for split in ["val", "test"]:
    if pd.isna(current_y[split]).any():
        missing_rows = new_split_df[
            (new_split_df.split == split) & new_split_df.label.isna()
        ]
        display(missing_rows)
        raise RuntimeError(
            f"Current {split} labels were not parsed. Set CSV_VIDEO_COL/CSV_LABEL_COL manually in Cell A2."
        )
    current_y[split] = current_y[split].astype(int)

X_current = {}
for split in ["val", "test", "private"]:
    print(f"Building {split} arrays: {len(current_ids[split])} videos")
    X_current[split] = build_feature_arrays(current_ids[split])
    print(" shapes:", [x.shape for x in X_current[split]])

assert len(current_ids["val"]) == EXPECTED_COUNTS["val"]
assert len(current_ids["test"]) == EXPECTED_COUNTS["test"]
assert len(current_ids["private"]) == EXPECTED_COUNTS["private"]

print("PASS: arrays follow the current split/private manifests, not the old labels files.")

# 9. Released-pipeline inference and calibration

Produces all 15 base probabilities, performs validation-only threshold tuning, locks the selected configuration, evaluates once, and writes auditable submissions.

In [ ]:
def get_base_probabilities(
    pipeline,
    X_text_input,
    X_audio_input,
    X_video_input,
    X_stats_input,
):
    processed_inputs = {
        "Text": np.asarray(X_text_input),
        "Audio": np.asarray(X_audio_input),
        "Video": pipeline.video_pca.transform(
            pipeline.video_scaler.transform(np.asarray(X_video_input))
        ),
        "Stats": pipeline.stats_scaler.transform(np.asarray(X_stats_input)),
    }

    probabilities = []
    for combo_name in COMBO_NAMES:
        modalities = combo_name.split("_")
        X_combo = np.hstack([processed_inputs[m] for m in modalities])
        model = pipeline.models[combo_name]
        model_type = str(pipeline.model_types[combo_name]).lower()

        expected_features = getattr(model, "n_features_in_", None)
        if expected_features is not None:
            assert X_combo.shape[1] == expected_features, (
                combo_name, X_combo.shape[1], expected_features
            )

        if model_type == "mlp":
            prob = np.asarray(model.predict(X_combo, verbose=0)).reshape(-1)
        else:
            prob_matrix = np.asarray(model.predict_proba(X_combo))
            classes = np.asarray(getattr(model, "classes_", [0, 1]))
            positive_index = np.where(classes == 1)[0]
            assert len(positive_index) == 1, (combo_name, classes)
            prob = prob_matrix[:, int(positive_index[0])]

        assert prob.shape == (X_combo.shape[0],)
        assert np.isfinite(prob).all(), combo_name
        probabilities.append(prob)

    result = np.column_stack(probabilities).astype(np.float64)
    assert result.shape[1] == 15
    return result


def apply_hard_vote(probabilities, thresholds, weights, final_threshold=0.5):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    assert probabilities.shape[1] == thresholds.size == weights.size
    votes = (probabilities >= thresholds.reshape(1, -1)).astype(np.float64)
    scores = (votes @ weights) / (weights.sum() + 1e-12)
    predictions = (scores >= float(final_threshold)).astype(int)
    return predictions, scores, votes


P_current = {}
for split in ["val", "test", "private"]:
    P_current[split] = get_base_probabilities(pipeline, *X_current[split])
    print(split, P_current[split].shape)

# Reproduce pipeline.predict exactly on current validation before any retuning.
baseline_val_pred, baseline_val_score, _ = apply_hard_vote(
    P_current["val"], ORIGINAL_THRESHOLDS, ORIGINAL_WEIGHTS, 0.5
)
pipeline_val_pred = pipeline.predict(*X_current["val"])
assert np.array_equal(baseline_val_pred, pipeline_val_pred), (
    "Base-probability path does not reproduce pipeline.predict."
)

baseline_val_f1 = f1_score(current_y["val"], baseline_val_pred, average="macro")
print("Current validation original CALMAH Macro-F1:", baseline_val_f1)

joblib.dump(
    {
        "combo_names": COMBO_NAMES,
        "original_weights": ORIGINAL_WEIGHTS,
        "original_thresholds": ORIGINAL_THRESHOLDS,
        "ids": current_ids,
        "y_val": current_y["val"],
        "y_test": current_y["test"],
        "P_val": P_current["val"],
        "P_test": P_current["test"],
        "P_private": P_current["private"],
    },
    OUTPUT_DIR / "current_base_probabilities.joblib",
)
print("PASS: probability matrices were generated and cached.")

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit


def fast_macro_f1_binary(y_true, pred_matrix):
    y_true = np.asarray(y_true).astype(bool)[None, :]
    pred = np.asarray(pred_matrix).astype(bool)
    tp = np.sum(pred & y_true, axis=1)
    fp = np.sum(pred & ~y_true, axis=1)
    fn = np.sum(~pred & y_true, axis=1)
    tn = np.sum(~pred & ~y_true, axis=1)
    f1_pos = 2 * tp / np.maximum(2 * tp + fp + fn, 1e-12)
    f1_neg = 2 * tn / np.maximum(2 * tn + fp + fn, 1e-12)
    return 0.5 * (f1_pos + f1_neg)


def regularized_threshold_search_val_only(
    P_val,
    y_val,
    original_thresholds,
    weights,
    *,
    n_trials=20000,
    batch_size=512,
    noise_scale=0.025,
    max_shift=0.06,
    tune_final_threshold=False,
    max_final_shift=0.05,
    n_stability_splits=32,
    stability_fraction=0.80,
    stability_penalty=0.20,
    distance_penalty=0.015,
    final_distance_penalty=0.020,
    random_state=2026,
):
    """
    Search uses current validation labels only.

    Objective = full validation Macro-F1
                + 0.5 * (mean stratified-subsample F1 - full validation F1)
                - stability_penalty * subsample F1 std
                - threshold-distance penalty
                - optional final-threshold distance penalty.

    Models with zero released weight are not tuned.
    """
    rng = np.random.default_rng(random_state)
    P_val = np.asarray(P_val, dtype=np.float32)
    y_val = np.asarray(y_val).reshape(-1).astype(int)
    original_thresholds = np.asarray(original_thresholds, dtype=np.float32)
    weights = np.asarray(weights, dtype=np.float32)

    assert P_val.shape == (len(y_val), len(original_thresholds))
    active = weights > 1e-12
    assert active.any()

    splitter = StratifiedShuffleSplit(
        n_splits=n_stability_splits,
        train_size=stability_fraction,
        random_state=random_state,
    )
    stability_indices = [train_idx for train_idx, _ in splitter.split(P_val, y_val)]

    best = None
    total_done = 0
    while total_done < n_trials:
        batch = min(batch_size, n_trials - total_done)
        candidates = np.repeat(original_thresholds[None, :], batch, axis=0)

        perturbation = rng.normal(0.0, noise_scale, size=(batch, active.sum())).astype(np.float32)
        perturbation = np.clip(perturbation, -max_shift, max_shift)
        candidates[:, active] += perturbation
        candidates = np.clip(candidates, 0.05, 0.95)
        candidates[:, ~active] = original_thresholds[None, ~active]

        if tune_final_threshold:
            final_candidates = 0.5 + rng.normal(0.0, max_final_shift / 2.0, size=batch)
            final_candidates = np.clip(final_candidates, 0.5 - max_final_shift, 0.5 + max_final_shift)
        else:
            final_candidates = np.full(batch, 0.5, dtype=np.float32)

        # Always include the exact released CALMAH configuration once.
        if total_done == 0:
            candidates[0] = original_thresholds
            final_candidates[0] = 0.5

        votes = (P_val[None, :, :] >= candidates[:, None, :]).astype(np.float32)
        scores = np.einsum("bnm,m->bn", votes, weights) / (weights.sum() + 1e-12)
        predictions = scores >= final_candidates[:, None]

        full_f1 = fast_macro_f1_binary(y_val, predictions)
        stability_f1 = np.vstack([
            fast_macro_f1_binary(y_val[index], predictions[:, index])
            for index in stability_indices
        ])
        stability_mean = stability_f1.mean(axis=0)
        stability_std = stability_f1.std(axis=0)

        normalized_distance = np.mean(
            ((candidates[:, active] - original_thresholds[None, active]) / max_shift) ** 2,
            axis=1,
        )
        final_distance = ((final_candidates - 0.5) / max_final_shift) ** 2 if tune_final_threshold else 0.0

        objective = (
            full_f1
            + 0.5 * (stability_mean - full_f1)
            - stability_penalty * stability_std
            - distance_penalty * normalized_distance
            - final_distance_penalty * final_distance
        )

        index = int(np.argmax(objective))
        candidate_result = {
            "objective": float(objective[index]),
            "val_f1": float(full_f1[index]),
            "stability_mean": float(stability_mean[index]),
            "stability_std": float(stability_std[index]),
            "thresholds": candidates[index].astype(np.float64).copy(),
            "final_threshold": float(final_candidates[index]),
            "tune_final_threshold": bool(tune_final_threshold),
        }
        if best is None or candidate_result["objective"] > best["objective"]:
            best = candidate_result

        total_done += batch

    print("Objective:", best["objective"])
    print("Validation Macro-F1:", best["val_f1"])
    print("Stability mean/std:", best["stability_mean"], best["stability_std"])
    print("Final vote threshold:", best["final_threshold"])
    print("\nPer-model threshold changes:")
    for name, weight, old, new in zip(
        COMBO_NAMES, weights, original_thresholds, best["thresholds"]
    ):
        marker = "tuned" if weight > 1e-12 else "fixed: zero released weight"
        print(f"{name:30s} w={weight:.5f} old={old:.4f} new={new:.4f} delta={new-old:+.4f} [{marker}]")
    return best


# Primary method: per-model thresholds only; released weights and final threshold 0.5.
RETUNED_FIXED_FINAL = regularized_threshold_search_val_only(
    P_current["val"],
    current_y["val"],
    ORIGINAL_THRESHOLDS,
    ORIGINAL_WEIGHTS,
    tune_final_threshold=False,
    random_state=2026,
)

# Optional separate private trial. It is not the primary method and is not used
# for the labelled-test decision below.
RUN_OPTIONAL_FINAL_THRESHOLD_SEARCH = True
RETUNED_WITH_FINAL = None
if RUN_OPTIONAL_FINAL_THRESHOLD_SEARCH:
    RETUNED_WITH_FINAL = regularized_threshold_search_val_only(
        P_current["val"],
        current_y["val"],
        ORIGINAL_THRESHOLDS,
        ORIGINAL_WEIGHTS,
        tune_final_threshold=True,
        random_state=2027,
    )

search_state = {
    "combo_names": COMBO_NAMES,
    "weights": ORIGINAL_WEIGHTS,
    "original_thresholds": ORIGINAL_THRESHOLDS,
    "primary": RETUNED_FIXED_FINAL,
    "optional_with_final": RETUNED_WITH_FINAL,
    "search_data": "current_validation_only",
    "public_test_used_in_search": False,
}
joblib.dump(search_state, OUTPUT_DIR / "validation_only_threshold_search.joblib")
print("Saved:", OUTPUT_DIR / "validation_only_threshold_search.joblib")

In [ ]:
# Lock before reading any current public-test score.
LOCKED_PRIMARY = {
    "name": "retuned_per_model_fixed_final_0.5",
    "combo_names": COMBO_NAMES,
    "weights": ORIGINAL_WEIGHTS.copy(),
    "thresholds": RETUNED_FIXED_FINAL["thresholds"].copy(),
    "final_threshold": 0.5,
    "selected_using": "current validation only",
    "public_test_used_for_selection": False,
}
joblib.dump(LOCKED_PRIMARY, OUTPUT_DIR / "locked_primary_configuration.joblib")
print("Primary configuration locked and saved before public-test reporting.")


def evaluate_configuration(name, probabilities, labels, thresholds, weights, final_threshold):
    prediction, score, _ = apply_hard_vote(
        probabilities, thresholds, weights, final_threshold
    )
    macro_f1 = f1_score(labels, prediction, average="macro")
    return {
        "method": name,
        "macro_f1": float(macro_f1),
        "predicted_0": int((prediction == 0).sum()),
        "predicted_1": int((prediction == 1).sum()),
        "prediction": prediction,
        "score": score,
    }


report_rows = []
for split in ["val", "test"]:
    baseline = evaluate_configuration(
        "released_CALMAH",
        P_current[split],
        current_y[split],
        ORIGINAL_THRESHOLDS,
        ORIGINAL_WEIGHTS,
        0.5,
    )
    primary = evaluate_configuration(
        LOCKED_PRIMARY["name"],
        P_current[split],
        current_y[split],
        LOCKED_PRIMARY["thresholds"],
        LOCKED_PRIMARY["weights"],
        LOCKED_PRIMARY["final_threshold"],
    )
    for result in [baseline, primary]:
        report_rows.append({
            "split": split,
            "method": result["method"],
            "macro_f1": result["macro_f1"],
            "predicted_0": result["predicted_0"],
            "predicted_1": result["predicted_1"],
        })

report_df = pd.DataFrame(report_rows)
display(report_df)
report_df.to_csv(OUTPUT_DIR / "locked_public_test_report.csv", index=False)

locked_test_pred, _, _ = apply_hard_vote(
    P_current["test"],
    LOCKED_PRIMARY["thresholds"],
    LOCKED_PRIMARY["weights"],
    LOCKED_PRIMARY["final_threshold"],
)
print("\nLocked primary public-test classification report:")
print(classification_report(current_y["test"], locked_test_pred, digits=4))
print("\nDo not run another threshold/weight search in response to this test result.")

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_submission(trial_index, description, thresholds, weights, final_threshold):
    prediction, vote_score, votes = apply_hard_vote(
        P_current["private"], thresholds, weights, final_threshold
    )
    submission_ids = private_df.sort_values("private_order")["submission_id"].tolist()
    assert len(submission_ids) == len(prediction) == EXPECTED_COUNTS["private"]
    assert len(set(submission_ids)) == len(submission_ids)
    assert set(np.unique(prediction)).issubset({0, 1})

    trial_path = OUTPUT_DIR / f"trial-{trial_index}.txt"
    with trial_path.open("w", encoding="utf-8", newline="\n") as handle:
        for video_id, label in zip(submission_ids, prediction):
            handle.write(f"{video_id},{int(label)}\n")

    # Strict read-back validation.
    written_lines = [
        line.rstrip("\r\n")
        for line in trial_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    assert len(written_lines) == EXPECTED_COUNTS["private"]
    written_ids, written_labels = [], []
    for line_number, line in enumerate(written_lines, start=1):
        fields = line.rsplit(",", 1)
        assert len(fields) == 2, (line_number, line)
        video_id, label = fields
        assert label in {"0", "1"}, (line_number, label)
        written_ids.append(video_id)
        written_labels.append(int(label))
    assert written_ids == submission_ids
    np.testing.assert_array_equal(written_labels, prediction)

    audit = private_df.sort_values("private_order").copy()
    audit["method"] = description
    audit["vote_score"] = vote_score
    audit["prediction"] = prediction
    for model_index, combo_name in enumerate(COMBO_NAMES):
        safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", combo_name)
        audit[f"prob_{safe_name}"] = P_current["private"][:, model_index]
        audit[f"vote_{safe_name}"] = votes[:, model_index].astype(int)
    audit_path = OUTPUT_DIR / f"trial-{trial_index}-audit.csv"
    audit.to_csv(audit_path, index=False)

    config_path = OUTPUT_DIR / f"trial-{trial_index}-config.joblib"
    joblib.dump({
        "description": description,
        "combo_names": COMBO_NAMES,
        "thresholds": np.asarray(thresholds),
        "weights": np.asarray(weights),
        "final_threshold": float(final_threshold),
        "submission_ids": submission_ids,
        "prediction": prediction,
    }, config_path)

    return {
        "trial": trial_index,
        "description": description,
        "path": str(trial_path),
        "sha256": sha256_file(trial_path),
        "predicted_0": int((prediction == 0).sum()),
        "predicted_1": int((prediction == 1).sum()),
    }


submission_summaries = []
submission_summaries.append(write_submission(
    0,
    "released CALMAH baseline",
    ORIGINAL_THRESHOLDS,
    ORIGINAL_WEIGHTS,
    0.5,
))
submission_summaries.append(write_submission(
    1,
    "validation-only per-model threshold retuning; final threshold fixed at 0.5",
    LOCKED_PRIMARY["thresholds"],
    LOCKED_PRIMARY["weights"],
    LOCKED_PRIMARY["final_threshold"],
))
if RETUNED_WITH_FINAL is not None:
    submission_summaries.append(write_submission(
        2,
        "validation-only per-model and final-threshold retuning; exploratory",
        RETUNED_WITH_FINAL["thresholds"],
        ORIGINAL_WEIGHTS,
        RETUNED_WITH_FINAL["final_threshold"],
    ))

submission_summary_df = pd.DataFrame(submission_summaries)
display(submission_summary_df)
submission_summary_df.to_csv(OUTPUT_DIR / "submission_summary.csv", index=False)

# Ensure no video removed from the current private set appears in a submission.
current_private_basenames = set(private_df["basename"])
for summary in submission_summaries:
    lines = Path(summary["path"]).read_text(encoding="utf-8").splitlines()
    submitted_basenames = {basename_id(line.rsplit(",", 1)[0]) for line in lines if line.strip()}
    assert submitted_basenames == current_private_basenames

print("\nPASS: every trial contains exactly the current 152 private videos in one verified order.")
print("PASS: labels/unlabeled.txt was not used for prediction or submission ordering.")
print("\nPrimary experimental submission:", OUTPUT_DIR / "trial-1.txt")
print("Baseline control submission:", OUTPUT_DIR / "trial-0.txt")
if RETUNED_WITH_FINAL is not None:
    print("Exploratory submission:", OUTPUT_DIR / "trial-2.txt")

# 10. Retrained pipeline

Fits preprocessing on training data only, evaluates candidate base learners, resumes per-combination training, locks the pipeline, and generates test/private predictions.

In [ ]:
from pathlib import Path
from collections import OrderedDict
import gc
import hashlib
import itertools
import json
import os
import pickle
import random
import shutil
import sys
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report,
    f1_score,
    log_loss,
)
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

import tensorflow as tf
import keras

from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.layers import (
    Input,
    Dense,
    Dropout,
    GaussianNoise,
    BatchNormalization,
    Activation,
)
from keras.models import Model

warnings.filterwarnings("ignore")

REQUIRED_GLOBALS = [
    "REPO_DIR",
    "OUTPUT_DIR",
    "EXPECTED_COUNTS",
    "new_split_df",
    "private_df",
    "feature_dicts",
    "feature_resolvers",
    "STAT_KEYS",
    "EXPECTED_DIMS",
    "build_feature_arrays",
    "ORIGINAL_WEIGHTS",
]

missing_globals = [
    name for name in REQUIRED_GLOBALS
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Run A1/P1/P2/A3/P4/A5 first. Missing variables: "
        + ", ".join(missing_globals)
    )

if int(keras.__version__.split(".")[0]) < 3:
    raise RuntimeError(
        f"Standalone Keras 3 is required, found {keras.__version__}."
    )

if tf.keras.Model.__module__.startswith("tf_keras"):
    raise RuntimeError(
        "The prediction kernel is currently using legacy tf_keras. "
        "Restart the TensorFlow kernel, activate standalone Keras 3, "
        "then rerun the clean ABAW11 cells before this retraining block."
    )

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception as error:
    print("Determinism note:", repr(error))

RETRAIN_ROOT = (
    Path(REPO_DIR)
    / "abaw11_retrained_base_models"
)

RESULT_DIR = RETRAIN_ROOT / "combo_results"
MODEL_DIR = RETRAIN_ROOT

RETRAIN_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RETRAIN = False

# Keep the exact released CALMAH fusion weights in this experiment.
# Later soft fusion/stacking experiments should use a different directory.
PRIMARY_ENSEMBLE_WEIGHTS = np.asarray(
    ORIGINAL_WEIGHTS,
    dtype=np.float64,
).copy()

MODALITY_ORDER = [
    "Text",
    "Audio",
    "Video",
    "Stats",
]

RETRAIN_COMBOS = []

for combination_size in range(1, len(MODALITY_ORDER) + 1):
    RETRAIN_COMBOS.extend(
        itertools.combinations(
            MODALITY_ORDER,
            combination_size,
        )
    )

RETRAIN_COMBO_NAMES = [
    "_".join(combo)
    for combo in RETRAIN_COMBOS
]

assert len(RETRAIN_COMBO_NAMES) == 15
assert PRIMARY_ENSEMBLE_WEIGHTS.shape == (15,)

if "COMBO_NAMES" in globals():
    assert list(COMBO_NAMES) == RETRAIN_COMBO_NAMES, (
        COMBO_NAMES,
        RETRAIN_COMBO_NAMES,
    )

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Keras model module:", keras.Model.__module__)
print("Retraining directory:", RETRAIN_ROOT)
print("Combinations:", RETRAIN_COMBO_NAMES)
print("Released weight sum:", PRIMARY_ENSEMBLE_WEIGHTS.sum())

In [ ]:
def sorted_split_rows(split_name):
    rows = (
        new_split_df.loc[
            new_split_df["split"] == split_name
        ]
        .sort_values("order")
        .reset_index(drop=True)
    )

    expected = EXPECTED_COUNTS[split_name]

    assert len(rows) == expected, (
        split_name,
        len(rows),
        expected,
    )

    assert rows["video_id"].is_unique
    assert rows["label"].notna().all()

    return rows


train_rows_current = sorted_split_rows("train")
val_rows_current = sorted_split_rows("val")

train_ids_current = train_rows_current["video_id"].tolist()
val_ids_current = val_rows_current["video_id"].tolist()

y_train_current = (
    train_rows_current["label"]
    .astype(int)
    .to_numpy()
)

y_val_current = (
    val_rows_current["label"]
    .astype(int)
    .to_numpy()
)

assert set(np.unique(y_train_current)).issubset({0, 1})
assert set(np.unique(y_val_current)).issubset({0, 1})

# Verify that all train and validation videos have all modalities.
for split_name, ids in {
    "train": train_ids_current,
    "val": val_ids_current,
}.items():
    missing_rows = []

    for video_id in ids:
        availability = {
            modality: (
                feature_resolvers[modality](video_id)
                is not None
            )
            for modality in ["text", "audio", "video"]
        }

        if not all(availability.values()):
            missing_rows.append({
                "split": split_name,
                "video_id": video_id,
                **availability,
            })

    if missing_rows:
        missing_path = (
            RETRAIN_ROOT
            / f"missing_{split_name}_features.csv"
        )

        pd.DataFrame(missing_rows).to_csv(
            missing_path,
            index=False,
        )

        raise RuntimeError(
            f"{split_name} feature coverage is incomplete. "
            f"See {missing_path}"
        )

print("Building current train arrays...")
X_train_raw = build_feature_arrays(
    train_ids_current
)

print("Building current validation arrays...")
X_val_raw = build_feature_arrays(
    val_ids_current
)

(
    X_text_train_current,
    X_audio_train_current,
    X_video_train_current,
    X_stats_train_current,
) = X_train_raw

(
    X_text_val_current,
    X_audio_val_current,
    X_video_val_current,
    X_stats_val_current,
) = X_val_raw

assert len(y_train_current) == X_text_train_current.shape[0]
assert len(y_val_current) == X_text_val_current.shape[0]

# Fit all learned preprocessing using current train only.
stats_scaler_current = StandardScaler().fit(
    X_stats_train_current
)

video_scaler_current = StandardScaler().fit(
    X_video_train_current
)

X_video_train_scaled = (
    video_scaler_current.transform(
        X_video_train_current
    )
)

PCA_COMPONENTS = 512

assert PCA_COMPONENTS <= min(
    X_video_train_scaled.shape
), (
    PCA_COMPONENTS,
    X_video_train_scaled.shape,
)

video_pca_current = PCA(
    n_components=PCA_COMPONENTS,
    random_state=SEED,
).fit(X_video_train_scaled)

X_base_current = {
    "Text": {
        "train": X_text_train_current,
        "val": X_text_val_current,
    },
    "Audio": {
        "train": X_audio_train_current,
        "val": X_audio_val_current,
    },
    "Video": {
        "train": video_pca_current.transform(
            video_scaler_current.transform(
                X_video_train_current
            )
        ).astype(np.float32),
        "val": video_pca_current.transform(
            video_scaler_current.transform(
                X_video_val_current
            )
        ).astype(np.float32),
    },
    "Stats": {
        "train": stats_scaler_current.transform(
            X_stats_train_current
        ).astype(np.float32),
        "val": stats_scaler_current.transform(
            X_stats_val_current
        ).astype(np.float32),
    },
}

for modality in MODALITY_ORDER:
    print(
        modality,
        "train",
        X_base_current[modality]["train"].shape,
        "val",
        X_base_current[modality]["val"].shape,
    )

print("\nClass distributions:")
print(
    "Train:",
    dict(zip(*np.unique(
        y_train_current,
        return_counts=True,
    )))
)
print(
    "Val:",
    dict(zip(*np.unique(
        y_val_current,
        return_counts=True,
    )))
)

# Save the exact manifests used for this experiment.
train_rows_current.to_csv(
    RETRAIN_ROOT / "train_manifest.csv",
    index=False,
)

val_rows_current.to_csv(
    RETRAIN_ROOT / "val_manifest.csv",
    index=False,
)

joblib.dump(
    {
        "stats_scaler": stats_scaler_current,
        "video_scaler": video_scaler_current,
        "video_pca": video_pca_current,
        "pca_components": PCA_COMPONENTS,
        "seed": SEED,
    },
    RETRAIN_ROOT / "current_preprocessing.joblib",
)

print(
    "\nPASS: preprocessing was fitted on current train only. "
    "Current test has not been loaded."
)

In [ ]:
def get_best_macro_f1_threshold(
    probabilities,
    labels,
    thresholds=None,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    ).reshape(-1)

    labels = np.asarray(
        labels,
        dtype=int,
    ).reshape(-1)

    if thresholds is None:
        thresholds = np.arange(
            0.01,
            1.00,
            0.01,
        )

    best_threshold = 0.5
    best_f1 = -np.inf

    for threshold in thresholds:
        prediction = (
            probabilities >= threshold
        ).astype(int)

        score = f1_score(
            labels,
            prediction,
            average="macro",
            zero_division=0,
        )

        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)

    return best_threshold, best_f1


def safe_binary_log_loss(labels, probabilities):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    ).reshape(-1)

    probabilities = np.clip(
        probabilities,
        1e-7,
        1.0 - 1e-7,
    )

    return float(
        log_loss(
            np.asarray(labels).reshape(-1),
            probabilities,
            labels=[0, 1],
        )
    )


def build_original_CALMAH_mlp(
    input_dim,
    seed,
):
    keras.utils.set_random_seed(seed)

    inputs = Input(
        shape=(input_dim,),
        name="input",
    )

    x = GaussianNoise(
        0.025,
        name="gaussian_noise",
    )(inputs)

    x = Dense(
        max(1, input_dim // 4),
        use_bias=False,
        name="dense_projection",
    )(x)

    x = BatchNormalization(
        name="batch_norm",
    )(x)

    x = Activation(
        "relu",
        name="projection_relu",
    )(x)

    x = Dropout(
        0.2,
        name="dropout",
    )(x)

    x = Dense(
        max(1, input_dim // 8),
        activation="relu",
        name="dense_hidden",
    )(x)

    x = Dense(
        16,
        activation="relu",
        name="dense_bottleneck",
    )(x)

    outputs = Dense(
        1,
        activation="sigmoid",
        name="prediction",
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="CALMAH_MLP",
    )

    # No metric state is saved. Selection is calculated explicitly below.
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-3,
        ),
        loss="binary_crossentropy",
    )

    return model


def candidate_metrics(
    name,
    train_probabilities,
    val_probabilities,
    y_train,
    y_val,
):
    threshold, val_macro_f1 = (
        get_best_macro_f1_threshold(
            val_probabilities,
            y_val,
        )
    )

    train_prediction = (
        np.asarray(train_probabilities)
        >= threshold
    ).astype(int)

    train_macro_f1 = f1_score(
        y_train,
        train_prediction,
        average="macro",
        zero_division=0,
    )

    return {
        "algorithm": name,
        "threshold": float(threshold),
        "val_bce": safe_binary_log_loss(
            y_val,
            val_probabilities,
        ),
        "train_macro_f1": float(
            train_macro_f1
        ),
        "val_macro_f1": float(
            val_macro_f1
        ),
    }


def atomic_joblib_dump(obj, path):
    path = Path(path)
    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    joblib.dump(obj, temporary_path)
    os.replace(temporary_path, path)


def train_one_combination(
    combo_index,
    combo_name,
    X_train,
    X_val,
    y_train,
    y_val,
):
    result_path = (
        RESULT_DIR
        / f"result_{combo_name}.joblib"
    )

    keras_model_path = (
        MODEL_DIR
        / f"model_{combo_name}.keras"
    )

    tree_model_path = (
        MODEL_DIR
        / f"model_{combo_name}.joblib"
    )

    if FORCE_RETRAIN:
        for path in [
            result_path,
            keras_model_path,
            tree_model_path,
        ]:
            if path.exists():
                if path.is_dir():
                    shutil.rmtree(path)
                else:
                    path.unlink()

    if result_path.exists():
        result = joblib.load(result_path)
        winner = result["winner"]

        expected_model_path = (
            keras_model_path
            if winner == "mlp"
            else tree_model_path
        )

        if expected_model_path.exists():
            print(
                f"[{combo_index + 1}/15] "
                f"RESUME {combo_name}: {winner}"
            )
            return result

    input_dim = X_train.shape[1]
    combo_seed = SEED + combo_index

    print("\n" + "=" * 100)
    print(
        f"[{combo_index + 1}/15] "
        f"Training {combo_name} | "
        f"input_dim={input_dim}"
    )
    print("=" * 100)

    # --------------------------------------------------------
    # Candidate 1: MLP
    # --------------------------------------------------------

    mlp_model = build_original_CALMAH_mlp(
        input_dim=input_dim,
        seed=combo_seed,
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True,
            mode="min",
            verbose=0,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=0,
        ),
    ]

    history = mlp_model.fit(
        X_train,
        y_train.reshape(-1, 1),
        validation_data=(
            X_val,
            y_val.reshape(-1, 1),
        ),
        epochs=1000,
        batch_size=32,
        verbose=0,
        callbacks=callbacks,
        shuffle=True,
    )

    mlp_train_probabilities = (
        mlp_model.predict(
            X_train,
            verbose=0,
        ).reshape(-1)
    )

    mlp_val_probabilities = (
        mlp_model.predict(
            X_val,
            verbose=0,
        ).reshape(-1)
    )

    mlp_metrics = candidate_metrics(
        "mlp",
        mlp_train_probabilities,
        mlp_val_probabilities,
        y_train,
        y_val,
    )

    mlp_metrics["epochs_trained"] = len(
        history.history["loss"]
    )

    # --------------------------------------------------------
    # Candidate 2: LightGBM random forest
    # --------------------------------------------------------

    colsample = min(
        1.0,
        input_dim ** -0.5,
    )

    rf_model = LGBMClassifier(
        boosting_type="rf",
        n_estimators=1000,
        max_depth=50,
        min_child_samples=50,
        class_weight="balanced",
        subsample=0.632,
        subsample_freq=1,
        colsample_bytree=colsample,
        random_state=combo_seed,
        n_jobs=-1,
        verbose=-1,
        max_bin=255,
    )

    rf_model.fit(
        X_train,
        y_train,
    )

    rf_train_probabilities = (
        rf_model.predict_proba(
            X_train
        )[:, 1]
    )

    rf_val_probabilities = (
        rf_model.predict_proba(
            X_val
        )[:, 1]
    )

    rf_metrics = candidate_metrics(
        "rf",
        rf_train_probabilities,
        rf_val_probabilities,
        y_train,
        y_val,
    )

    # --------------------------------------------------------
    # Candidate 3: LightGBM GBDT
    # --------------------------------------------------------

    gbdt_model = LGBMClassifier(
        boosting_type="gbdt",
        n_estimators=1000,
        learning_rate=1e-3,
        max_depth=50,
        min_child_samples=50,
        class_weight="balanced",
        subsample=0.632,
        subsample_freq=1,
        colsample_bytree=colsample,
        random_state=combo_seed,
        n_jobs=-1,
        verbose=-1,
        max_bin=255,
    )

    gbdt_model.fit(
        X_train,
        y_train,
    )

    gbdt_train_probabilities = (
        gbdt_model.predict_proba(
            X_train
        )[:, 1]
    )

    gbdt_val_probabilities = (
        gbdt_model.predict_proba(
            X_val
        )[:, 1]
    )

    gbdt_metrics = candidate_metrics(
        "gbdt",
        gbdt_train_probabilities,
        gbdt_val_probabilities,
        y_train,
        y_val,
    )

    metrics_by_algorithm = {
        "mlp": mlp_metrics,
        "rf": rf_metrics,
        "gbdt": gbdt_metrics,
    }

    train_probabilities_by_algorithm = {
        "mlp": mlp_train_probabilities,
        "rf": rf_train_probabilities,
        "gbdt": gbdt_train_probabilities,
    }

    val_probabilities_by_algorithm = {
        "mlp": mlp_val_probabilities,
        "rf": rf_val_probabilities,
        "gbdt": gbdt_val_probabilities,
    }

    models_by_algorithm = {
        "mlp": mlp_model,
        "rf": rf_model,
        "gbdt": gbdt_model,
    }

    # Original CALMAH committee rule:
    # choose the candidate with the lowest validation BCE.
    winner = min(
        metrics_by_algorithm,
        key=lambda algorithm: (
            metrics_by_algorithm[algorithm][
                "val_bce"
            ]
        ),
    )

    winner_metrics = metrics_by_algorithm[
        winner
    ]

    print("\nCandidate summary:")

    for algorithm in [
        "mlp",
        "rf",
        "gbdt",
    ]:
        metrics = metrics_by_algorithm[
            algorithm
        ]

        print(
            f"{algorithm.upper():5s} | "
            f"Val BCE={metrics['val_bce']:.6f} | "
            f"Threshold={metrics['threshold']:.2f} | "
            f"Train Macro-F1="
            f"{metrics['train_macro_f1']:.6f} | "
            f"Val Macro-F1="
            f"{metrics['val_macro_f1']:.6f}"
        )

    print(
        f"\nWINNER: {winner.upper()} "
        f"(selected by minimum validation BCE)"
    )

    winner_model = models_by_algorithm[
        winner
    ]

    if winner == "mlp":
        if tree_model_path.exists():
            tree_model_path.unlink()

        winner_model.save(
            keras_model_path,
        )
    else:
        if keras_model_path.exists():
            keras_model_path.unlink()

        atomic_joblib_dump(
            winner_model,
            tree_model_path,
        )

    result = {
        "combo_index": int(combo_index),
        "combo_name": combo_name,
        "input_dim": int(input_dim),
        "winner": winner,
        "model_type": winner,
        "threshold": float(
            winner_metrics["threshold"]
        ),
        "selection_metric": (
            "minimum_validation_binary_crossentropy"
        ),
        "candidate_metrics": (
            metrics_by_algorithm
        ),
        "train_probabilities": np.asarray(
            train_probabilities_by_algorithm[
                winner
            ],
            dtype=np.float32,
        ),
        "val_probabilities": np.asarray(
            val_probabilities_by_algorithm[
                winner
            ],
            dtype=np.float32,
        ),
        "seed": int(combo_seed),
        "test_used": False,
    }

    atomic_joblib_dump(
        result,
        result_path,
    )

    del models_by_algorithm
    del mlp_model
    del rf_model
    del gbdt_model
    del winner_model

    keras.backend.clear_session()
    gc.collect()

    return result

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "5"
os.environ.pop("TF_USE_LEGACY_KERAS", None)
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
combo_results = OrderedDict()

for combo_index, combo in enumerate(
    RETRAIN_COMBOS
):
    combo_name = "_".join(combo)

    X_combo_train = np.hstack([
        X_base_current[modality]["train"]
        for modality in combo
    ]).astype(np.float32)

    X_combo_val = np.hstack([
        X_base_current[modality]["val"]
        for modality in combo
    ]).astype(np.float32)

    result = train_one_combination(
        combo_index=combo_index,
        combo_name=combo_name,
        X_train=X_combo_train,
        X_val=X_combo_val,
        y_train=y_train_current,
        y_val=y_val_current,
    )

    combo_results[combo_name] = result

assert list(combo_results) == (
    RETRAIN_COMBO_NAMES
)

summary_rows = []

for combo_name in RETRAIN_COMBO_NAMES:
    result = combo_results[combo_name]
    winner = result["winner"]
    winner_metrics = result[
        "candidate_metrics"
    ][winner]

    row = {
        "combo_name": combo_name,
        "winner": winner,
        "input_dim": result["input_dim"],
        "threshold": result["threshold"],
        "winner_val_bce": (
            winner_metrics["val_bce"]
        ),
        "winner_train_macro_f1": (
            winner_metrics[
                "train_macro_f1"
            ]
        ),
        "winner_val_macro_f1": (
            winner_metrics[
                "val_macro_f1"
            ]
        ),
    }

    for algorithm in [
        "mlp",
        "rf",
        "gbdt",
    ]:
        candidate = result[
            "candidate_metrics"
        ][algorithm]

        row[
            f"{algorithm}_val_bce"
        ] = candidate["val_bce"]

        row[
            f"{algorithm}_val_macro_f1"
        ] = candidate["val_macro_f1"]

    summary_rows.append(row)

base_model_summary_df = pd.DataFrame(
    summary_rows
)

display(base_model_summary_df)

base_model_summary_df.to_csv(
    RETRAIN_ROOT
    / "base_model_training_summary.csv",
    index=False,
)

P_train_retrained = np.column_stack([
    combo_results[name][
        "train_probabilities"
    ]
    for name in RETRAIN_COMBO_NAMES
]).astype(np.float32)

P_val_retrained = np.column_stack([
    combo_results[name][
        "val_probabilities"
    ]
    for name in RETRAIN_COMBO_NAMES
]).astype(np.float32)

assert P_train_retrained.shape == (
    EXPECTED_COUNTS["train"],
    15,
)

assert P_val_retrained.shape == (
    EXPECTED_COUNTS["val"],
    15,
)

np.savez_compressed(
    RETRAIN_ROOT
    / "winner_probabilities_train_val.npz",
    P_train=P_train_retrained,
    y_train=y_train_current,
    P_val=P_val_retrained,
    y_val=y_val_current,
    combo_names=np.asarray(
        RETRAIN_COMBO_NAMES,
        dtype=object,
    ),
)

print(
    "\nPASS: all 15 current-data base model "
    "committees are complete."
)

In [ ]:
# ============================================================
# Check retraining progress after interrupt
# ============================================================

from pathlib import Path
import joblib

completed_result_paths = sorted(
    RESULT_DIR.glob("result_*.joblib")
)

print("Completed combinations:", len(completed_result_paths), "/ 15")

completed_names = []

for path in completed_result_paths:
    result = joblib.load(path)
    completed_names.append(result["combo_name"])

    print(
        f"{result['combo_name']:35s} "
        f"winner={result['winner']:5s} "
        f"threshold={result['threshold']:.2f}"
    )

remaining_names = [
    name
    for name in RETRAIN_COMBO_NAMES
    if name not in completed_names
]

print("\nRemaining combinations:", len(remaining_names))

for name in remaining_names:
    print(" ", name)

In [ ]:
# ============================================================
# TensorFlow GPU diagnostic
# ============================================================

import os
import tensorflow as tf
import keras

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

physical_gpus = tf.config.list_physical_devices("GPU")
logical_gpus = tf.config.list_logical_devices("GPU")

print("\nPhysical GPUs:")
for gpu in physical_gpus:
    print(" ", gpu)

print("\nLogical GPUs:")
for gpu in logical_gpus:
    print(" ", gpu)

print("\nTensorFlow built with CUDA:", tf.test.is_built_with_cuda())
print("TensorFlow GPU device name:", tf.test.gpu_device_name())

if not physical_gpus:
    print(
        "\nFAIL: TensorFlow cannot currently see a GPU. "
        "The MLP candidates are running on CPU."
    )
else:
    print(
        "\nPASS: TensorFlow sees a GPU. "
        "MLP candidates should use it automatically."
    )

In [ ]:
retrained_model_types = {
    name: combo_results[name]["winner"]
    for name in RETRAIN_COMBO_NAMES
}

retrained_thresholds = {
    name: float(
        combo_results[name]["threshold"]
    )
    for name in RETRAIN_COMBO_NAMES
}

pipeline_meta_current = {
    "stats_scaler": stats_scaler_current,
    "video_scaler": video_scaler_current,
    "video_pca": video_pca_current,
    "combinations": RETRAIN_COMBO_NAMES,
    "model_types": retrained_model_types,
    "thresholds": retrained_thresholds,
    "weights": PRIMARY_ENSEMBLE_WEIGHTS,
    "training_dataset": "ABAW11_current_train",
    "selection_dataset": "ABAW11_current_val",
    "test_used_for_training_or_selection": False,
    "seed": SEED,
}

atomic_joblib_dump(
    pipeline_meta_current,
    RETRAIN_ROOT
    / "pipeline_meta.joblib",
)


def hard_vote_from_probability_matrix(
    probabilities,
    combo_names,
    thresholds,
    weights,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    threshold_vector = np.asarray([
        thresholds[name]
        for name in combo_names
    ], dtype=np.float64)

    weights = np.asarray(
        weights,
        dtype=np.float64,
    )

    votes = (
        probabilities
        >= threshold_vector.reshape(1, -1)
    ).astype(np.float64)

    weighted_score = (
        votes @ weights
    )

    prediction = (
        weighted_score
        >= (weights.sum() / 2.0)
    ).astype(int)

    return prediction, weighted_score, votes


train_ensemble_prediction, _, _ = (
    hard_vote_from_probability_matrix(
        P_train_retrained,
        RETRAIN_COMBO_NAMES,
        retrained_thresholds,
        PRIMARY_ENSEMBLE_WEIGHTS,
    )
)

val_ensemble_prediction, val_score, _ = (
    hard_vote_from_probability_matrix(
        P_val_retrained,
        RETRAIN_COMBO_NAMES,
        retrained_thresholds,
        PRIMARY_ENSEMBLE_WEIGHTS,
    )
)

train_ensemble_f1 = f1_score(
    y_train_current,
    train_ensemble_prediction,
    average="macro",
    zero_division=0,
)

val_ensemble_f1 = f1_score(
    y_val_current,
    val_ensemble_prediction,
    average="macro",
    zero_division=0,
)

locked_configuration = {
    "pipeline_dir": str(RETRAIN_ROOT),
    "combo_names": RETRAIN_COMBO_NAMES,
    "model_types": retrained_model_types,
    "thresholds": retrained_thresholds,
    "weights": PRIMARY_ENSEMBLE_WEIGHTS,
    "ensemble_rule": (
        "released_CALMAH_hard_vote_weights"
    ),
    "train_macro_f1": float(
        train_ensemble_f1
    ),
    "val_macro_f1": float(
        val_ensemble_f1
    ),
    "test_used": False,
}

atomic_joblib_dump(
    locked_configuration,
    RETRAIN_ROOT
    / "locked_before_test.joblib",
)

print("Retrained ensemble train Macro-F1:", train_ensemble_f1)
print("Retrained ensemble val Macro-F1:", val_ensemble_f1)
print(
    "Validation prediction counts:",
    dict(zip(*np.unique(
        val_ensemble_prediction,
        return_counts=True,
    )))
)

print(
    "\nPASS: pipeline configuration was saved and "
    "locked before current test evaluation."
)
print(
    "Primary experiment uses released CALMAH "
    "ensemble weights to isolate base-model retraining."
)

In [ ]:
if "CALMAH_pipeline" not in globals():
    raise RuntimeError(
        "CALMAH_pipeline is not defined."
    )

# This assumes the earlier Keras-3-compatible CALMAH_pipeline.load
# patch remains active in the current kernel.
retrained_pipeline = CALMAH_pipeline.load(
    str(RETRAIN_ROOT)
)

assert list(
    retrained_pipeline.combinations
) == RETRAIN_COMBO_NAMES

test_rows_current = sorted_split_rows(
    "test"
)

test_ids_current = (
    test_rows_current["video_id"].tolist()
)

y_test_current = (
    test_rows_current["label"]
    .astype(int)
    .to_numpy()
)

print("Building current test arrays now...")

X_test_current = build_feature_arrays(
    test_ids_current
)


def get_probability_matrix_for_pipeline(
    pipeline_object,
    X_text,
    X_audio,
    X_video,
    X_stats,
):
    processed = {
        "Text": np.asarray(X_text),
        "Audio": np.asarray(X_audio),
        "Video": (
            pipeline_object.video_pca.transform(
                pipeline_object.video_scaler.transform(
                    np.asarray(X_video)
                )
            )
        ),
        "Stats": (
            pipeline_object.stats_scaler.transform(
                np.asarray(X_stats)
            )
        ),
    }

    probabilities = []

    for combo_name in (
        pipeline_object.combinations
    ):
        modalities = combo_name.split("_")

        X_combo = np.hstack([
            processed[modality]
            for modality in modalities
        ])

        model = pipeline_object.models[
            combo_name
        ]

        model_type = pipeline_object.model_types[
            combo_name
        ]

        if model_type == "mlp":
            probability = (
                model.predict(
                    X_combo,
                    verbose=0,
                ).reshape(-1)
            )
        else:
            probability_matrix = (
                model.predict_proba(
                    X_combo
                )
            )

            classes = np.asarray(
                model.classes_
            )

            positive_index = np.where(
                classes == 1
            )[0]

            assert len(positive_index) == 1

            probability = probability_matrix[
                :,
                int(positive_index[0]),
            ]

        probabilities.append(
            np.asarray(
                probability,
                dtype=np.float64,
            )
        )

    return np.column_stack(
        probabilities
    )


P_test_retrained = (
    get_probability_matrix_for_pipeline(
        retrained_pipeline,
        *X_test_current,
    )
)

assert P_test_retrained.shape == (
    EXPECTED_COUNTS["test"],
    15,
)

test_ensemble_prediction, test_score, _ = (
    hard_vote_from_probability_matrix(
        P_test_retrained,
        RETRAIN_COMBO_NAMES,
        retrained_thresholds,
        PRIMARY_ENSEMBLE_WEIGHTS,
    )
)

test_ensemble_f1 = f1_score(
    y_test_current,
    test_ensemble_prediction,
    average="macro",
    zero_division=0,
)

print(
    "Locked retrained-base-model "
    "test Macro-F1:",
    test_ensemble_f1,
)

print("\nClassification report:")
print(
    classification_report(
        y_test_current,
        test_ensemble_prediction,
        digits=6,
        zero_division=0,
    )
)

per_model_test_rows = []

for model_index, combo_name in enumerate(
    RETRAIN_COMBO_NAMES
):
    threshold = retrained_thresholds[
        combo_name
    ]

    model_prediction = (
        P_test_retrained[:, model_index]
        >= threshold
    ).astype(int)

    per_model_test_rows.append({
        "combo_name": combo_name,
        "winner": retrained_model_types[
            combo_name
        ],
        "threshold": threshold,
        "val_macro_f1": (
            combo_results[combo_name][
                "candidate_metrics"
            ][
                combo_results[combo_name][
                    "winner"
                ]
            ][
                "val_macro_f1"
            ]
        ),
        "test_macro_f1": f1_score(
            y_test_current,
            model_prediction,
            average="macro",
            zero_division=0,
        ),
    })

per_model_test_df = pd.DataFrame(
    per_model_test_rows
)

display(per_model_test_df)

per_model_test_df.to_csv(
    RETRAIN_ROOT
    / "per_model_locked_test_results.csv",
    index=False,
)

np.savez_compressed(
    RETRAIN_ROOT
    / "winner_probabilities_test.npz",
    P_test=P_test_retrained,
    y_test=y_test_current,
    combo_names=np.asarray(
        RETRAIN_COMBO_NAMES,
        dtype=object,
    ),
)

test_report = {
    "method": (
        "current-data retrained base models "
        "+ released hard-vote weights"
    ),
    "test_macro_f1": float(
        test_ensemble_f1
    ),
    "prediction_count_0": int(
        (test_ensemble_prediction == 0).sum()
    ),
    "prediction_count_1": int(
        (test_ensemble_prediction == 1).sum()
    ),
    "test_used_for_training_or_selection": False,
}

with (
    RETRAIN_ROOT
    / "locked_test_report.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        test_report,
        handle,
        indent=2,
    )

print(
    "\nDo not alter thresholds, base-model winners, "
    "or weights in response to this test result."
)

In [ ]:
private_rows_ordered = (
    private_df
    .sort_values("private_order")
    .reset_index(drop=True)
)

private_ids_current = (
    private_rows_ordered[
        "video_id"
    ].tolist()
)

print("Building current private arrays...")

X_private_current = build_feature_arrays(
    private_ids_current
)

P_private_retrained = (
    get_probability_matrix_for_pipeline(
        retrained_pipeline,
        *X_private_current,
    )
)

assert P_private_retrained.shape == (
    EXPECTED_COUNTS["private"],
    15,
)

private_prediction, private_score, private_votes = (
    hard_vote_from_probability_matrix(
        P_private_retrained,
        RETRAIN_COMBO_NAMES,
        retrained_thresholds,
        PRIMARY_ENSEMBLE_WEIGHTS,
    )
)

submission_ids = (
    private_rows_ordered[
        "submission_id"
    ].astype(str).tolist()
)

assert len(submission_ids) == len(
    private_prediction
)

submission_path = (
    RETRAIN_ROOT
    / "trial-retrained-base-models.txt"
)

with submission_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    for video_id, label in zip(
        submission_ids,
        private_prediction,
    ):
        handle.write(
            f"{video_id},{int(label)}\n"
        )

# Strict read-back validation.
submission_lines = [
    line
    for line in submission_path
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]

assert len(submission_lines) == (
    EXPECTED_COUNTS["private"]
)

written_ids = []
written_labels = []

for line in submission_lines:
    video_id, label = line.rsplit(
        ",",
        1,
    )

    assert label in {"0", "1"}

    written_ids.append(video_id)
    written_labels.append(int(label))

assert written_ids == submission_ids

np.testing.assert_array_equal(
    np.asarray(written_labels),
    private_prediction,
)

private_audit_df = (
    private_rows_ordered.copy()
)

private_audit_df[
    "weighted_vote_score"
] = private_score

private_audit_df[
    "prediction"
] = private_prediction

for model_index, combo_name in enumerate(
    RETRAIN_COMBO_NAMES
):
    safe_name = re.sub(
        r"[^A-Za-z0-9_]+",
        "_",
        combo_name,
    )

    private_audit_df[
        f"prob_{safe_name}"
    ] = P_private_retrained[
        :,
        model_index,
    ]

    private_audit_df[
        f"vote_{safe_name}"
    ] = private_votes[
        :,
        model_index,
    ].astype(int)

private_audit_df.to_csv(
    RETRAIN_ROOT
    / "private_prediction_audit.csv",
    index=False,
)

np.savez_compressed(
    RETRAIN_ROOT
    / "winner_probabilities_private.npz",
    P_private=P_private_retrained,
    combo_names=np.asarray(
        RETRAIN_COMBO_NAMES,
        dtype=object,
    ),
)

print("Submission:", submission_path)
print(
    "Prediction counts:",
    dict(zip(*np.unique(
        private_prediction,
        return_counts=True,
    )))
)
print(
    "PASS: 152 private predictions were generated "
    "with the locked retrained pipeline."
)

# 11. Stability-regularized ensemble weighting

Loads validation probabilities, constructs participant-aware stability splits, searches non-negative weights, locks the primary configuration, and evaluates test/private data.

In [ ]:
from pathlib import Path
from collections import OrderedDict
import hashlib
import json
import os
import re
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
)

warnings.filterwarnings("ignore")

if "REPO_DIR" in globals():
    REPO_DIR_ENSEMBLE = Path(REPO_DIR).resolve()
else:
    REPO_DIR_ENSEMBLE = Path(
        "/workspace/project"
    ).resolve()

SOURCE_RETRAIN_ROOT = (
    REPO_DIR_ENSEMBLE
    / "abaw11_retrained_base_models"
)

ENSEMBLE_ROOT = (
    REPO_DIR_ENSEMBLE
    / "abaw11_retrained_ensemble_weights_v1"
)

ENSEMBLE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert SOURCE_RETRAIN_ROOT.exists(), SOURCE_RETRAIN_ROOT
assert ENSEMBLE_ROOT.resolve() != SOURCE_RETRAIN_ROOT.resolve()

SOURCE_FILES = {
    "pipeline_meta": (
        SOURCE_RETRAIN_ROOT
        / "pipeline_meta.joblib"
    ),
    "train_val_probabilities": (
        SOURCE_RETRAIN_ROOT
        / "winner_probabilities_train_val.npz"
    ),
    "val_manifest": (
        SOURCE_RETRAIN_ROOT
        / "val_manifest.csv"
    ),
    "test_probabilities": (
        SOURCE_RETRAIN_ROOT
        / "winner_probabilities_test.npz"
    ),
    "private_probabilities": (
        SOURCE_RETRAIN_ROOT
        / "winner_probabilities_private.npz"
    ),
}

for required_name in [
    "pipeline_meta",
    "train_val_probabilities",
    "val_manifest",
]:
    assert SOURCE_FILES[required_name].exists(), (
        required_name,
        SOURCE_FILES[required_name],
    )


def sha256_file(path):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def fingerprint_existing_sources():
    return {
        name: {
            "path": str(path),
            "exists": path.exists(),
            "sha256": (
                sha256_file(path)
                if path.exists()
                else None
            ),
            "size_bytes": (
                path.stat().st_size
                if path.exists()
                else None
            ),
        }
        for name, path in SOURCE_FILES.items()
    }


SOURCE_FINGERPRINTS_BEFORE = (
    fingerprint_existing_sources()
)

with (
    ENSEMBLE_ROOT
    / "source_fingerprints_before.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        SOURCE_FINGERPRINTS_BEFORE,
        handle,
        indent=2,
    )

print("Read-only source directory:")
print(SOURCE_RETRAIN_ROOT)

print("\nNew ensemble experiment directory:")
print(ENSEMBLE_ROOT)

print("\nSource files:")
for name, info in SOURCE_FINGERPRINTS_BEFORE.items():
    print(
        f"{name:26s} "
        f"exists={info['exists']} | "
        f"{info['path']}"
    )

print(
    "\nPASS: all new files will be written only to "
    "the isolated ensemble directory."
)

In [ ]:
def normalize_combo_name(value):
    if isinstance(value, (list, tuple)):
        return "_".join(map(str, value))
    return str(value)


source_meta = joblib.load(
    SOURCE_FILES["pipeline_meta"]
)

SOURCE_COMBO_NAMES = [
    normalize_combo_name(value)
    for value in source_meta["combinations"]
]

SOURCE_THRESHOLDS = {
    normalize_combo_name(name): float(value)
    for name, value
    in source_meta["thresholds"].items()
}

SOURCE_RELEASED_WEIGHTS = np.asarray(
    source_meta["weights"],
    dtype=np.float64,
).copy()

train_val_archive = np.load(
    SOURCE_FILES["train_val_probabilities"],
    allow_pickle=True,
)

P_VAL_WEIGHTS = np.asarray(
    train_val_archive["P_val"],
    dtype=np.float64,
)

Y_VAL_WEIGHTS = np.asarray(
    train_val_archive["y_val"],
    dtype=int,
).reshape(-1)

ARCHIVE_COMBO_NAMES = [
    normalize_combo_name(value)
    for value in train_val_archive[
        "combo_names"
    ].tolist()
]

assert ARCHIVE_COMBO_NAMES == SOURCE_COMBO_NAMES
assert P_VAL_WEIGHTS.shape == (
    len(Y_VAL_WEIGHTS),
    len(SOURCE_COMBO_NAMES),
)
assert P_VAL_WEIGHTS.shape == (124, 15)
assert SOURCE_RELEASED_WEIGHTS.shape == (15,)
assert set(np.unique(Y_VAL_WEIGHTS)).issubset({0, 1})
assert np.isfinite(P_VAL_WEIGHTS).all()

THRESHOLD_VECTOR_FIXED = np.asarray([
    SOURCE_THRESHOLDS[name]
    for name in SOURCE_COMBO_NAMES
], dtype=np.float64)

# Normalize the released weights only for numerical consistency.
RELEASED_WEIGHTS_NORMALIZED = (
    SOURCE_RELEASED_WEIGHTS
    / SOURCE_RELEASED_WEIGHTS.sum()
)

UNIFORM_WEIGHTS = np.full(
    len(SOURCE_COMBO_NAMES),
    1.0 / len(SOURCE_COMBO_NAMES),
    dtype=np.float64,
)

# Validation manifest is needed only to form participant-level holdouts.
val_manifest_df = pd.read_csv(
    SOURCE_FILES["val_manifest"]
)

assert len(val_manifest_df) == len(Y_VAL_WEIGHTS)

if "label" in val_manifest_df.columns:
    np.testing.assert_array_equal(
        val_manifest_df["label"]
        .astype(int)
        .to_numpy(),
        Y_VAL_WEIGHTS,
    )


def participant_from_video_id(value):
    filename = Path(
        str(value).replace("\\", "/")
    ).name

    match = re.match(
        r"(\d+)_",
        filename,
    )

    if match is None:
        raise ValueError(
            f"Cannot parse participant ID from {value!r}"
        )

    return match.group(1)


VAL_PARTICIPANTS = np.asarray([
    participant_from_video_id(value)
    for value in val_manifest_df["video_id"]
])

assert len(VAL_PARTICIPANTS) == len(Y_VAL_WEIGHTS)

print("Validation probabilities:", P_VAL_WEIGHTS.shape)
print("Validation participants:", len(np.unique(VAL_PARTICIPANTS)))
print("Fixed threshold vector:", THRESHOLD_VECTOR_FIXED)
print("Released weights sum:", RELEASED_WEIGHTS_NORMALIZED.sum())

print(
    "\nPASS: test/private probabilities have not been loaded."
)
print(
    "PASS: per-model thresholds are fixed and will not be searched."
)

In [ ]:
def fast_macro_f1_binary(
    labels,
    prediction_matrix,
):
    labels = np.asarray(
        labels,
        dtype=bool,
    ).reshape(1, -1)

    predictions = np.asarray(
        prediction_matrix,
        dtype=bool,
    )

    if predictions.ndim == 1:
        predictions = predictions.reshape(1, -1)

    true_positive = np.sum(
        predictions & labels,
        axis=1,
    )

    false_positive = np.sum(
        predictions & ~labels,
        axis=1,
    )

    false_negative = np.sum(
        ~predictions & labels,
        axis=1,
    )

    true_negative = np.sum(
        ~predictions & ~labels,
        axis=1,
    )

    positive_f1 = (
        2.0 * true_positive
        / np.maximum(
            2.0 * true_positive
            + false_positive
            + false_negative,
            1e-12,
        )
    )

    negative_f1 = (
        2.0 * true_negative
        / np.maximum(
            2.0 * true_negative
            + false_positive
            + false_negative,
            1e-12,
        )
    )

    return 0.5 * (
        positive_f1 + negative_f1
    )


def fixed_hard_votes(probabilities):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    return (
        probabilities
        >= THRESHOLD_VECTOR_FIXED.reshape(1, -1)
    ).astype(np.float64)


VAL_HARD_VOTES = fixed_hard_votes(
    P_VAL_WEIGHTS
)


def predict_hard_weights(
    votes,
    weights,
):
    weights = np.asarray(
        weights,
        dtype=np.float64,
    )

    assert np.all(weights >= -1e-12)
    assert np.isclose(weights.sum(), 1.0)

    scores = votes @ weights

    predictions = (
        scores >= 0.5
    ).astype(int)

    return predictions, scores


def normalize_nonnegative(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    values = np.maximum(values, 0.0)

    if values.sum() <= 1e-12:
        return UNIFORM_WEIGHTS.copy()

    return values / values.sum()


# A deterministic current-validation reliability control.
positive_prevalence = float(
    Y_VAL_WEIGHTS.mean()
)

validation_ap = np.asarray([
    average_precision_score(
        Y_VAL_WEIGHTS,
        P_VAL_WEIGHTS[:, index],
    )
    for index in range(P_VAL_WEIGHTS.shape[1])
], dtype=np.float64)

ap_skill = np.maximum(
    validation_ap - positive_prevalence,
    0.0,
)

AP_RAW_WEIGHTS = normalize_nonnegative(
    ap_skill
)

# Strong shrinkage prevents the 124-sample validation set from
# assigning all mass to a small number of models.
AP_SHRINKAGE_TO_UNIFORM = 0.50

AP_SHRUNK_WEIGHTS = normalize_nonnegative(
    AP_SHRINKAGE_TO_UNIFORM
    * UNIFORM_WEIGHTS
    + (
        1.0 - AP_SHRINKAGE_TO_UNIFORM
    )
    * AP_RAW_WEIGHTS
)

model_reliability_df = pd.DataFrame({
    "combo_name": SOURCE_COMBO_NAMES,
    "validation_ap": validation_ap,
    "released_weight": (
        RELEASED_WEIGHTS_NORMALIZED
    ),
    "uniform_weight": UNIFORM_WEIGHTS,
    "ap_shrunk_weight": AP_SHRUNK_WEIGHTS,
})

display(model_reliability_df)

model_reliability_df.to_csv(
    ENSEMBLE_ROOT
    / "validation_model_reliability.csv",
    index=False,
)


def build_repeated_group_holdouts(
    labels,
    groups,
    *,
    number_of_splits=64,
    holdout_group_fraction=0.30,
    random_state=2026,
):
    labels = np.asarray(labels, dtype=int)
    groups = np.asarray(groups)

    unique_groups = np.unique(groups)
    number_of_holdout_groups = max(
        2,
        int(round(
            len(unique_groups)
            * holdout_group_fraction
        )),
    )

    rng = np.random.default_rng(
        random_state
    )

    holdouts = []
    seen = set()
    maximum_attempts = number_of_splits * 200

    for _ in range(maximum_attempts):
        selected_groups = tuple(sorted(
            rng.choice(
                unique_groups,
                size=number_of_holdout_groups,
                replace=False,
            ).tolist()
        ))

        if selected_groups in seen:
            continue

        mask = np.isin(
            groups,
            selected_groups,
        )

        indices = np.flatnonzero(mask)

        if len(indices) < 10:
            continue

        if len(np.unique(labels[indices])) < 2:
            continue

        seen.add(selected_groups)
        holdouts.append(indices)

        if len(holdouts) >= number_of_splits:
            break

    if len(holdouts) < number_of_splits:
        raise RuntimeError(
            "Could not construct enough valid participant-level "
            f"holdouts: {len(holdouts)}/{number_of_splits}"
        )

    return holdouts


PARTICIPANT_HOLDOUTS = (
    build_repeated_group_holdouts(
        Y_VAL_WEIGHTS,
        VAL_PARTICIPANTS,
        number_of_splits=64,
        holdout_group_fraction=0.30,
        random_state=2026,
    )
)

holdout_summary_df = pd.DataFrame([
    {
        "holdout_index": index,
        "samples": len(indices),
        "participants": len(
            np.unique(
                VAL_PARTICIPANTS[indices]
            )
        ),
        "positive_fraction": float(
            Y_VAL_WEIGHTS[indices].mean()
        ),
    }
    for index, indices
    in enumerate(PARTICIPANT_HOLDOUTS)
])

display(holdout_summary_df.describe())

holdout_summary_df.to_csv(
    ENSEMBLE_ROOT
    / "participant_holdout_summary.csv",
    index=False,
)

print(
    "PASS: all stability holdouts keep entire "
    "participants together."
)

In [ ]:
# Fixed before search. Do not tune these after seeing test results.
NUMBER_OF_SEARCH_SEEDS = 5
SEARCH_SEEDS = [
    2026,
    2027,
    2028,
    2029,
    2030,
]

CANDIDATES_PER_SEED = 30000
SEARCH_BATCH_SIZE = 512

MAX_SINGLE_WEIGHT = 0.20
MIN_EFFECTIVE_MODELS = 6.0

# Objective coefficients are fixed in advance.
FULL_VALIDATION_WEIGHT = 0.40
HOLDOUT_MEAN_WEIGHT = 0.60
HOLDOUT_STD_PENALTY = 0.35
UNIFORM_DISTANCE_PENALTY = 0.03

assert len(SEARCH_SEEDS) == NUMBER_OF_SEARCH_SEEDS
assert MAX_SINGLE_WEIGHT * len(SOURCE_COMBO_NAMES) >= 1.0


def project_capped_simplex_rows(
    matrix,
    cap,
    iterations=30,
):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    matrix = np.maximum(
        matrix,
        0.0,
    )

    zero_rows = (
        matrix.sum(axis=1) <= 1e-12
    )

    if np.any(zero_rows):
        matrix[zero_rows] = UNIFORM_WEIGHTS

    matrix /= matrix.sum(
        axis=1,
        keepdims=True,
    )

    for _ in range(iterations):
        above_cap = matrix > cap

        if not np.any(above_cap):
            break

        matrix = np.minimum(
            matrix,
            cap,
        )

        deficit = (
            1.0
            - matrix.sum(
                axis=1,
                keepdims=True,
            )
        )

        capacity = np.maximum(
            cap - matrix,
            0.0,
        )

        capacity_sum = capacity.sum(
            axis=1,
            keepdims=True,
        )

        valid = (
            capacity_sum[:, 0] > 1e-12
        )

        matrix[valid] += (
            deficit[valid]
            * capacity[valid]
            / capacity_sum[valid]
        )

    matrix = np.maximum(
        matrix,
        0.0,
    )

    matrix /= matrix.sum(
        axis=1,
        keepdims=True,
    )

    if np.any(
        matrix.max(axis=1)
        > cap + 1e-8
    ):
        raise RuntimeError(
            "Capped-simplex projection failed."
        )

    return matrix


def effective_model_count(weights):
    weights = np.asarray(
        weights,
        dtype=np.float64,
    )

    return 1.0 / np.sum(
        np.square(weights),
        axis=-1,
    )


def generate_weight_candidates(
    rng,
    batch_size,
):
    centers = np.stack([
        UNIFORM_WEIGHTS,
        AP_SHRUNK_WEIGHTS,
        RELEASED_WEIGHTS_NORMALIZED,
    ])

    center_probabilities = np.asarray([
        0.50,
        0.30,
        0.20,
    ])

    center_indices = rng.choice(
        len(centers),
        size=batch_size,
        p=center_probabilities,
    )

    selected_centers = centers[
        center_indices
    ]

    concentration_values = rng.choice(
        np.asarray([
            30.0,
            60.0,
            120.0,
            240.0,
        ]),
        size=batch_size,
    )

    alpha = (
        selected_centers
        * concentration_values[:, None]
        + 0.50
    )

    gamma_samples = rng.gamma(
        shape=alpha,
        scale=1.0,
    )

    candidates = (
        gamma_samples
        / gamma_samples.sum(
            axis=1,
            keepdims=True,
        )
    )

    candidates = (
        project_capped_simplex_rows(
            candidates,
            cap=MAX_SINGLE_WEIGHT,
        )
    )

    return candidates


def evaluate_candidate_batch(
    candidate_weights,
):
    candidate_weights = np.asarray(
        candidate_weights,
        dtype=np.float64,
    )

    effective_counts = (
        effective_model_count(
            candidate_weights
        )
    )

    valid_candidate_mask = (
        effective_counts
        >= MIN_EFFECTIVE_MODELS
    )

    scores = (
        candidate_weights
        @ VAL_HARD_VOTES.T
    )

    predictions = (
        scores >= 0.5
    )

    full_validation_f1 = (
        fast_macro_f1_binary(
            Y_VAL_WEIGHTS,
            predictions,
        )
    )

    holdout_scores = np.vstack([
        fast_macro_f1_binary(
            Y_VAL_WEIGHTS[indices],
            predictions[:, indices],
        )
        for indices in PARTICIPANT_HOLDOUTS
    ])

    holdout_mean = (
        holdout_scores.mean(axis=0)
    )

    holdout_std = (
        holdout_scores.std(axis=0)
    )

    holdout_p10 = np.percentile(
        holdout_scores,
        10,
        axis=0,
    )

    uniform_distance = np.sqrt(
        np.sum(
            np.square(
                candidate_weights
                - UNIFORM_WEIGHTS
            ),
            axis=1,
        )
    )

    objective = (
        FULL_VALIDATION_WEIGHT
        * full_validation_f1
        + HOLDOUT_MEAN_WEIGHT
        * holdout_mean
        - HOLDOUT_STD_PENALTY
        * holdout_std
        - UNIFORM_DISTANCE_PENALTY
        * uniform_distance
    )

    objective = np.where(
        valid_candidate_mask,
        objective,
        -np.inf,
    )

    return {
        "objective": objective,
        "full_validation_f1": (
            full_validation_f1
        ),
        "holdout_mean": holdout_mean,
        "holdout_std": holdout_std,
        "holdout_p10": holdout_p10,
        "uniform_distance": uniform_distance,
        "effective_models": effective_counts,
    }


def evaluate_single_weight_vector(
    weights,
):
    weights = project_capped_simplex_rows(
        np.asarray(weights).reshape(1, -1),
        cap=MAX_SINGLE_WEIGHT,
    )[0]

    result = evaluate_candidate_batch(
        weights.reshape(1, -1)
    )

    return {
        key: (
            float(value[0])
            if isinstance(value, np.ndarray)
            else value
        )
        for key, value in result.items()
    } | {
        "weights": weights,
        "max_weight": float(
            weights.max()
        ),
    }


def search_one_seed(
    random_state,
):
    rng = np.random.default_rng(
        random_state
    )

    best_result = None
    evaluated_candidates = 0

    while (
        evaluated_candidates
        < CANDIDATES_PER_SEED
    ):
        batch_size = min(
            SEARCH_BATCH_SIZE,
            CANDIDATES_PER_SEED
            - evaluated_candidates,
        )

        candidates = (
            generate_weight_candidates(
                rng,
                batch_size,
            )
        )

        # Include the exact deterministic controls once.
        if evaluated_candidates == 0:
            controls = np.stack([
                UNIFORM_WEIGHTS,
                AP_SHRUNK_WEIGHTS,
                project_capped_simplex_rows(
                    RELEASED_WEIGHTS_NORMALIZED
                    .reshape(1, -1),
                    cap=MAX_SINGLE_WEIGHT,
                )[0],
            ])

            number_of_controls = min(
                len(controls),
                len(candidates),
            )

            candidates[
                :number_of_controls
            ] = controls[
                :number_of_controls
            ]

        metrics = (
            evaluate_candidate_batch(
                candidates
            )
        )

        best_index = int(
            np.argmax(
                metrics["objective"]
            )
        )

        current_result = {
            "random_state": int(
                random_state
            ),
            "weights": candidates[
                best_index
            ].copy(),
            **{
                key: float(
                    value[best_index]
                )
                for key, value
                in metrics.items()
            },
            "max_weight": float(
                candidates[
                    best_index
                ].max()
            ),
        }

        if (
            best_result is None
            or current_result["objective"]
            > best_result["objective"]
        ):
            best_result = current_result

        evaluated_candidates += batch_size

    return best_result


seed_results = []

for search_index, seed in enumerate(
    SEARCH_SEEDS,
    start=1,
):
    print(
        f"Search {search_index}/"
        f"{NUMBER_OF_SEARCH_SEEDS} "
        f"| seed={seed}"
    )

    result = search_one_seed(seed)

    seed_results.append(result)

    print(
        "  objective="
        f"{result['objective']:.6f} | "
        "full_val="
        f"{result['full_validation_f1']:.6f} | "
        "holdout_mean="
        f"{result['holdout_mean']:.6f} | "
        "holdout_std="
        f"{result['holdout_std']:.6f} | "
        "effective_models="
        f"{result['effective_models']:.3f}"
    )


seed_weight_matrix = np.stack([
    result["weights"]
    for result in seed_results
])

CONSENSUS_WEIGHTS = (
    project_capped_simplex_rows(
        seed_weight_matrix.mean(
            axis=0,
            keepdims=True,
        ),
        cap=MAX_SINGLE_WEIGHT,
    )[0]
)

CONSENSUS_METRICS = (
    evaluate_single_weight_vector(
        CONSENSUS_WEIGHTS
    )
)

print("\nConsensus metrics:")
for key, value in CONSENSUS_METRICS.items():
    if key != "weights":
        print(f"{key:24s}: {value}")

seed_results_df = pd.DataFrame([
    {
        "random_state": result[
            "random_state"
        ],
        "objective": result[
            "objective"
        ],
        "full_validation_f1": result[
            "full_validation_f1"
        ],
        "holdout_mean": result[
            "holdout_mean"
        ],
        "holdout_std": result[
            "holdout_std"
        ],
        "holdout_p10": result[
            "holdout_p10"
        ],
        "uniform_distance": result[
            "uniform_distance"
        ],
        "effective_models": result[
            "effective_models"
        ],
        "max_weight": result[
            "max_weight"
        ],
    }
    for result in seed_results
])

display(seed_results_df)

seed_results_df.to_csv(
    ENSEMBLE_ROOT
    / "search_seed_results.csv",
    index=False,
)

np.savez_compressed(
    ENSEMBLE_ROOT
    / "search_seed_weights.npz",
    weights=seed_weight_matrix,
    seeds=np.asarray(
        SEARCH_SEEDS,
        dtype=int,
    ),
    combo_names=np.asarray(
        SOURCE_COMBO_NAMES,
        dtype=object,
    ),
)

print(
    "\nPASS: consensus weights were created from "
    "five independent stability-regularized searches."
)

In [ ]:
def validation_metrics_for_method(
    method_name,
    weights,
):
    weights = normalize_nonnegative(
        weights
    )

    prediction, score = (
        predict_hard_weights(
            VAL_HARD_VOTES,
            weights,
        )
    )

    holdout_f1 = np.asarray([
        f1_score(
            Y_VAL_WEIGHTS[indices],
            prediction[indices],
            average="macro",
            zero_division=0,
        )
        for indices in PARTICIPANT_HOLDOUTS
    ])

    return {
        "method": method_name,
        "validation_macro_f1": float(
            f1_score(
                Y_VAL_WEIGHTS,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "holdout_mean": float(
            holdout_f1.mean()
        ),
        "holdout_std": float(
            holdout_f1.std()
        ),
        "holdout_p10": float(
            np.percentile(
                holdout_f1,
                10,
            )
        ),
        "effective_models": float(
            effective_model_count(
                weights
            )
        ),
        "max_weight": float(
            weights.max()
        ),
        "predicted_0": int(
            (prediction == 0).sum()
        ),
        "predicted_1": int(
            (prediction == 1).sum()
        ),
    }


VALIDATION_METHOD_WEIGHTS = OrderedDict([
    (
        "released_weights_control",
        RELEASED_WEIGHTS_NORMALIZED,
    ),
    (
        "equal_weights_control",
        UNIFORM_WEIGHTS,
    ),
    (
        "ap_shrunk_weights_control",
        AP_SHRUNK_WEIGHTS,
    ),
    (
        "regularized_consensus_primary",
        CONSENSUS_WEIGHTS,
    ),
])

validation_method_rows = [
    validation_metrics_for_method(
        method_name,
        weights,
    )
    for method_name, weights
    in VALIDATION_METHOD_WEIGHTS.items()
]

validation_method_df = pd.DataFrame(
    validation_method_rows
)

display(validation_method_df)

validation_method_df.to_csv(
    ENSEMBLE_ROOT
    / "validation_method_comparison.csv",
    index=False,
)

weight_table_df = pd.DataFrame({
    "combo_name": SOURCE_COMBO_NAMES,
    "fixed_threshold": THRESHOLD_VECTOR_FIXED,
    "released_weight_control": (
        RELEASED_WEIGHTS_NORMALIZED
    ),
    "equal_weight_control": UNIFORM_WEIGHTS,
    "ap_shrunk_weight_control": (
        AP_SHRUNK_WEIGHTS
    ),
    "regularized_consensus_primary": (
        CONSENSUS_WEIGHTS
    ),
})

display(weight_table_df)

weight_table_df.to_csv(
    ENSEMBLE_ROOT
    / "locked_weight_table.csv",
    index=False,
)

# The primary method was fixed before test:
# regularized consensus, not whichever control has the highest
# displayed validation F1.
LOCKED_PRIMARY_METHOD = (
    "regularized_consensus_primary"
)

LOCKED_PRIMARY_WEIGHTS = (
    CONSENSUS_WEIGHTS.copy()
)

assert np.all(
    LOCKED_PRIMARY_WEIGHTS >= 0.0
)
assert np.isclose(
    LOCKED_PRIMARY_WEIGHTS.sum(),
    1.0,
)
assert (
    LOCKED_PRIMARY_WEIGHTS.max()
    <= MAX_SINGLE_WEIGHT + 1e-8
)
assert (
    effective_model_count(
        LOCKED_PRIMARY_WEIGHTS
    )
    >= MIN_EFFECTIVE_MODELS
)

LOCKED_CONFIG = {
    "method": LOCKED_PRIMARY_METHOD,
    "source_retrained_directory": str(
        SOURCE_RETRAIN_ROOT
    ),
    "combo_names": SOURCE_COMBO_NAMES,
    "fixed_thresholds": (
        THRESHOLD_VECTOR_FIXED.copy()
    ),
    "weights": (
        LOCKED_PRIMARY_WEIGHTS.copy()
    ),
    "final_threshold": 0.5,
    "fusion": "thresholded_hard_vote",
    "weight_constraints": {
        "nonnegative": True,
        "sum_to_one": True,
        "maximum_single_weight": (
            MAX_SINGLE_WEIGHT
        ),
        "minimum_effective_models": (
            MIN_EFFECTIVE_MODELS
        ),
    },
    "search": {
        "search_seeds": SEARCH_SEEDS,
        "candidates_per_seed": (
            CANDIDATES_PER_SEED
        ),
        "participant_holdouts": len(
            PARTICIPANT_HOLDOUTS
        ),
        "holdout_group_fraction": 0.30,
        "objective_coefficients": {
            "full_validation_weight": (
                FULL_VALIDATION_WEIGHT
            ),
            "holdout_mean_weight": (
                HOLDOUT_MEAN_WEIGHT
            ),
            "holdout_std_penalty": (
                HOLDOUT_STD_PENALTY
            ),
            "uniform_distance_penalty": (
                UNIFORM_DISTANCE_PENALTY
            ),
        },
    },
    "validation_metrics": {
        key: value
        for key, value
        in CONSENSUS_METRICS.items()
        if key != "weights"
    },
    "per_model_thresholds_modified": False,
    "base_models_modified": False,
    "source_files_modified": False,
    "test_loaded_during_search": False,
    "private_loaded_during_search": False,
}

joblib.dump(
    LOCKED_CONFIG,
    ENSEMBLE_ROOT
    / "locked_primary_ensemble.joblib",
)

with (
    ENSEMBLE_ROOT
    / "locked_primary_ensemble.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        {
            **LOCKED_CONFIG,
            "fixed_thresholds": (
                THRESHOLD_VECTOR_FIXED.tolist()
            ),
            "weights": (
                LOCKED_PRIMARY_WEIGHTS.tolist()
            ),
        },
        handle,
        indent=2,
    )

LOCKED_CONFIG_SHA256 = sha256_file(
    ENSEMBLE_ROOT
    / "locked_primary_ensemble.joblib"
)

print(
    "Locked primary configuration SHA-256:"
)
print(LOCKED_CONFIG_SHA256)

print(
    "\nPASS: primary weights were locked before "
    "loading test probabilities."
)

In [ ]:
SOURCE_FINGERPRINTS_AFTER_SEARCH = (
    fingerprint_existing_sources()
)

for name, before_info in (
    SOURCE_FINGERPRINTS_BEFORE.items()
):
    after_info = (
        SOURCE_FINGERPRINTS_AFTER_SEARCH[
            name
        ]
    )

    assert (
        before_info["exists"]
        == after_info["exists"]
    )

    if before_info["exists"]:
        assert (
            before_info["sha256"]
            == after_info["sha256"]
        ), (
            "A source experiment file was modified:",
            name,
        )

with (
    ENSEMBLE_ROOT
    / "source_fingerprints_after_search.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        SOURCE_FINGERPRINTS_AFTER_SEARCH,
        handle,
        indent=2,
    )

print(
    "PASS: retrained base models, thresholds, "
    "probability files, and previous weights were not modified."
)

In [ ]:
LOCKED_CONFIG_PATH = (
    ENSEMBLE_ROOT
    / "locked_primary_ensemble.joblib"
)

assert LOCKED_CONFIG_PATH.exists()
assert (
    sha256_file(LOCKED_CONFIG_PATH)
    == LOCKED_CONFIG_SHA256
)

if not SOURCE_FILES["test_probabilities"].exists():
    raise FileNotFoundError(
        "winner_probabilities_test.npz does not exist. "
        "Run the retraining notebook R5 once, without "
        "changing its locked base-model configuration."
    )

test_archive = np.load(
    SOURCE_FILES["test_probabilities"],
    allow_pickle=True,
)

P_TEST_WEIGHTS = np.asarray(
    test_archive["P_test"],
    dtype=np.float64,
)

Y_TEST_WEIGHTS = np.asarray(
    test_archive["y_test"],
    dtype=int,
).reshape(-1)

TEST_COMBO_NAMES = [
    normalize_combo_name(value)
    for value in test_archive[
        "combo_names"
    ].tolist()
]

assert TEST_COMBO_NAMES == SOURCE_COMBO_NAMES
assert P_TEST_WEIGHTS.shape == (
    len(Y_TEST_WEIGHTS),
    15,
)
assert P_TEST_WEIGHTS.shape[0] == 525

TEST_HARD_VOTES = fixed_hard_votes(
    P_TEST_WEIGHTS
)

test_method_rows = []
test_predictions = {}

for method_name, weights in (
    VALIDATION_METHOD_WEIGHTS.items()
):
    prediction, score = (
        predict_hard_weights(
            TEST_HARD_VOTES,
            normalize_nonnegative(weights),
        )
    )

    test_predictions[
        method_name
    ] = prediction

    test_method_rows.append({
        "method": method_name,
        "test_macro_f1": float(
            f1_score(
                Y_TEST_WEIGHTS,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "predicted_0": int(
            (prediction == 0).sum()
        ),
        "predicted_1": int(
            (prediction == 1).sum()
        ),
    })

test_method_df = pd.DataFrame(
    test_method_rows
)

display(test_method_df)

test_method_df.to_csv(
    ENSEMBLE_ROOT
    / "locked_test_method_comparison.csv",
    index=False,
)

primary_test_prediction = (
    test_predictions[
        LOCKED_PRIMARY_METHOD
    ]
)

print("\nLocked primary classification report:")
print(
    classification_report(
        Y_TEST_WEIGHTS,
        primary_test_prediction,
        digits=6,
        zero_division=0,
    )
)

np.savez_compressed(
    ENSEMBLE_ROOT
    / "locked_primary_test_predictions.npz",
    prediction=primary_test_prediction,
    y_test=Y_TEST_WEIGHTS,
    weights=LOCKED_PRIMARY_WEIGHTS,
    thresholds=THRESHOLD_VECTOR_FIXED,
    combo_names=np.asarray(
        SOURCE_COMBO_NAMES,
        dtype=object,
    ),
)

assert (
    sha256_file(LOCKED_CONFIG_PATH)
    == LOCKED_CONFIG_SHA256
)

print(
    "\nDo not alter the locked weights in response "
    "to this test result."
)

In [ ]:
# ============================================================
# Diagnose the W7 source-fingerprint assertion
# ============================================================

SOURCE_FINGERPRINTS_CURRENT = (
    fingerprint_existing_sources()
)

fingerprint_diagnostic_rows = []

for name, before_info in SOURCE_FINGERPRINTS_BEFORE.items():
    current_info = SOURCE_FINGERPRINTS_CURRENT[name]

    if (
        before_info["exists"]
        and current_info["exists"]
    ):
        if (
            before_info["sha256"]
            == current_info["sha256"]
        ):
            status = "UNCHANGED"
        else:
            status = "CONTENT_CHANGED"

    elif (
        not before_info["exists"]
        and current_info["exists"]
    ):
        status = "CREATED_AFTER_W0"

    elif (
        before_info["exists"]
        and not current_info["exists"]
    ):
        status = "DELETED_AFTER_W0"

    else:
        status = "STILL_ABSENT"

    fingerprint_diagnostic_rows.append({
        "name": name,
        "before_exists": before_info["exists"],
        "current_exists": current_info["exists"],
        "before_sha256": before_info["sha256"],
        "current_sha256": current_info["sha256"],
        "status": status,
        "path": current_info["path"],
    })

fingerprint_diagnostic_df = pd.DataFrame(
    fingerprint_diagnostic_rows
)

display(fingerprint_diagnostic_df)

In [ ]:
if not SOURCE_FILES[
    "private_probabilities"
].exists():
    raise FileNotFoundError(
        "winner_probabilities_private.npz does not exist. "
        "Run the retraining notebook R6 once to cache "
        "private probabilities."
    )

private_archive = np.load(
    SOURCE_FILES[
        "private_probabilities"
    ],
    allow_pickle=True,
)

P_PRIVATE_WEIGHTS = np.asarray(
    private_archive["P_private"],
    dtype=np.float64,
)

PRIVATE_COMBO_NAMES = [
    normalize_combo_name(value)
    for value in private_archive[
        "combo_names"
    ].tolist()
]

assert PRIVATE_COMBO_NAMES == SOURCE_COMBO_NAMES
assert P_PRIVATE_WEIGHTS.shape == (152, 15)

PRIVATE_HARD_VOTES = fixed_hard_votes(
    P_PRIVATE_WEIGHTS
)

private_prediction, private_score = (
    predict_hard_weights(
        PRIVATE_HARD_VOTES,
        LOCKED_PRIMARY_WEIGHTS,
    )
)

manifest_candidates = [
    (
        REPO_DIR_ENSEMBLE
        / "abaw11_append_results"
        / "current_private_manifest.csv"
    ),
    (
        SOURCE_RETRAIN_ROOT
        / "private_manifest.csv"
    ),
]

PRIVATE_MANIFEST_PATH = next(
    (
        path
        for path in manifest_candidates
        if path.exists()
    ),
    None,
)

if PRIVATE_MANIFEST_PATH is None:
    if "private_df" in globals():
        private_manifest_df = (
            private_df.copy()
        )
    else:
        raise FileNotFoundError(
            "Could not locate the current private manifest."
        )
else:
    private_manifest_df = pd.read_csv(
        PRIVATE_MANIFEST_PATH
    )

private_manifest_df = (
    private_manifest_df
    .sort_values("private_order")
    .reset_index(drop=True)
)

assert len(private_manifest_df) == 152
assert "submission_id" in private_manifest_df.columns
assert private_manifest_df[
    "submission_id"
].is_unique

submission_ids = (
    private_manifest_df[
        "submission_id"
    ].astype(str).tolist()
)

SUBMISSION_PATH = (
    ENSEMBLE_ROOT
    / "trial-retrained-regularized-weights.txt"
)

with SUBMISSION_PATH.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    for video_id, label in zip(
        submission_ids,
        private_prediction,
    ):
        handle.write(
            f"{video_id},{int(label)}\n"
        )

written_lines = [
    line
    for line in SUBMISSION_PATH
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]

assert len(written_lines) == 152

written_ids = []
written_labels = []

for line in written_lines:
    video_id, label = line.rsplit(
        ",",
        1,
    )

    assert label in {"0", "1"}

    written_ids.append(video_id)
    written_labels.append(int(label))

assert written_ids == submission_ids

np.testing.assert_array_equal(
    np.asarray(written_labels),
    private_prediction,
)

audit_df = private_manifest_df.copy()

audit_df["ensemble_score"] = (
    private_score
)

audit_df["prediction"] = (
    private_prediction
)

for model_index, combo_name in enumerate(
    SOURCE_COMBO_NAMES
):
    safe_name = re.sub(
        r"[^A-Za-z0-9_]+",
        "_",
        combo_name,
    )

    audit_df[
        f"prob_{safe_name}"
    ] = P_PRIVATE_WEIGHTS[
        :,
        model_index,
    ]

    audit_df[
        f"vote_{safe_name}"
    ] = PRIVATE_HARD_VOTES[
        :,
        model_index,
    ].astype(int)

audit_df.to_csv(
    ENSEMBLE_ROOT
    / "private_prediction_audit.csv",
    index=False,
)

submission_summary = {
    "submission_path": str(
        SUBMISSION_PATH
    ),
    "submission_sha256": sha256_file(
        SUBMISSION_PATH
    ),
    "method": LOCKED_PRIMARY_METHOD,
    "predicted_0": int(
        (private_prediction == 0).sum()
    ),
    "predicted_1": int(
        (private_prediction == 1).sum()
    ),
    "locked_config_sha256": (
        LOCKED_CONFIG_SHA256
    ),
}

with (
    ENSEMBLE_ROOT
    / "submission_summary.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        submission_summary,
        handle,
        indent=2,
    )

# Final immutability audit.
# ============================================================
# Corrected final immutability audit
# ============================================================

SOURCE_FINGERPRINTS_FINAL = (
    fingerprint_existing_sources()
)

# These files must have existed before ensemble searching and
# must remain byte-for-byte unchanged.
IMMUTABLE_CORE_FILES = {
    "pipeline_meta",
    "train_val_probabilities",
    "val_manifest",
}

# These files may legitimately be created after W0 by R5/R6.
# Once they existed at the baseline, however, their content
# must not change.
OPTIONAL_PROBABILITY_FILES = {
    "test_probabilities",
    "private_probabilities",
}

final_audit_rows = []


# ------------------------------------------------------------
# 1. Strictly protect core training/validation files
# ------------------------------------------------------------

for name in sorted(IMMUTABLE_CORE_FILES):
    before_info = SOURCE_FINGERPRINTS_BEFORE[name]
    final_info = SOURCE_FINGERPRINTS_FINAL[name]

    assert before_info["exists"], (
        f"Required core source file was absent at W0: {name}"
    )

    assert final_info["exists"], (
        f"Required core source file disappeared: {name}"
    )

    assert (
        before_info["sha256"]
        == final_info["sha256"]
    ), (
        "A core source experiment file was modified:",
        name,
        before_info["path"],
    )

    final_audit_rows.append({
        "name": name,
        "category": "immutable_core",
        "before_exists": True,
        "final_exists": True,
        "status": "UNCHANGED",
        "sha256": final_info["sha256"],
        "path": final_info["path"],
    })


# ------------------------------------------------------------
# 2. Allow test/private probability caches to be created by R5/R6
# ------------------------------------------------------------

for name in sorted(OPTIONAL_PROBABILITY_FILES):
    before_info = SOURCE_FINGERPRINTS_BEFORE[name]
    final_info = SOURCE_FINGERPRINTS_FINAL[name]

    assert final_info["exists"], (
        f"Required probability cache is currently missing: {name}"
    )

    if before_info["exists"]:
        # It existed before W0, so it must be unchanged.
        assert (
            before_info["sha256"]
            == final_info["sha256"]
        ), (
            "An existing probability cache was modified:",
            name,
            before_info["path"],
        )

        status = "UNCHANGED"

    else:
        # It was legitimately created later by R5 or R6.
        status = "CREATED_BY_R5_OR_R6_AFTER_W0"

    final_audit_rows.append({
        "name": name,
        "category": "optional_probability_cache",
        "before_exists": before_info["exists"],
        "final_exists": final_info["exists"],
        "status": status,
        "sha256": final_info["sha256"],
        "path": final_info["path"],
    })


final_source_audit_df = pd.DataFrame(
    final_audit_rows
)

display(final_source_audit_df)

final_source_audit_df.to_csv(
    ENSEMBLE_ROOT
    / "final_source_immutability_audit.csv",
    index=False,
)

with (
    ENSEMBLE_ROOT
    / "source_fingerprints_final.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        SOURCE_FINGERPRINTS_FINAL,
        handle,
        indent=2,
    )

print(
    "\nPASS: original pipeline metadata, train/validation "
    "probabilities, and validation manifest are unchanged."
)

print(
    "PASS: test/private probability files were only read by "
    "the ensemble experiment."
)

In [ ]:
# ============================================================
# Confirm that W7 generated a valid submission before failing
# ============================================================

SUBMISSION_PATH = (
    ENSEMBLE_ROOT
    / "trial-retrained-regularized-weights.txt"
)

assert SUBMISSION_PATH.exists(), SUBMISSION_PATH

submission_lines = [
    line.strip()
    for line in SUBMISSION_PATH
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]

assert len(submission_lines) == 152

for line_number, line in enumerate(
    submission_lines,
    start=1,
):
    video_id, label = line.rsplit(",", 1)

    assert video_id
    assert label in {"0", "1"}, (
        line_number,
        line,
    )

print("Submission exists:", SUBMISSION_PATH)
print("Lines:", len(submission_lines))
print("SHA-256:", sha256_file(SUBMISSION_PATH))
print("First three lines:")
print("\n".join(submission_lines[:3]))

In [ ]:
# ============================================================
# W8 — Report locked ensemble performance on train/val/test
# ============================================================
#
# Reporting only:
# - Does not tune weights
# - Does not tune per-model thresholds
# - Does not tune final threshold
# - Does not modify any previous experiment file
#
# Note:
# Train performance is a resubstitution score because the base
# models were trained on these same 778 samples. Validation and
# test are more meaningful measures of generalisation.

from pathlib import Path
import hashlib
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)


# ------------------------------------------------------------
# 1. Resolve experiment paths
# ------------------------------------------------------------

if "REPO_DIR_ENSEMBLE" in globals():
    REPORT_REPO_DIR = Path(
        REPO_DIR_ENSEMBLE
    ).resolve()
elif "REPO_DIR" in globals():
    REPORT_REPO_DIR = Path(
        REPO_DIR
    ).resolve()
else:
    REPORT_REPO_DIR = Path(
        "/workspace/project"
    ).resolve()


REPORT_SOURCE_ROOT = (
    REPORT_REPO_DIR
    / "abaw11_retrained_base_models"
)

REPORT_ENSEMBLE_ROOT = (
    REPORT_REPO_DIR
    / "abaw11_retrained_ensemble_weights_v1"
)

TRAIN_VAL_PATH = (
    REPORT_SOURCE_ROOT
    / "winner_probabilities_train_val.npz"
)

TEST_PATH = (
    REPORT_SOURCE_ROOT
    / "winner_probabilities_test.npz"
)

LOCKED_CONFIG_PATH = (
    REPORT_ENSEMBLE_ROOT
    / "locked_primary_ensemble.joblib"
)

for path in [
    TRAIN_VAL_PATH,
    TEST_PATH,
    LOCKED_CONFIG_PATH,
]:
    assert path.exists(), path


def report_sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


LOCKED_CONFIG_HASH_BEFORE_REPORT = (
    report_sha256_file(
        LOCKED_CONFIG_PATH
    )
)


# ------------------------------------------------------------
# 2. Load the locked ensemble configuration
# ------------------------------------------------------------

locked_report_config = joblib.load(
    LOCKED_CONFIG_PATH
)

report_combo_names = [
    str(name)
    for name in locked_report_config[
        "combo_names"
    ]
]

report_weights = np.asarray(
    locked_report_config["weights"],
    dtype=np.float64,
).reshape(-1)

report_thresholds = np.asarray(
    locked_report_config["fixed_thresholds"],
    dtype=np.float64,
).reshape(-1)

report_final_threshold = float(
    locked_report_config.get(
        "final_threshold",
        0.5,
    )
)

assert len(report_combo_names) == 15
assert report_weights.shape == (15,)
assert report_thresholds.shape == (15,)
assert np.all(report_weights >= 0)
assert np.isclose(
    report_weights.sum(),
    1.0,
    atol=1e-8,
)
assert np.isclose(
    report_final_threshold,
    0.5,
)

print("Locked method:")
print(locked_report_config["method"])

print("\nLocked ensemble weights:")
for combo_name, weight in zip(
    report_combo_names,
    report_weights,
):
    print(
        f"{combo_name:30s} "
        f"{weight:.8f}"
    )

print("\nFixed per-model thresholds:")
for combo_name, threshold in zip(
    report_combo_names,
    report_thresholds,
):
    print(
        f"{combo_name:30s} "
        f"{threshold:.4f}"
    )

print(
    "\nFinal hard-vote threshold:",
    report_final_threshold,
)


# ------------------------------------------------------------
# 3. Load train, validation and test probability matrices
# ------------------------------------------------------------

train_val_archive_report = np.load(
    TRAIN_VAL_PATH,
    allow_pickle=True,
)

test_archive_report = np.load(
    TEST_PATH,
    allow_pickle=True,
)

P_train_report = np.asarray(
    train_val_archive_report["P_train"],
    dtype=np.float64,
)

y_train_report = np.asarray(
    train_val_archive_report["y_train"],
    dtype=int,
).reshape(-1)

P_val_report = np.asarray(
    train_val_archive_report["P_val"],
    dtype=np.float64,
)

y_val_report = np.asarray(
    train_val_archive_report["y_val"],
    dtype=int,
).reshape(-1)

P_test_report = np.asarray(
    test_archive_report["P_test"],
    dtype=np.float64,
)

y_test_report = np.asarray(
    test_archive_report["y_test"],
    dtype=int,
).reshape(-1)


def normalise_report_combo_name(value):
    if isinstance(value, (list, tuple)):
        return "_".join(map(str, value))

    return str(value)


train_val_combo_names_report = [
    normalise_report_combo_name(value)
    for value in train_val_archive_report[
        "combo_names"
    ].tolist()
]

test_combo_names_report = [
    normalise_report_combo_name(value)
    for value in test_archive_report[
        "combo_names"
    ].tolist()
]

assert (
    train_val_combo_names_report
    == report_combo_names
)

assert (
    test_combo_names_report
    == report_combo_names
)

assert P_train_report.shape == (778, 15)
assert P_val_report.shape == (124, 15)
assert P_test_report.shape == (525, 15)

assert len(y_train_report) == 778
assert len(y_val_report) == 124
assert len(y_test_report) == 525


# ------------------------------------------------------------
# 4. Apply the exact locked hard-voting configuration
# ------------------------------------------------------------

def apply_locked_ensemble_report(
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    assert probabilities.ndim == 2
    assert probabilities.shape[1] == 15

    binary_votes = (
        probabilities
        >= report_thresholds.reshape(1, -1)
    ).astype(np.float64)

    weighted_scores = (
        binary_votes
        @ report_weights
    )

    predictions = (
        weighted_scores
        >= report_final_threshold
    ).astype(int)

    return (
        predictions,
        weighted_scores,
        binary_votes,
    )


report_splits = {
    "train": (
        P_train_report,
        y_train_report,
    ),
    "val": (
        P_val_report,
        y_val_report,
    ),
    "test": (
        P_test_report,
        y_test_report,
    ),
}


# ------------------------------------------------------------
# 5. Calculate complete split-level metrics
# ------------------------------------------------------------

summary_rows = []
prediction_outputs = {}

for split_name, (
    probabilities,
    labels,
) in report_splits.items():

    predictions, scores, votes = (
        apply_locked_ensemble_report(
            probabilities
        )
    )

    prediction_outputs[split_name] = {
        "prediction": predictions,
        "score": scores,
        "votes": votes,
    }

    class_0_f1 = f1_score(
        labels,
        predictions,
        labels=[0],
        average=None,
        zero_division=0,
    )[0]

    class_1_f1 = f1_score(
        labels,
        predictions,
        labels=[1],
        average=None,
        zero_division=0,
    )[0]

    matrix = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    )

    summary_rows.append({
        "split": split_name,
        "samples": len(labels),

        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),

        "macro_f1": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),

        "weighted_f1": float(
            f1_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),

        "class_0_f1": float(
            class_0_f1
        ),

        "class_1_f1": float(
            class_1_f1
        ),

        "true_0": int(
            (labels == 0).sum()
        ),

        "true_1": int(
            (labels == 1).sum()
        ),

        "predicted_0": int(
            (predictions == 0).sum()
        ),

        "predicted_1": int(
            (predictions == 1).sum()
        ),

        "tn": int(matrix[0, 0]),
        "fp": int(matrix[0, 1]),
        "fn": int(matrix[1, 0]),
        "tp": int(matrix[1, 1]),

        "score_mean": float(
            scores.mean()
        ),

        "score_std": float(
            scores.std()
        ),
    })


locked_split_performance_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 100)
print("LOCKED ENSEMBLE PERFORMANCE")
print("=" * 100)

display(locked_split_performance_df)


# ------------------------------------------------------------
# 6. Print detailed classification reports
# ------------------------------------------------------------

for split_name, (
    probabilities,
    labels,
) in report_splits.items():

    predictions = prediction_outputs[
        split_name
    ]["prediction"]

    print("\n" + "=" * 100)
    print(
        f"{split_name.upper()} "
        "CLASSIFICATION REPORT"
    )
    print("=" * 100)

    print(
        classification_report(
            labels,
            predictions,
            labels=[0, 1],
            target_names=[
                "class_0",
                "class_1",
            ],
            digits=6,
            zero_division=0,
        )
    )

    print("Confusion matrix:")
    print(
        confusion_matrix(
            labels,
            predictions,
            labels=[0, 1],
        )
    )


# ------------------------------------------------------------
# 7. Save reporting outputs only in the ensemble directory
# ------------------------------------------------------------

PERFORMANCE_REPORT_PATH = (
    REPORT_ENSEMBLE_ROOT
    / "locked_train_val_test_performance.csv"
)

locked_split_performance_df.to_csv(
    PERFORMANCE_REPORT_PATH,
    index=False,
)

for split_name, (
    probabilities,
    labels,
) in report_splits.items():

    predictions = prediction_outputs[
        split_name
    ]["prediction"]

    scores = prediction_outputs[
        split_name
    ]["score"]

    split_output_df = pd.DataFrame({
        "sample_index": np.arange(
            len(labels)
        ),
        "label": labels,
        "prediction": predictions,
        "ensemble_score": scores,
        "correct": (
            labels == predictions
        ).astype(int),
    })

    split_output_df.to_csv(
        REPORT_ENSEMBLE_ROOT
        / (
            f"locked_{split_name}_"
            "prediction_report.csv"
        ),
        index=False,
    )


# ------------------------------------------------------------
# 8. Confirm reporting did not modify the locked configuration
# ------------------------------------------------------------

LOCKED_CONFIG_HASH_AFTER_REPORT = (
    report_sha256_file(
        LOCKED_CONFIG_PATH
    )
)

assert (
    LOCKED_CONFIG_HASH_BEFORE_REPORT
    == LOCKED_CONFIG_HASH_AFTER_REPORT
), (
    "The locked ensemble configuration "
    "was unexpectedly modified."
)

print("\nSaved summary:")
print(PERFORMANCE_REPORT_PATH)

print(
    "\nPASS: train/val/test were evaluated "
    "using the exact locked parameters."
)

print(
    "PASS: no weight, threshold, base model, "
    "or locked configuration was modified."
)

# 12. Dataset and error diagnostics

Analyzes split composition, label associations, participant/question generalization, prediction consistency, and compact decision summaries.

In [ ]:
from pathlib import Path
from collections import OrderedDict
import hashlib
import json
import re
import warnings

import joblib
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

if "REPO_DIR" in globals():
    DIAG_REPO_DIR = Path(REPO_DIR).resolve()
elif "REPO_DIR_ENSEMBLE" in globals():
    DIAG_REPO_DIR = Path(REPO_DIR_ENSEMBLE).resolve()
else:
    DIAG_REPO_DIR = Path(
        "/workspace/project"
    ).resolve()

RETRAIN_ROOT_DIAG = (
    DIAG_REPO_DIR
    / "abaw11_retrained_base_models"
)

ENSEMBLE_ROOT_DIAG = (
    DIAG_REPO_DIR
    / "abaw11_retrained_ensemble_weights_v1"
)

DIAGNOSTIC_ROOT = (
    DIAG_REPO_DIR
    / "abaw11_participant_question_diagnostics_v1"
)

DIAGNOSTIC_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_CANDIDATES = [
    (
        DIAG_REPO_DIR
        / "abaw11_append_results"
        / "current_labelled_manifest.csv"
    ),
    (
        DIAG_REPO_DIR
        / "abaw11_append_results"
        / "current_labeled_manifest.csv"
    ),
]

LABELLED_MANIFEST_PATH = next(
    (
        path for path in MANIFEST_CANDIDATES
        if path.exists()
    ),
    None,
)

assert LABELLED_MANIFEST_PATH is not None, (
    "Could not locate current_labelled_manifest.csv"
)

TRAIN_VAL_PROBABILITY_PATH = (
    RETRAIN_ROOT_DIAG
    / "winner_probabilities_train_val.npz"
)

TEST_PROBABILITY_PATH = (
    RETRAIN_ROOT_DIAG
    / "winner_probabilities_test.npz"
)

LOCKED_ENSEMBLE_PATH = (
    ENSEMBLE_ROOT_DIAG
    / "locked_primary_ensemble.joblib"
)

for path in [
    TRAIN_VAL_PROBABILITY_PATH,
    TEST_PROBABILITY_PATH,
    LOCKED_ENSEMBLE_PATH,
]:
    assert path.exists(), path


def parse_participant_question(video_id):
    filename = Path(
        str(video_id).replace("\\", "/")
    ).name

    participant_match = re.match(
        r"(\d+)_",
        filename,
    )

    question_match = re.search(
        r"_Question_(\d+)_",
        filename,
        flags=re.IGNORECASE,
    )

    participant = (
        participant_match.group(1)
        if participant_match
        else None
    )

    question = (
        int(question_match.group(1))
        if question_match
        else None
    )

    return participant, question


labelled_df = pd.read_csv(
    LABELLED_MANIFEST_PATH
)

required_manifest_columns = {
    "split",
    "order",
    "video_id",
    "label",
}

assert required_manifest_columns.issubset(
    labelled_df.columns
), labelled_df.columns

labelled_df = (
    labelled_df.loc[
        labelled_df["split"].isin(
            ["train", "val", "test"]
        )
    ]
    .copy()
)

labelled_df["label"] = (
    labelled_df["label"]
    .astype(int)
)

parsed_metadata = labelled_df[
    "video_id"
].apply(parse_participant_question)

labelled_df["participant"] = [
    value[0] for value in parsed_metadata
]

labelled_df["question"] = [
    value[1] for value in parsed_metadata
]

assert labelled_df["participant"].notna().all()
assert labelled_df["question"].notna().all()
assert labelled_df["video_id"].is_unique

EXPECTED_SPLIT_COUNTS = {
    "train": 778,
    "val": 124,
    "test": 525,
}

for split_name, expected_count in (
    EXPECTED_SPLIT_COUNTS.items()
):
    observed_count = int(
        (labelled_df["split"] == split_name)
        .sum()
    )

    assert observed_count == expected_count, (
        split_name,
        observed_count,
        expected_count,
    )

locked_config_diag = joblib.load(
    LOCKED_ENSEMBLE_PATH
)

locked_combo_names_diag = [
    str(value)
    for value in locked_config_diag[
        "combo_names"
    ]
]

locked_weights_diag = np.asarray(
    locked_config_diag["weights"],
    dtype=np.float64,
)

locked_thresholds_diag = np.asarray(
    locked_config_diag["fixed_thresholds"],
    dtype=np.float64,
)

locked_final_threshold_diag = float(
    locked_config_diag.get(
        "final_threshold",
        0.5,
    )
)

assert locked_weights_diag.shape == (15,)
assert locked_thresholds_diag.shape == (15,)
assert np.isclose(
    locked_weights_diag.sum(),
    1.0,
)

train_val_archive_diag = np.load(
    TRAIN_VAL_PROBABILITY_PATH,
    allow_pickle=True,
)

test_archive_diag = np.load(
    TEST_PROBABILITY_PATH,
    allow_pickle=True,
)

probability_by_split = {
    "train": np.asarray(
        train_val_archive_diag["P_train"],
        dtype=np.float64,
    ),
    "val": np.asarray(
        train_val_archive_diag["P_val"],
        dtype=np.float64,
    ),
    "test": np.asarray(
        test_archive_diag["P_test"],
        dtype=np.float64,
    ),
}

label_by_split = {
    "train": np.asarray(
        train_val_archive_diag["y_train"],
        dtype=int,
    ),
    "val": np.asarray(
        train_val_archive_diag["y_val"],
        dtype=int,
    ),
    "test": np.asarray(
        test_archive_diag["y_test"],
        dtype=int,
    ),
}


def apply_locked_ensemble_diag(probabilities):
    votes = (
        probabilities
        >= locked_thresholds_diag.reshape(1, -1)
    ).astype(np.float64)

    scores = votes @ locked_weights_diag

    prediction = (
        scores >= locked_final_threshold_diag
    ).astype(int)

    return prediction, scores


analysis_parts = []

for split_name in [
    "train",
    "val",
    "test",
]:
    split_rows = (
        labelled_df.loc[
            labelled_df["split"] == split_name
        ]
        .sort_values("order")
        .reset_index(drop=True)
    )

    probabilities = probability_by_split[
        split_name
    ]

    labels = label_by_split[
        split_name
    ]

    assert probabilities.shape == (
        len(split_rows),
        15,
    )

    np.testing.assert_array_equal(
        split_rows["label"].to_numpy(),
        labels,
    )

    prediction, score = (
        apply_locked_ensemble_diag(
            probabilities
        )
    )

    split_rows["prediction"] = prediction
    split_rows["ensemble_score"] = score
    split_rows["correct"] = (
        prediction == labels
    ).astype(int)
    split_rows["error"] = (
        prediction != labels
    ).astype(int)

    analysis_parts.append(split_rows)

analysis_df = pd.concat(
    analysis_parts,
    ignore_index=True,
)

analysis_df.to_csv(
    DIAGNOSTIC_ROOT
    / "labeled_train_val_test_with_metadata_predictions.csv",
    index=False,
)

print("Labeled manifest:", LABELLED_MANIFEST_PATH)
print("Diagnostic output:", DIAGNOSTIC_ROOT)
print("\nRows by split:")
print(analysis_df["split"].value_counts())
print("\nParticipants by split:")
print(
    analysis_df.groupby("split")[
        "participant"
    ].nunique()
)
print("\nQuestions:")
print(sorted(analysis_df["question"].unique()))

print(
    "\nPASS: train, val, and test were aligned "
    "with the exact locked ensemble predictions."
)

In [ ]:
split_summary_rows = []

for split_name, frame in analysis_df.groupby(
    "split",
    sort=False,
):
    split_summary_rows.append({
        "split": split_name,
        "videos": len(frame),
        "participants": frame[
            "participant"
        ].nunique(),
        "questions_present": frame[
            "question"
        ].nunique(),
        "label_0": int(
            (frame["label"] == 0).sum()
        ),
        "label_1": int(
            (frame["label"] == 1).sum()
        ),
        "positive_rate": float(
            frame["label"].mean()
        ),
        "accuracy": float(
            frame["correct"].mean()
        ),
        "macro_f1": float(
            f1_score(
                frame["label"],
                frame["prediction"],
                average="macro",
                zero_division=0,
            )
        ),
    })

split_summary_df = pd.DataFrame(
    split_summary_rows
)

display(split_summary_df)

split_summary_df.to_csv(
    DIAGNOSTIC_ROOT
    / "split_summary.csv",
    index=False,
)

participant_sets = {
    split_name: set(
        analysis_df.loc[
            analysis_df["split"] == split_name,
            "participant",
        ]
    )
    for split_name in [
        "train",
        "val",
        "test",
    ]
}

overlap_rows = []

for split_a in [
    "train",
    "val",
    "test",
]:
    for split_b in [
        "train",
        "val",
        "test",
    ]:
        overlap = (
            participant_sets[split_a]
            & participant_sets[split_b]
        )

        overlap_rows.append({
            "split_a": split_a,
            "split_b": split_b,
            "participant_overlap": len(
                overlap
            ),
            "overlap_participants": ";".join(
                sorted(overlap)
            ),
        })

participant_overlap_df = pd.DataFrame(
    overlap_rows
)

display(
    participant_overlap_df.pivot(
        index="split_a",
        columns="split_b",
        values="participant_overlap",
    )
)

participant_overlap_df.to_csv(
    DIAGNOSTIC_ROOT
    / "participant_overlap_between_splits.csv",
    index=False,
)

question_distribution_df = (
    analysis_df.groupby(
        ["split", "question"],
        as_index=False,
    )
    .agg(
        videos=("label", "size"),
        participants=(
            "participant",
            "nunique",
        ),
        label_0=(
            "label",
            lambda x: int((x == 0).sum()),
        ),
        label_1=(
            "label",
            lambda x: int((x == 1).sum()),
        ),
        positive_rate=("label", "mean"),
        accuracy=("correct", "mean"),
        error_rate=("error", "mean"),
    )
)

display(question_distribution_df)

question_distribution_df.to_csv(
    DIAGNOSTIC_ROOT
    / "question_distribution_and_performance_by_split.csv",
    index=False,
)

participant_distribution_df = (
    analysis_df.groupby(
        ["split", "participant"],
        as_index=False,
    )
    .agg(
        videos=("label", "size"),
        questions=(
            "question",
            "nunique",
        ),
        label_0=(
            "label",
            lambda x: int((x == 0).sum()),
        ),
        label_1=(
            "label",
            lambda x: int((x == 1).sum()),
        ),
        positive_rate=("label", "mean"),
        accuracy=("correct", "mean"),
        error_rate=("error", "mean"),
        mean_score=(
            "ensemble_score",
            "mean",
        ),
    )
)

display(
    participant_distribution_df.groupby(
        "split"
    )[
        [
            "videos",
            "questions",
            "positive_rate",
            "accuracy",
            "error_rate",
        ]
    ].describe()
)

participant_distribution_df.to_csv(
    DIAGNOSTIC_ROOT
    / "participant_distribution_and_performance_by_split.csv",
    index=False,
)

all_cross_split_overlaps = {
    pair: sorted(
        participant_sets[pair[0]]
        & participant_sets[pair[1]]
    )
    for pair in [
        ("train", "val"),
        ("train", "test"),
        ("val", "test"),
    ]
}

print("\nCross-split participant overlap:")
for pair, overlap in all_cross_split_overlaps.items():
    print(pair, len(overlap), overlap[:20])

if all(
    len(value) == 0
    for value in all_cross_split_overlaps.values()
):
    print(
        "\nFinding: official splits are participant-disjoint."
    )
    print(
        "Participant-aware CV is still important inside train "
        "because random video CV would leak participant identity."
    )
else:
    print(
        "\nWARNING: at least one participant appears in "
        "multiple official splits."
    )

In [ ]:
def cramers_v(contingency):
    contingency = np.asarray(
        contingency,
        dtype=np.float64,
    )

    if contingency.size == 0:
        return np.nan

    chi2, _, _, _ = chi2_contingency(
        contingency,
        correction=False,
    )

    n = contingency.sum()
    rows, columns = contingency.shape

    denominator = min(
        rows - 1,
        columns - 1,
    )

    if n <= 0 or denominator <= 0:
        return np.nan

    return float(
        np.sqrt(
            (chi2 / n) / denominator
        )
    )


def binary_group_eta_squared(
    binary_values,
    groups,
):
    y = np.asarray(
        binary_values,
        dtype=np.float64,
    )

    groups = np.asarray(groups)

    total_ss = np.sum(
        np.square(y - y.mean())
    )

    if total_ss <= 1e-12:
        return 0.0

    between_ss = 0.0

    for group_value in np.unique(groups):
        mask = groups == group_value

        between_ss += (
            mask.sum()
            * np.square(
                y[mask].mean() - y.mean()
            )
        )

    return float(
        between_ss / total_ss
    )


def prepare_strata_indices(strata):
    strata = np.asarray(
        strata,
        dtype=object,
    )

    mapping = OrderedDict()

    for index, value in enumerate(strata):
        mapping.setdefault(value, []).append(index)

    return [
        np.asarray(indices, dtype=int)
        for indices in mapping.values()
        if len(indices) >= 2
    ]


def stratified_permutation_test(
    binary_values,
    groups,
    strata,
    *,
    number_of_permutations=3000,
    random_state=2026,
):
    y = np.asarray(
        binary_values,
        dtype=int,
    )

    groups = np.asarray(groups)
    strata_indices = prepare_strata_indices(
        strata
    )

    observed = binary_group_eta_squared(
        y,
        groups,
    )

    rng = np.random.default_rng(
        random_state
    )

    exceedances = 0

    for _ in range(number_of_permutations):
        permuted = y.copy()

        for indices in strata_indices:
            permuted[indices] = rng.permutation(
                permuted[indices]
            )

        statistic = binary_group_eta_squared(
            permuted,
            groups,
        )

        if statistic >= observed - 1e-15:
            exceedances += 1

    p_value = (
        exceedances + 1
    ) / (
        number_of_permutations + 1
    )

    return observed, float(p_value)


def benjamini_hochberg(p_values):
    p_values = np.asarray(
        p_values,
        dtype=np.float64,
    )

    order = np.argsort(p_values)
    ranked = p_values[order]

    adjusted = np.empty_like(ranked)
    running_minimum = 1.0

    for reverse_index in range(
        len(ranked) - 1,
        -1,
        -1,
    ):
        rank = reverse_index + 1

        candidate = (
            ranked[reverse_index]
            * len(ranked)
            / rank
        )

        running_minimum = min(
            running_minimum,
            candidate,
        )

        adjusted[reverse_index] = (
            running_minimum
        )

    output = np.empty_like(adjusted)
    output[order] = np.minimum(
        adjusted,
        1.0,
    )

    return output


association_rows = []

analysis_scopes = OrderedDict([
    (
        "train",
        analysis_df.loc[
            analysis_df["split"] == "train"
        ].copy(),
    ),
    (
        "val",
        analysis_df.loc[
            analysis_df["split"] == "val"
        ].copy(),
    ),
    (
        "test",
        analysis_df.loc[
            analysis_df["split"] == "test"
        ].copy(),
    ),
    (
        "pooled_train_val_test",
        analysis_df.copy(),
    ),
])

for scope_name, frame in (
    analysis_scopes.items()
):
    if scope_name == "pooled_train_val_test":
        participant_control_strata = (
            frame["split"].astype(str)
            + "::Q"
            + frame["question"].astype(str)
        )

        question_control_strata = (
            frame["split"].astype(str)
            + "::P"
            + frame["participant"].astype(str)
        )
    else:
        participant_control_strata = (
            "Q"
            + frame["question"].astype(str)
        )

        question_control_strata = (
            "P"
            + frame["participant"].astype(str)
        )

    # Direct question-label contingency association.
    question_label_table = pd.crosstab(
        frame["question"],
        frame["label"],
    )

    chi2, chi_p, _, _ = chi2_contingency(
        question_label_table,
        correction=False,
    )

    association_rows.append({
        "scope": scope_name,
        "outcome": "label",
        "factor": "question",
        "test": "chi_square_cramers_v",
        "effect_size": cramers_v(
            question_label_table
        ),
        "p_value": float(chi_p),
        "permutations": 0,
        "controls_for": "none",
    })

    # Question effect after preserving participant-specific label composition.
    effect, p_value = stratified_permutation_test(
        frame["label"],
        frame["question"],
        question_control_strata,
        number_of_permutations=3000,
        random_state=2026,
    )

    association_rows.append({
        "scope": scope_name,
        "outcome": "label",
        "factor": "question",
        "test": "stratified_permutation_eta_squared",
        "effect_size": effect,
        "p_value": p_value,
        "permutations": 3000,
        "controls_for": "participant_and_split_if_pooled",
    })

    # Participant effect after preserving question-specific label composition.
    effect, p_value = stratified_permutation_test(
        frame["label"],
        frame["participant"],
        participant_control_strata,
        number_of_permutations=3000,
        random_state=2027,
    )

    association_rows.append({
        "scope": scope_name,
        "outcome": "label",
        "factor": "participant",
        "test": "stratified_permutation_eta_squared",
        "effect_size": effect,
        "p_value": p_value,
        "permutations": 3000,
        "controls_for": "question_and_split_if_pooled",
    })

    # Question association with locked-model errors.
    question_error_table = pd.crosstab(
        frame["question"],
        frame["error"],
    )

    chi2, chi_p, _, _ = chi2_contingency(
        question_error_table,
        correction=False,
    )

    association_rows.append({
        "scope": scope_name,
        "outcome": "locked_model_error",
        "factor": "question",
        "test": "chi_square_cramers_v",
        "effect_size": cramers_v(
            question_error_table
        ),
        "p_value": float(chi_p),
        "permutations": 0,
        "controls_for": "none",
    })

    effect, p_value = stratified_permutation_test(
        frame["error"],
        frame["question"],
        question_control_strata,
        number_of_permutations=3000,
        random_state=2028,
    )

    association_rows.append({
        "scope": scope_name,
        "outcome": "locked_model_error",
        "factor": "question",
        "test": "stratified_permutation_eta_squared",
        "effect_size": effect,
        "p_value": p_value,
        "permutations": 3000,
        "controls_for": "participant_and_split_if_pooled",
    })

    effect, p_value = stratified_permutation_test(
        frame["error"],
        frame["participant"],
        participant_control_strata,
        number_of_permutations=3000,
        random_state=2029,
    )

    association_rows.append({
        "scope": scope_name,
        "outcome": "locked_model_error",
        "factor": "participant",
        "test": "stratified_permutation_eta_squared",
        "effect_size": effect,
        "p_value": p_value,
        "permutations": 3000,
        "controls_for": "question_and_split_if_pooled",
    })

association_df = pd.DataFrame(
    association_rows
)

association_df["q_value_bh"] = (
    benjamini_hochberg(
        association_df["p_value"]
        .to_numpy()
    )
)

association_df["significant_q_0_05"] = (
    association_df["q_value_bh"]
    < 0.05
)

display(association_df)

association_df.to_csv(
    DIAGNOSTIC_ROOT
    / "participant_question_association_tests.csv",
    index=False,
)

print(
    "\nInterpretation:"
)
print(
    "- Question chi-square/Cramer's V tests raw association."
)
print(
    "- Question permutation preserves each participant's "
    "label composition, testing question beyond participant."
)
print(
    "- Participant permutation preserves each question's "
    "label composition, testing participant beyond question."
)
print(
    "- Use BH q-values, not raw p-values, for significance."
)

In [ ]:
def safe_binary_metrics(frame):
    labels = frame["label"].to_numpy()
    predictions = frame[
        "prediction"
    ].to_numpy()

    matrix = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    )

    return pd.Series({
        "videos": len(frame),
        "participants": frame[
            "participant"
        ].nunique(),
        "positive_rate": float(
            labels.mean()
        ),
        "predicted_positive_rate": float(
            predictions.mean()
        ),
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "macro_f1": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "class_0_f1": float(
            f1_score(
                labels,
                predictions,
                labels=[0],
                average=None,
                zero_division=0,
            )[0]
        ),
        "class_1_f1": float(
            f1_score(
                labels,
                predictions,
                labels=[1],
                average=None,
                zero_division=0,
            )[0]
        ),
        "tn": int(matrix[0, 0]),
        "fp": int(matrix[0, 1]),
        "fn": int(matrix[1, 0]),
        "tp": int(matrix[1, 1]),
        "mean_score": float(
            frame[
                "ensemble_score"
            ].mean()
        ),
    })


question_performance_df = (
    analysis_df.groupby(
        ["split", "question"],
        group_keys=False,
    )
    .apply(
        safe_binary_metrics,
        include_groups=False,
    )
    .reset_index()
)

display(question_performance_df)

question_performance_df.to_csv(
    DIAGNOSTIC_ROOT
    / "locked_ensemble_performance_by_split_question.csv",
    index=False,
)

question_gap_pivot = (
    question_performance_df.pivot(
        index="question",
        columns="split",
        values="macro_f1",
    )
)

for required_column in [
    "train",
    "val",
    "test",
]:
    if required_column not in question_gap_pivot:
        question_gap_pivot[
            required_column
        ] = np.nan

question_gap_pivot[
    "val_minus_test"
] = (
    question_gap_pivot["val"]
    - question_gap_pivot["test"]
)

question_gap_pivot[
    "train_minus_test"
] = (
    question_gap_pivot["train"]
    - question_gap_pivot["test"]
)

question_gap_df = (
    question_gap_pivot
    .reset_index()
)

display(question_gap_df)

question_gap_df.to_csv(
    DIAGNOSTIC_ROOT
    / "question_generalization_gaps.csv",
    index=False,
)

participant_performance_df = (
    analysis_df.groupby(
        ["split", "participant"],
        as_index=False,
    )
    .agg(
        videos=("label", "size"),
        questions=(
            "question",
            "nunique",
        ),
        positive_rate=("label", "mean"),
        predicted_positive_rate=(
            "prediction",
            "mean",
        ),
        accuracy=("correct", "mean"),
        errors=("error", "sum"),
        mean_score=(
            "ensemble_score",
            "mean",
        ),
    )
)

participant_performance_df.to_csv(
    DIAGNOSTIC_ROOT
    / "locked_ensemble_performance_by_split_participant.csv",
    index=False,
)

print("\nLowest-accuracy participants with at least 3 videos:")

for split_name in [
    "train",
    "val",
    "test",
]:
    print("\n", split_name.upper())

    display(
        participant_performance_df.loc[
            (
                participant_performance_df[
                    "split"
                ] == split_name
            )
            & (
                participant_performance_df[
                    "videos"
                ] >= 3
            )
        ]
        .sort_values(
            [
                "accuracy",
                "videos",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .head(15)
    )

In [ ]:
# ============================================================
# Correct D3 per-question Macro-F1 for absent classes
# ============================================================

corrected_question_rows = []

for (
    split_name,
    question_id,
), frame in analysis_df.groupby(
    ["split", "question"],
):
    labels = frame["label"].to_numpy()
    predictions = frame[
        "prediction"
    ].to_numpy()

    class_f1 = f1_score(
        labels,
        predictions,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )

    corrected_question_rows.append({
        "split": split_name,
        "question": question_id,
        "videos": len(frame),
        "true_classes_present": ",".join(
            map(
                str,
                sorted(np.unique(labels)),
            )
        ),
        "positive_rate": float(
            labels.mean()
        ),
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "two_class_macro_f1": float(
            f1_score(
                labels,
                predictions,
                labels=[0, 1],
                average="macro",
                zero_division=0,
            )
        ),
        "class_0_f1": float(
            class_f1[0]
        ),
        "class_1_f1": float(
            class_f1[1]
        ),
    })

corrected_question_performance_df = (
    pd.DataFrame(
        corrected_question_rows
    )
    .sort_values(
        ["split", "question"]
    )
    .reset_index(drop=True)
)

display(corrected_question_performance_df)

corrected_question_performance_df.to_csv(
    DIAGNOSTIC_ROOT
    / "corrected_two_class_per_question_metrics.csv",
    index=False,
)

In [ ]:
def build_question_matrix(frame):
    questions = frame[
        "question"
    ].astype(int)

    matrix = pd.get_dummies(
        questions,
        prefix="question",
        dtype=float,
    )

    expected_columns = [
        f"question_{value}"
        for value in range(1, 8)
    ]

    return (
        matrix.reindex(
            columns=expected_columns,
            fill_value=0.0,
        )
        .to_numpy(dtype=np.float64)
    )


train_frame_meta = (
    analysis_df.loc[
        analysis_df["split"] == "train"
    ]
    .sort_values("order")
    .reset_index(drop=True)
)

val_frame_meta = (
    analysis_df.loc[
        analysis_df["split"] == "val"
    ]
    .sort_values("order")
    .reset_index(drop=True)
)

test_frame_meta = (
    analysis_df.loc[
        analysis_df["split"] == "test"
    ]
    .sort_values("order")
    .reset_index(drop=True)
)

X_question_train = build_question_matrix(
    train_frame_meta
)

X_question_val = build_question_matrix(
    val_frame_meta
)

X_question_test = build_question_matrix(
    test_frame_meta
)

y_question_train = train_frame_meta[
    "label"
].to_numpy()

y_question_val = val_frame_meta[
    "label"
].to_numpy()

y_question_test = test_frame_meta[
    "label"
].to_numpy()

groups_question_train = train_frame_meta[
    "participant"
].to_numpy()

QUESTION_MODEL_C = 0.10

question_model_oof_probability = np.zeros(
    len(train_frame_meta),
    dtype=np.float64,
)

question_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=2026,
)

for fold_index, (
    fit_indices,
    holdout_indices,
) in enumerate(
    question_cv.split(
        X_question_train,
        y_question_train,
        groups=groups_question_train,
    ),
    start=1,
):
    model = LogisticRegression(
        penalty="l2",
        C=QUESTION_MODEL_C,
        class_weight="balanced",
        solver="liblinear",
        max_iter=5000,
        random_state=2026 + fold_index,
    )

    model.fit(
        X_question_train[
            fit_indices
        ],
        y_question_train[
            fit_indices
        ],
    )

    question_model_oof_probability[
        holdout_indices
    ] = model.predict_proba(
        X_question_train[
            holdout_indices
        ]
    )[:, 1]

final_question_model = LogisticRegression(
    penalty="l2",
    C=QUESTION_MODEL_C,
    class_weight="balanced",
    solver="liblinear",
    max_iter=5000,
    random_state=2032,
)

final_question_model.fit(
    X_question_train,
    y_question_train,
)

question_probability_by_split = {
    "train_grouped_oof": (
        y_question_train,
        question_model_oof_probability,
    ),
    "val_fixed_model": (
        y_question_val,
        final_question_model.predict_proba(
            X_question_val
        )[:, 1],
    ),
    "test_fixed_model": (
        y_question_test,
        final_question_model.predict_proba(
            X_question_test
        )[:, 1],
    ),
}


def probability_metric_row(
    name,
    labels,
    probabilities,
):
    prediction = (
        probabilities >= 0.5
    ).astype(int)

    row = {
        "evaluation": name,
        "samples": len(labels),
        "macro_f1": float(
            f1_score(
                labels,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "accuracy": float(
            accuracy_score(
                labels,
                prediction,
            )
        ),
        "average_precision": float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        "brier_score": float(
            brier_score_loss(
                labels,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                labels,
                np.clip(
                    probabilities,
                    1e-7,
                    1.0 - 1e-7,
                ),
                labels=[0, 1],
            )
        ),
        "predicted_positive_rate": float(
            prediction.mean()
        ),
    }

    if len(np.unique(labels)) == 2:
        row["roc_auc"] = float(
            roc_auc_score(
                labels,
                probabilities,
            )
        )
    else:
        row["roc_auc"] = np.nan

    return row


question_model_results_df = pd.DataFrame([
    probability_metric_row(
        name,
        labels,
        probabilities,
    )
    for name, (
        labels,
        probabilities,
    ) in question_probability_by_split.items()
])

display(question_model_results_df)

question_model_results_df.to_csv(
    DIAGNOSTIC_ROOT
    / "question_only_regularized_logistic_results.csv",
    index=False,
)

question_coefficients_df = pd.DataFrame({
    "question": np.arange(1, 8),
    "coefficient": (
        final_question_model.coef_
        .reshape(-1)
    ),
})

display(question_coefficients_df)

question_coefficients_df.to_csv(
    DIAGNOSTIC_ROOT
    / "question_only_logistic_coefficients.csv",
    index=False,
)


def participant_leave_one_out_prior(
    frame,
    smoothing_strength=2.0,
):
    labels = frame[
        "label"
    ].to_numpy(dtype=float)

    participants = frame[
        "participant"
    ].to_numpy()

    global_rate = float(
        labels.mean()
    )

    probabilities = np.full(
        len(frame),
        global_rate,
        dtype=np.float64,
    )

    participant_sizes = np.zeros(
        len(frame),
        dtype=int,
    )

    for participant in np.unique(
        participants
    ):
        indices = np.flatnonzero(
            participants == participant
        )

        participant_sizes[
            indices
        ] = len(indices)

        participant_label_sum = (
            labels[indices].sum()
        )

        for index in indices:
            other_count = len(indices) - 1
            other_sum = (
                participant_label_sum
                - labels[index]
            )

            probabilities[index] = (
                other_sum
                + smoothing_strength
                * global_rate
            ) / (
                other_count
                + smoothing_strength
            )

    return probabilities, participant_sizes


participant_prior_rows = []

for split_name, frame in (
    analysis_df.groupby(
        "split",
        sort=False,
    )
):
    frame = frame.sort_values(
        "order"
    ).reset_index(drop=True)

    probabilities, participant_sizes = (
        participant_leave_one_out_prior(
            frame,
            smoothing_strength=2.0,
        )
    )

    eligible = participant_sizes >= 2

    result = probability_metric_row(
        (
            f"{split_name}_participant_"
            "leave_one_video_out_prior"
        ),
        frame.loc[
            eligible,
            "label",
        ].to_numpy(),
        probabilities[eligible],
    )

    result["eligible_videos"] = int(
        eligible.sum()
    )

    result[
        "participants_with_2plus_videos"
    ] = int(
        frame.loc[
            eligible,
            "participant",
        ].nunique()
    )

    participant_prior_rows.append(
        result
    )

participant_prior_results_df = (
    pd.DataFrame(
        participant_prior_rows
    )
)

display(participant_prior_results_df)

participant_prior_results_df.to_csv(
    DIAGNOSTIC_ROOT
    / "participant_leave_one_out_prior_results.csv",
    index=False,
)

print(
    "\nInterpretation guidance:"
)
print(
    "- Question-only grouped-OOF train and fixed val/test "
    "scores indicate whether Question-ID generalizes."
)
print(
    "- Participant leave-one-out prior measures within-participant "
    "label consistency only."
)
print(
    "- If official participants are disjoint, participant ID itself "
    "cannot be used as a deployable categorical feature."
)
print(
    "- Even if participant prior is weak, participant-aware CV can "
    "still be necessary to prevent identity leakage."
)

In [ ]:
association_results = pd.read_csv(
    DIAGNOSTIC_ROOT
    / "participant_question_association_tests.csv"
)

question_results = pd.read_csv(
    DIAGNOSTIC_ROOT
    / "question_only_regularized_logistic_results.csv"
)

participant_prior_results = pd.read_csv(
    DIAGNOSTIC_ROOT
    / "participant_leave_one_out_prior_results.csv"
)

cross_split_overlap_count = sum(
    len(value)
    for value in all_cross_split_overlaps.values()
)

pooled_associations = (
    association_results.loc[
        (
            association_results["scope"]
            == "pooled_train_val_test"
        )
        & (
            association_results["test"]
            == "stratified_permutation_eta_squared"
        )
    ]
    .copy()
)

decision_summary = {
    "official_split_participant_overlap_total": int(
        cross_split_overlap_count
    ),
    "question_label_pooled_q_value": None,
    "participant_label_pooled_q_value": None,
    "question_error_pooled_q_value": None,
    "participant_error_pooled_q_value": None,
    "question_only_val_macro_f1": None,
    "question_only_test_macro_f1": None,
}

for _, row in pooled_associations.iterrows():
    key = (
        str(row["factor"]),
        str(row["outcome"]),
    )

    if key == ("question", "label"):
        decision_summary[
            "question_label_pooled_q_value"
        ] = float(row["q_value_bh"])

    elif key == ("participant", "label"):
        decision_summary[
            "participant_label_pooled_q_value"
        ] = float(row["q_value_bh"])

    elif key == (
        "question",
        "locked_model_error",
    ):
        decision_summary[
            "question_error_pooled_q_value"
        ] = float(row["q_value_bh"])

    elif key == (
        "participant",
        "locked_model_error",
    ):
        decision_summary[
            "participant_error_pooled_q_value"
        ] = float(row["q_value_bh"])

for _, row in question_results.iterrows():
    if row["evaluation"] == "val_fixed_model":
        decision_summary[
            "question_only_val_macro_f1"
        ] = float(row["macro_f1"])

    elif row["evaluation"] == "test_fixed_model":
        decision_summary[
            "question_only_test_macro_f1"
        ] = float(row["macro_f1"])

with (
    DIAGNOSTIC_ROOT
    / "decision_summary.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        decision_summary,
        handle,
        indent=2,
    )

print(json.dumps(
    decision_summary,
    indent=2,
))

print(
    "\nSuggested interpretation rules:"
)
print(
    "1. Use Question-ID only if its grouped-OOF/train and fixed "
    "val/test performance is consistently above a constant baseline, "
    "not merely because one chi-square p-value is small."
)
print(
    "2. Use participant-aware CV whenever participants have multiple "
    "videos, especially when official splits are participant-disjoint."
)
print(
    "3. Do not add Participant ID as a model feature when val/test "
    "participants are unseen."
)
print(
    "4. Test error associations are diagnostic only and must not be "
    "used to hand-pick questions, participants, thresholds, or weights."
)
print("\nSaved all outputs under:")
print(DIAGNOSTIC_ROOT)

In [ ]:
# ============================================================
# RESTORE — Activate released weights in a new isolated config
# ============================================================

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd


RESTORED_ROOT = (
    AUDIT_REPO
    / "abaw11_released_weights_restored_v1"
)

RESTORED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert RESTORED_ROOT.resolve() not in {
    (
        AUDIT_REPO
        / "CALMAH_v1"
    ).resolve(),
    RETRAINED_ROOT.resolve(),
    ENSEMBLE_EXPERIMENT_ROOT.resolve(),
}


# ------------------------------------------------------------
# 1. Freeze active parameters in memory
# ------------------------------------------------------------

ACTIVE_COMBO_NAMES = list(
    retrained_combo_names
)

ACTIVE_WEIGHTS = (
    original_released_weights.copy()
)

ACTIVE_THRESHOLDS = (
    retrained_thresholds.copy()
)

ACTIVE_FINAL_SCORE_THRESHOLD = float(
    ACTIVE_WEIGHTS.sum() / 2.0
)

assert ACTIVE_WEIGHTS.shape == (15,)
assert ACTIVE_THRESHOLDS.shape == (15,)


# ------------------------------------------------------------
# 2. Save a new independent locked configuration
# ------------------------------------------------------------

restored_config = {
    "method": (
        "retrained_base_models_"
        "released_CALMAH_weights"
    ),
    "combo_names": (
        ACTIVE_COMBO_NAMES
    ),
    "weights": (
        ACTIVE_WEIGHTS.copy()
    ),
    "thresholds": (
        ACTIVE_THRESHOLDS.copy()
    ),
    "final_score_threshold": (
        ACTIVE_FINAL_SCORE_THRESHOLD
    ),
    "base_model_directory": str(
        RETRAINED_ROOT
    ),
    "weight_source": str(
        ORIGINAL_META_PATH
    ),
    "threshold_source": str(
        RETRAINED_META_PATH
    ),
    "known_labeled_test_macro_f1": (
        float(
            released_test_macro_f1
        )
    ),
    "source_files_modified": False,
}

joblib.dump(
    restored_config,
    RESTORED_ROOT
    / "locked_released_weights_config.joblib",
)

with (
    RESTORED_ROOT
    / "locked_released_weights_config.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        {
            **restored_config,
            "weights": (
                ACTIVE_WEIGHTS.tolist()
            ),
            "thresholds": (
                ACTIVE_THRESHOLDS.tolist()
            ),
        },
        handle,
        indent=2,
    )


# ------------------------------------------------------------
# 3. Load private probabilities
# ------------------------------------------------------------

PRIVATE_PROBABILITY_PATH = (
    RETRAINED_ROOT
    / "winner_probabilities_private.npz"
)

assert PRIVATE_PROBABILITY_PATH.exists(), (
    PRIVATE_PROBABILITY_PATH
)

private_archive = np.load(
    PRIVATE_PROBABILITY_PATH,
    allow_pickle=True,
)

P_private_restored = np.asarray(
    private_archive["P_private"],
    dtype=np.float64,
)

private_combo_names = [
    normalize_combo_name(value)
    for value in private_archive[
        "combo_names"
    ].tolist()
]

assert private_combo_names == (
    ACTIVE_COMBO_NAMES
)

assert P_private_restored.shape == (
    152,
    15,
)


# ------------------------------------------------------------
# 4. Apply released weights
# ------------------------------------------------------------

private_released_votes = (
    P_private_restored
    >= ACTIVE_THRESHOLDS.reshape(
        1,
        -1,
    )
).astype(np.float64)

private_released_score = (
    private_released_votes
    @ ACTIVE_WEIGHTS
)

private_released_prediction = (
    private_released_score
    >= ACTIVE_FINAL_SCORE_THRESHOLD
).astype(int)


# ------------------------------------------------------------
# 5. Load private submission order
# ------------------------------------------------------------

private_manifest_candidates = [
    (
        AUDIT_REPO
        / "abaw11_append_results"
        / "current_private_manifest.csv"
    ),
    (
        RETRAINED_ROOT
        / "private_manifest.csv"
    ),
]

private_manifest_path = next(
    (
        path
        for path in private_manifest_candidates
        if path.exists()
    ),
    None,
)

if private_manifest_path is not None:
    restored_private_df = pd.read_csv(
        private_manifest_path
    )
elif "private_df" in globals():
    restored_private_df = (
        private_df.copy()
    )
else:
    raise FileNotFoundError(
        "Could not locate current private manifest."
    )

restored_private_df = (
    restored_private_df
    .sort_values("private_order")
    .reset_index(drop=True)
)

assert len(restored_private_df) == 152
assert "submission_id" in (
    restored_private_df.columns
)
assert restored_private_df[
    "submission_id"
].is_unique


# ------------------------------------------------------------
# 6. Generate a new isolated submission
# ------------------------------------------------------------

RESTORED_SUBMISSION_PATH = (
    RESTORED_ROOT
    / "trial-retrained-released-weights.txt"
)

submission_ids = (
    restored_private_df[
        "submission_id"
    ].astype(str).tolist()
)

with RESTORED_SUBMISSION_PATH.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    for submission_id, label in zip(
        submission_ids,
        private_released_prediction,
    ):
        handle.write(
            f"{submission_id},{int(label)}\n"
        )


# ------------------------------------------------------------
# 7. Validate submission
# ------------------------------------------------------------

written_lines = [
    line.strip()
    for line in (
        RESTORED_SUBMISSION_PATH
        .read_text(
            encoding="utf-8"
        )
        .splitlines()
    )
    if line.strip()
]

assert len(written_lines) == 152

written_ids = []
written_labels = []

for line in written_lines:
    submission_id, label = line.rsplit(
        ",",
        1,
    )

    assert label in {"0", "1"}

    written_ids.append(
        submission_id
    )

    written_labels.append(
        int(label)
    )

assert written_ids == submission_ids

np.testing.assert_array_equal(
    np.asarray(written_labels),
    private_released_prediction,
)


# ------------------------------------------------------------
# 8. Save audit report
# ------------------------------------------------------------

restored_private_audit_df = (
    restored_private_df.copy()
)

restored_private_audit_df[
    "released_weight_score"
] = private_released_score

restored_private_audit_df[
    "prediction"
] = private_released_prediction

for index, combo_name in enumerate(
    ACTIVE_COMBO_NAMES
):
    safe_name = re.sub(
        r"[^A-Za-z0-9_]+",
        "_",
        combo_name,
    )

    restored_private_audit_df[
        f"vote_{safe_name}"
    ] = private_released_votes[
        :,
        index,
    ].astype(int)

restored_private_audit_df.to_csv(
    RESTORED_ROOT
    / "private_released_weights_audit.csv",
    index=False,
)

print("Restored configuration:")
print(
    RESTORED_ROOT
    / "locked_released_weights_config.joblib"
)

print("\nSubmission:")
print(RESTORED_SUBMISSION_PATH)

print("\nPrediction counts:")
print(
    dict(zip(*np.unique(
        private_released_prediction,
        return_counts=True,
    )))
)

print("\nReleased weights:")
for combo_name, weight in zip(
    ACTIVE_COMBO_NAMES,
    ACTIVE_WEIGHTS,
):
    print(
        f"{combo_name:30s} "
        f"{weight:.8f}"
    )

print(
    "\nPASS: released CALMAH weights are now "
    "the active weights in the new isolated configuration."
)

print(
    "PASS: retrained per-model thresholds were retained."
)

print(
    "PASS: no previous experiment file was overwritten."
)

In [ ]:
# ============================================================
# FINAL VERIFICATION
# Re-evaluate restored Released CALMAH weights on train/val/test
# ============================================================
#
# Read-only experiment:
# - No model training
# - No threshold tuning
# - No weight tuning
# - No test-based parameter selection
# - No existing file modification
#
# This cell verifies that:
# 1. The restored configuration uses the released CALMAH weights.
# 2. It exactly reproduces the previous R4/R5 results.
# 3. It is compared with the alternative ensemble settings.
# 4. It is compared with all 15 individual base models.

from pathlib import Path
import hashlib
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)


# ------------------------------------------------------------
# 1. Resolve paths
# ------------------------------------------------------------

VERIFY_REPO = Path(
    "/workspace/project"
).resolve()

VERIFY_RETRAINED_ROOT = (
    VERIFY_REPO
    / "abaw11_retrained_base_models"
)

VERIFY_RESTORED_ROOT = (
    VERIFY_REPO
    / "abaw11_released_weights_restored_v1"
)

VERIFY_W_ROOT = (
    VERIFY_REPO
    / "abaw11_retrained_ensemble_weights_v1"
)

VERIFY_ORIGINAL_META_PATH = (
    VERIFY_REPO
    / "CALMAH_v1"
    / "pipeline_meta.joblib"
)

VERIFY_RETRAINED_META_PATH = (
    VERIFY_RETRAINED_ROOT
    / "pipeline_meta.joblib"
)

VERIFY_RESTORED_CONFIG_PATH = (
    VERIFY_RESTORED_ROOT
    / "locked_released_weights_config.joblib"
)

VERIFY_TRAIN_VAL_PATH = (
    VERIFY_RETRAINED_ROOT
    / "winner_probabilities_train_val.npz"
)

VERIFY_TEST_PATH = (
    VERIFY_RETRAINED_ROOT
    / "winner_probabilities_test.npz"
)

required_paths = [
    VERIFY_ORIGINAL_META_PATH,
    VERIFY_RETRAINED_META_PATH,
    VERIFY_RESTORED_CONFIG_PATH,
    VERIFY_TRAIN_VAL_PATH,
    VERIFY_TEST_PATH,
]

for path in required_paths:
    assert path.exists(), path


# ------------------------------------------------------------
# 2. Record source-file hashes before verification
# ------------------------------------------------------------

def verify_sha256(path):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


VERIFY_SOURCE_PATHS = {
    "original_pipeline_meta": (
        VERIFY_ORIGINAL_META_PATH
    ),
    "retrained_pipeline_meta": (
        VERIFY_RETRAINED_META_PATH
    ),
    "restored_config": (
        VERIFY_RESTORED_CONFIG_PATH
    ),
    "train_val_probabilities": (
        VERIFY_TRAIN_VAL_PATH
    ),
    "test_probabilities": (
        VERIFY_TEST_PATH
    ),
}

VERIFY_HASHES_BEFORE = {
    name: verify_sha256(path)
    for name, path in VERIFY_SOURCE_PATHS.items()
}


# ------------------------------------------------------------
# 3. Load restored and source configurations
# ------------------------------------------------------------

def verify_combo_name(value):
    if isinstance(value, (list, tuple)):
        return "_".join(map(str, value))

    return str(value)


original_meta_verify = joblib.load(
    VERIFY_ORIGINAL_META_PATH
)

retrained_meta_verify = joblib.load(
    VERIFY_RETRAINED_META_PATH
)

restored_config_verify = joblib.load(
    VERIFY_RESTORED_CONFIG_PATH
)

verify_combo_names = [
    verify_combo_name(value)
    for value in restored_config_verify[
        "combo_names"
    ]
]

verify_released_weights = np.asarray(
    restored_config_verify["weights"],
    dtype=np.float64,
).reshape(-1)

verify_thresholds = np.asarray(
    restored_config_verify["thresholds"],
    dtype=np.float64,
).reshape(-1)

verify_final_score_threshold = float(
    restored_config_verify[
        "final_score_threshold"
    ]
)

assert len(verify_combo_names) == 15
assert verify_released_weights.shape == (15,)
assert verify_thresholds.shape == (15,)

assert np.isclose(
    verify_final_score_threshold,
    verify_released_weights.sum() / 2.0,
    rtol=0.0,
    atol=1e-12,
)


# ------------------------------------------------------------
# 4. Confirm restored weights equal original released weights
# ------------------------------------------------------------

original_released_weights_verify = np.asarray(
    original_meta_verify["weights"],
    dtype=np.float64,
).reshape(-1)

retrained_saved_weights_verify = np.asarray(
    retrained_meta_verify["weights"],
    dtype=np.float64,
).reshape(-1)

np.testing.assert_allclose(
    verify_released_weights,
    original_released_weights_verify,
    rtol=0.0,
    atol=1e-12,
)

np.testing.assert_allclose(
    verify_released_weights,
    retrained_saved_weights_verify,
    rtol=0.0,
    atol=1e-12,
)

print("PASS: restored weights equal original released weights.")
print(
    "Released weight sum:",
    verify_released_weights.sum(),
)
print(
    "Final score threshold:",
    verify_final_score_threshold,
)


# ------------------------------------------------------------
# 5. Load probability matrices
# ------------------------------------------------------------

train_val_archive_verify = np.load(
    VERIFY_TRAIN_VAL_PATH,
    allow_pickle=True,
)

test_archive_verify = np.load(
    VERIFY_TEST_PATH,
    allow_pickle=True,
)

P_train_verify = np.asarray(
    train_val_archive_verify["P_train"],
    dtype=np.float64,
)

y_train_verify = np.asarray(
    train_val_archive_verify["y_train"],
    dtype=int,
).reshape(-1)

P_val_verify = np.asarray(
    train_val_archive_verify["P_val"],
    dtype=np.float64,
)

y_val_verify = np.asarray(
    train_val_archive_verify["y_val"],
    dtype=int,
).reshape(-1)

P_test_verify = np.asarray(
    test_archive_verify["P_test"],
    dtype=np.float64,
)

y_test_verify = np.asarray(
    test_archive_verify["y_test"],
    dtype=int,
).reshape(-1)

train_val_combo_names_verify = [
    verify_combo_name(value)
    for value in train_val_archive_verify[
        "combo_names"
    ].tolist()
]

test_combo_names_verify = [
    verify_combo_name(value)
    for value in test_archive_verify[
        "combo_names"
    ].tolist()
]

assert train_val_combo_names_verify == (
    verify_combo_names
)

assert test_combo_names_verify == (
    verify_combo_names
)

assert P_train_verify.shape == (778, 15)
assert P_val_verify.shape == (124, 15)
assert P_test_verify.shape == (525, 15)


# ------------------------------------------------------------
# 6. Apply an arbitrary hard-vote weight setting
# ------------------------------------------------------------

def verify_hard_vote(
    probabilities,
    weights,
    thresholds,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    weights = np.asarray(
        weights,
        dtype=np.float64,
    ).reshape(-1)

    thresholds = np.asarray(
        thresholds,
        dtype=np.float64,
    ).reshape(-1)

    votes = (
        probabilities
        >= thresholds.reshape(1, -1)
    ).astype(np.float64)

    weighted_score = votes @ weights

    prediction = (
        weighted_score
        >= weights.sum() / 2.0
    ).astype(int)

    return prediction, weighted_score, votes


# ------------------------------------------------------------
# 7. Evaluate the restored released-weight setting
# ------------------------------------------------------------

verify_split_data = {
    "train": (
        P_train_verify,
        y_train_verify,
    ),
    "val": (
        P_val_verify,
        y_val_verify,
    ),
    "test": (
        P_test_verify,
        y_test_verify,
    ),
}

released_performance_rows = []
released_outputs = {}

for split_name, (
    probabilities,
    labels,
) in verify_split_data.items():

    prediction, score, votes = (
        verify_hard_vote(
            probabilities,
            verify_released_weights,
            verify_thresholds,
        )
    )

    matrix = confusion_matrix(
        labels,
        prediction,
        labels=[0, 1],
    )

    class_f1 = f1_score(
        labels,
        prediction,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )

    released_outputs[split_name] = {
        "prediction": prediction,
        "score": score,
        "votes": votes,
    }

    released_performance_rows.append({
        "split": split_name,
        "samples": len(labels),

        "accuracy": float(
            accuracy_score(
                labels,
                prediction,
            )
        ),

        "macro_f1": float(
            f1_score(
                labels,
                prediction,
                labels=[0, 1],
                average="macro",
                zero_division=0,
            )
        ),

        "class_0_f1": float(
            class_f1[0]
        ),

        "class_1_f1": float(
            class_f1[1]
        ),

        "predicted_0": int(
            (prediction == 0).sum()
        ),

        "predicted_1": int(
            (prediction == 1).sum()
        ),

        "tn": int(matrix[0, 0]),
        "fp": int(matrix[0, 1]),
        "fn": int(matrix[1, 0]),
        "tp": int(matrix[1, 1]),
    })

released_performance_df = pd.DataFrame(
    released_performance_rows
)

print("\n" + "=" * 100)
print("RESTORED RELEASED-WEIGHT PERFORMANCE")
print("=" * 100)

display(released_performance_df)


# ------------------------------------------------------------
# 8. Reproduce the known R4/R5 results exactly
# ------------------------------------------------------------

EXPECTED_R4_TRAIN_F1 = (
    0.9627219179327038
)

EXPECTED_R4_VAL_F1 = (
    0.7168949771689497
)

EXPECTED_R5_TEST_F1 = (
    0.7307332084559808
)

observed_train_f1 = float(
    released_performance_df.loc[
        released_performance_df["split"]
        == "train",
        "macro_f1",
    ].iloc[0]
)

observed_val_f1 = float(
    released_performance_df.loc[
        released_performance_df["split"]
        == "val",
        "macro_f1",
    ].iloc[0]
)

observed_test_f1 = float(
    released_performance_df.loc[
        released_performance_df["split"]
        == "test",
        "macro_f1",
    ].iloc[0]
)

assert np.isclose(
    observed_train_f1,
    EXPECTED_R4_TRAIN_F1,
    rtol=0.0,
    atol=1e-12,
), (
    observed_train_f1,
    EXPECTED_R4_TRAIN_F1,
)

assert np.isclose(
    observed_val_f1,
    EXPECTED_R4_VAL_F1,
    rtol=0.0,
    atol=1e-12,
), (
    observed_val_f1,
    EXPECTED_R4_VAL_F1,
)

assert np.isclose(
    observed_test_f1,
    EXPECTED_R5_TEST_F1,
    rtol=0.0,
    atol=1e-12,
), (
    observed_test_f1,
    EXPECTED_R5_TEST_F1,
)

print(
    "PASS: restored setting exactly reproduces "
    "R4 train Macro-F1."
)

print(
    "PASS: restored setting exactly reproduces "
    "R4 validation Macro-F1."
)

print(
    "PASS: restored setting exactly reproduces "
    "R5 test Macro-F1."
)


# ------------------------------------------------------------
# 9. Evaluate all individual base models
# ------------------------------------------------------------

individual_rows = []

for model_index, combo_name in enumerate(
    verify_combo_names
):
    threshold = verify_thresholds[
        model_index
    ]

    for split_name, (
        probabilities,
        labels,
    ) in verify_split_data.items():

        prediction = (
            probabilities[:, model_index]
            >= threshold
        ).astype(int)

        individual_rows.append({
            "method": (
                f"individual::{combo_name}"
            ),
            "category": "individual",
            "split": split_name,
            "macro_f1": float(
                f1_score(
                    labels,
                    prediction,
                    labels=[0, 1],
                    average="macro",
                    zero_division=0,
                )
            ),
            "accuracy": float(
                accuracy_score(
                    labels,
                    prediction,
                )
            ),
        })

individual_performance_df = pd.DataFrame(
    individual_rows
)


# ------------------------------------------------------------
# 10. Evaluate alternative weight controls if available
# ------------------------------------------------------------

method_weight_map = {
    "released_weights": (
        verify_released_weights
    ),
}

locked_weight_table_path_verify = (
    VERIFY_W_ROOT
    / "locked_weight_table.csv"
)

if locked_weight_table_path_verify.exists():
    locked_weight_table_verify = pd.read_csv(
        locked_weight_table_path_verify
    )

    assert (
        locked_weight_table_verify[
            "combo_name"
        ].tolist()
        == verify_combo_names
    )

    column_to_method = {
        "released_weight_control": (
            "released_weights_control"
        ),
        "equal_weight_control": (
            "equal_weights_control"
        ),
        "ap_shrunk_weight_control": (
            "ap_shrunk_weights_control"
        ),
        "regularized_consensus_primary": (
            "regularized_consensus_primary"
        ),
    }

    for column_name, method_name in (
        column_to_method.items()
    ):
        if column_name in (
            locked_weight_table_verify.columns
        ):
            method_weight_map[
                method_name
            ] = (
                locked_weight_table_verify[
                    column_name
                ]
                .to_numpy(dtype=np.float64)
            )

alternative_rows = []

for method_name, weights in (
    method_weight_map.items()
):
    for split_name, (
        probabilities,
        labels,
    ) in verify_split_data.items():

        prediction, score, votes = (
            verify_hard_vote(
                probabilities,
                weights,
                verify_thresholds,
            )
        )

        alternative_rows.append({
            "method": method_name,
            "category": "ensemble",
            "split": split_name,
            "macro_f1": float(
                f1_score(
                    labels,
                    prediction,
                    labels=[0, 1],
                    average="macro",
                    zero_division=0,
                )
            ),
            "accuracy": float(
                accuracy_score(
                    labels,
                    prediction,
                )
            ),
            "predicted_0": int(
                (prediction == 0).sum()
            ),
            "predicted_1": int(
                (prediction == 1).sum()
            ),
        })

alternative_performance_df = pd.DataFrame(
    alternative_rows
)


# ------------------------------------------------------------
# 11. Build full method comparison
# ------------------------------------------------------------

full_method_comparison_df = pd.concat(
    [
        alternative_performance_df,
        individual_performance_df,
    ],
    ignore_index=True,
    sort=False,
)

test_ranking_df = (
    full_method_comparison_df.loc[
        full_method_comparison_df[
            "split"
        ] == "test"
    ]
    .sort_values(
        [
            "macro_f1",
            "accuracy",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

test_ranking_df.insert(
    0,
    "test_rank",
    np.arange(
        1,
        len(test_ranking_df) + 1,
    ),
)

print("\n" + "=" * 100)
print("LABELED TEST RANKING")
print("=" * 100)

display(test_ranking_df)

best_test_row = test_ranking_df.iloc[0]

print("\nBest tested method:")
print(best_test_row["method"])

print(
    "Best labeled-test Macro-F1:",
    best_test_row["macro_f1"],
)


# ------------------------------------------------------------
# 12. Confirm released setting is the best compared setting
# ------------------------------------------------------------

released_test_rows = test_ranking_df.loc[
    test_ranking_df["method"].isin([
        "released_weights",
        "released_weights_control",
    ])
]

assert not released_test_rows.empty

released_best_test_f1 = float(
    released_test_rows[
        "macro_f1"
    ].max()
)

assert np.isclose(
    released_best_test_f1,
    EXPECTED_R5_TEST_F1,
    rtol=0.0,
    atol=1e-12,
)

if np.isclose(
    float(best_test_row["macro_f1"]),
    released_best_test_f1,
    rtol=0.0,
    atol=1e-12,
):
    print(
        "\nPASS: Released CALMAH weights are "
        "the best labeled-test setting among "
        "all compared individual and ensemble methods."
    )
else:
    print(
        "\nNOTE: another stored method has a higher "
        "labeled-test Macro-F1:"
    )
    print(best_test_row)


# ------------------------------------------------------------
# 13. Detailed released-setting test report
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RELEASED-WEIGHT TEST CLASSIFICATION REPORT")
print("=" * 100)

print(
    classification_report(
        y_test_verify,
        released_outputs["test"][
            "prediction"
        ],
        labels=[0, 1],
        digits=6,
        zero_division=0,
    )
)


# ------------------------------------------------------------
# 14. Save verification reports only in restored directory
# ------------------------------------------------------------

VERIFY_REPORT_ROOT = (
    VERIFY_RESTORED_ROOT
    / "verification"
)

VERIFY_REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

released_performance_df.to_csv(
    VERIFY_REPORT_ROOT
    / "released_weights_train_val_test.csv",
    index=False,
)

full_method_comparison_df.to_csv(
    VERIFY_REPORT_ROOT
    / "all_method_split_comparison.csv",
    index=False,
)

test_ranking_df.to_csv(
    VERIFY_REPORT_ROOT
    / "labeled_test_method_ranking.csv",
    index=False,
)

verification_summary = {
    "method": (
        "retrained_models_released_CALMAH_weights"
    ),
    "train_macro_f1": observed_train_f1,
    "val_macro_f1": observed_val_f1,
    "test_macro_f1": observed_test_f1,
    "best_compared_test_method": str(
        best_test_row["method"]
    ),
    "best_compared_test_macro_f1": float(
        best_test_row["macro_f1"]
    ),
    "released_weights_are_best": bool(
        np.isclose(
            float(
                best_test_row["macro_f1"]
            ),
            released_best_test_f1,
            rtol=0.0,
            atol=1e-12,
        )
    ),
    "weights_modified": False,
    "thresholds_modified": False,
    "models_modified": False,
}

with (
    VERIFY_REPORT_ROOT
    / "verification_summary.json"
).open("w", encoding="utf-8") as handle:
    json.dump(
        verification_summary,
        handle,
        indent=2,
    )


# ------------------------------------------------------------
# 15. Confirm no source files changed
# ------------------------------------------------------------

VERIFY_HASHES_AFTER = {
    name: verify_sha256(path)
    for name, path in VERIFY_SOURCE_PATHS.items()
}

assert (
    VERIFY_HASHES_AFTER
    == VERIFY_HASHES_BEFORE
), (
    "At least one source experiment file "
    "was unexpectedly modified."
)

print("\n" + "=" * 100)
print("FINAL VERIFICATION SUMMARY")
print("=" * 100)

print(
    f"Train Macro-F1: {observed_train_f1:.12f}"
)

print(
    f"Val Macro-F1:   {observed_val_f1:.12f}"
)

print(
    f"Test Macro-F1:  {observed_test_f1:.12f}"
)

print(
    "\nBest labeled-test method:",
    verification_summary[
        "best_compared_test_method"
    ],
)

print(
    "Best labeled-test Macro-F1:",
    verification_summary[
        "best_compared_test_macro_f1"
    ],
)

print(
    "\nPASS: released weights and retrained "
    "thresholds were re-evaluated from disk."
)

print(
    "PASS: R4/R5 performance was exactly reproduced."
)

print(
    "PASS: no model, threshold, weight, probability "
    "cache, or prior experiment file was modified."
)

print("\nReports saved under:")
print(VERIFY_REPORT_ROOT)

# 13. Expert agents and role-aware four-source decision

This section adds two independent expert adapters and a final decision agent.

The four prediction sources have intentionally different roles:

1. **CALM-AH best prediction** — the primary calibrated multimodal classifier. Its confidence can activate a veto.
2. **Released-weight anchor** — the historical stable prediction generated with the previous released weights.
3. **AffectGPT expert** — an external multimodal affect expert using video, audio, and subtitle inputs.
4. **GPT-5.6 Sol semantic expert** — an independent semantic reasoner using sampled video frames and transcript text.

The two external models implement the same `PredictionExpert` contract and save resumable, auditable prediction files. The final `RoleAwareDecisionAgent` is not an equal-vote ensemble: the primary classifier has the largest base weight and a configurable high-confidence veto. A unanimous, sufficiently confident secondary coalition may override it only when the veto is inactive.

> External inference and final export are disabled by default. Configure the paths below, run the two expert cells, inspect their audit files, and then enable the final fusion cell.


In [ ]:
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Optional
import base64
import json
import os
import re
import shlex
import subprocess
import time

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Expert-layer paths and execution switches
# ---------------------------------------------------------------------
EXPERT_OUTPUT_ROOT = Path(
    os.getenv(
        "EXPERT_OUTPUT_ROOT",
        str(OUTPUT_ROOT / "expert_layer"),
    )
).expanduser().resolve()
EXPERT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPERT_MANIFEST_PATH = (
    EXPERT_OUTPUT_ROOT / "private_expert_manifest.csv"
)
AFFECTGPT_OUTPUT_PATH = (
    EXPERT_OUTPUT_ROOT / "affectgpt_private_predictions.csv"
)
GPT56_OUTPUT_PATH = (
    EXPERT_OUTPUT_ROOT / "gpt56_sol_private_predictions.csv"
)
FOUR_SOURCE_AUDIT_PATH = (
    EXPERT_OUTPUT_ROOT / "four_source_decision_audit.csv"
)
FOUR_SOURCE_SUBMISSION_PATH = (
    EXPERT_OUTPUT_ROOT / "trial-four-source-role-aware.txt"
)

RUN_AFFECTGPT_EXPERT = (
    os.getenv("RUN_AFFECTGPT_EXPERT", "0") == "1"
)
RUN_GPT56_EXPERT = (
    os.getenv("RUN_GPT56_EXPERT", "0") == "1"
)
RUN_FINAL_FUSION = (
    os.getenv("RUN_FINAL_FUSION", "0") == "1"
)
FORCE_RERUN_EXPERTS = (
    os.getenv("FORCE_RERUN_EXPERTS", "0") == "1"
)


# ---------------------------------------------------------------------
# Task definition shared by both experts
# ---------------------------------------------------------------------
POSITIVE_CLASS_NAME = os.getenv(
    "POSITIVE_CLASS_NAME",
    "A-H / hesitation or ambivalence is present",
)
NEGATIVE_CLASS_NAME = os.getenv(
    "NEGATIVE_CLASS_NAME",
    "No A-H / hesitation or ambivalence is present",
)

TASK_DEFINITION = os.getenv(
    "TASK_DEFINITION",
    (
        "Binary A-H classification. Predict class 1 when the clip "
        "contains meaningful evidence of hesitation, uncertainty, "
        "ambivalence, reluctance, indecision, or conflict-related "
        "affective behaviour. Predict class 0 when this evidence is "
        "absent or the behaviour is clearly decisive/confident. "
        "Use only observable multimodal evidence."
    ),
)


# ---------------------------------------------------------------------
# AffectGPT external runner
# ---------------------------------------------------------------------
AFFECTGPT_ROOT = Path(
    os.getenv(
        "AFFECTGPT_ROOT",
        "/workspace/AffectGPT",
    )
).expanduser()

AFFECTGPT_PYTHON = Path(
    os.getenv(
        "AFFECTGPT_PYTHON",
        "/opt/conda/envs/affectgpt/bin/python",
    )
).expanduser()

AFFECTGPT_RUNNER = Path(
    os.getenv(
        "AFFECTGPT_RUNNER",
        str(AFFECTGPT_ROOT / "inference_expert.py"),
    )
).expanduser()

AFFECTGPT_AUDIO_ROOT = Path(
    os.getenv(
        "AFFECTGPT_AUDIO_ROOT",
        str(PROJECT_ROOT / "audios"),
    )
).expanduser()

# The external runner must accept an input manifest and write one row per
# video. Required output fields are documented in AffectGPTExpert below.
#
# Replace this template if the local AffectGPT script uses different flags.
AFFECTGPT_COMMAND_TEMPLATE = os.getenv(
    "AFFECTGPT_COMMAND_TEMPLATE",
    (
        "{python} {runner} "
        "--manifest {manifest} "
        "--output {output}"
    ),
)


# ---------------------------------------------------------------------
# Subtitle/transcript source
# ---------------------------------------------------------------------
TRANSCRIPT_TABLE_PATH = Path(
    os.getenv(
        "TRANSCRIPT_TABLE_PATH",
        str(PROJECT_ROOT / "sub.xlsx"),
    )
).expanduser()


# ---------------------------------------------------------------------
# GPT-5.6 Sol API configuration
# ---------------------------------------------------------------------
GPT56_MODEL = os.getenv(
    "GPT56_MODEL",
    "gpt-5.6-sol",
)
GPT56_REASONING_EFFORT = os.getenv(
    "GPT56_REASONING_EFFORT",
    "medium",
)
GPT56_FRAME_COUNT = int(
    os.getenv("GPT56_FRAME_COUNT", "6")
)
GPT56_MAX_OUTPUT_TOKENS = int(
    os.getenv("GPT56_MAX_OUTPUT_TOKENS", "400")
)
GPT56_CACHE_PATH = (
    EXPERT_OUTPUT_ROOT / "gpt56_sol_cache.jsonl"
)
GPT56_MAX_RETRIES = int(
    os.getenv("GPT56_MAX_RETRIES", "3")
)
GPT56_RETRY_SECONDS = float(
    os.getenv("GPT56_RETRY_SECONDS", "3")
)


# ---------------------------------------------------------------------
# Role-aware fusion configuration
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class FusionPolicy:
    primary_source: str = "calm_ah"
    source_weights: Mapping[str, float] = field(
        default_factory=lambda: {
            "calm_ah": 0.55,
            "anchor": 0.15,
            "affectgpt": 0.15,
            "gpt56_sol": 0.15,
        }
    )
    final_threshold: float = 0.50

    # High-confidence CALM-AH predictions cannot be overturned.
    primary_veto_confidence: float = 0.85

    # If the veto is inactive, every secondary source must agree against
    # CALM-AH and satisfy this confidence threshold for an explicit override.
    secondary_override_min_confidence: float = 0.65
    secondary_override_required: int = 3

    # When an explicit override does not apply, use confidence-adjusted
    # weighted voting. Exact ties return the CALM-AH label.
    require_all_four_sources: bool = True


FUSION_POLICY = FusionPolicy()

print("Expert output root:", EXPERT_OUTPUT_ROOT)
print("AffectGPT output:", AFFECTGPT_OUTPUT_PATH)
print("GPT-5.6 Sol output:", GPT56_OUTPUT_PATH)
print("GPT-5.6 model:", GPT56_MODEL)
print("Fusion policy:", FUSION_POLICY)

In [ ]:
# ---------------------------------------------------------------------
# Standard expert result contract
# ---------------------------------------------------------------------
STANDARD_EXPERT_COLUMNS = [
    "submission_id",
    "video_id",
    "label",
    "probability_class_1",
    "confidence",
    "rationale",
    "status",
    "expert",
]


def normalize_item_id(value: Any) -> str:
    value = str(value).strip().strip('"').strip("'")
    value = value.replace("\\", "/")
    return Path(value).name.lower()


def normalize_stem_id(value: Any) -> str:
    return Path(normalize_item_id(value)).stem.lower()


def coerce_binary_label(value: Any) -> int:
    if isinstance(value, (bool, np.bool_)):
        return int(value)

    if isinstance(value, (int, np.integer)):
        if int(value) in {0, 1}:
            return int(value)

    if isinstance(value, (float, np.floating)):
        if float(value) in {0.0, 1.0}:
            return int(value)

    text = str(value).strip().lower()

    if text in {"0", "0.0", "false", "negative", "no"}:
        return 0
    if text in {"1", "1.0", "true", "positive", "yes"}:
        return 1

    match = re.search(
        r"(?:label|prediction|class)\s*[:=]?\s*([01])\b",
        text,
        flags=re.IGNORECASE,
    )
    if match:
        return int(match.group(1))

    raise ValueError(f"Cannot parse binary label from: {value!r}")


def clip_probability(value: Any) -> float:
    probability = float(value)
    if not np.isfinite(probability):
        raise ValueError(f"Non-finite probability: {value!r}")
    return float(np.clip(probability, 0.0, 1.0))


def probability_to_confidence(probability: Any) -> float:
    probability = clip_probability(probability)
    return float(abs(probability - 0.5) * 2.0)


def infer_binary_from_text(text: str) -> tuple[int, float]:
    """
    Conservative fallback for AffectGPT outputs that contain prose rather
    than explicit label/probability columns.

    Explicit label patterns are preferred. Keyword inference is used only
    when one cue family clearly dominates the other.
    """
    text = str(text).strip()
    lowered = text.lower()

    explicit = re.search(
        r"(?:label|prediction|class)\s*[:=]?\s*([01])\b",
        lowered,
    )
    if explicit:
        label = int(explicit.group(1))
        return label, 0.80 if label == 1 else 0.20

    positive_terms = [
        "hesitat",
        "ambivalen",
        "uncertain",
        "uncertainty",
        "indecis",
        "reluctan",
        "conflicted",
        "doubt",
        "unsure",
    ]
    negative_terms = [
        "no hesitation",
        "not hesitant",
        "confident",
        "decisive",
        "certain",
        "without doubt",
    ]

    positive_hits = sum(term in lowered for term in positive_terms)
    negative_hits = sum(term in lowered for term in negative_terms)

    if positive_hits > negative_hits and positive_hits > 0:
        return 1, min(0.90, 0.60 + 0.07 * positive_hits)

    if negative_hits > positive_hits and negative_hits > 0:
        return 0, max(0.10, 0.40 - 0.07 * negative_hits)

    raise ValueError(
        "AffectGPT prose does not contain an unambiguous binary decision. "
        "Modify the external runner to emit label and probability columns."
    )


def validate_expert_frame(
    frame: pd.DataFrame,
    expected_manifest: pd.DataFrame,
    expert_name: str,
) -> pd.DataFrame:
    frame = frame.copy()

    missing = set(STANDARD_EXPERT_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(
            f"{expert_name} output is missing columns: {sorted(missing)}"
        )

    frame["submission_id"] = frame["submission_id"].astype(str)
    frame["video_id"] = frame["video_id"].astype(str)
    frame["label"] = frame["label"].map(coerce_binary_label).astype(int)
    frame["probability_class_1"] = (
        frame["probability_class_1"].map(clip_probability)
    )
    frame["confidence"] = frame["confidence"].map(clip_probability)

    if frame["submission_id"].duplicated().any():
        duplicates = frame.loc[
            frame["submission_id"].duplicated(keep=False),
            "submission_id",
        ].tolist()
        raise ValueError(
            f"{expert_name} has duplicate submission IDs: {duplicates[:10]}"
        )

    expected_ids = expected_manifest["submission_id"].astype(str).tolist()
    actual_ids = set(frame["submission_id"])
    missing_ids = [item for item in expected_ids if item not in actual_ids]
    extra_ids = sorted(actual_ids - set(expected_ids))

    if missing_ids or extra_ids:
        raise ValueError(
            f"{expert_name} ID mismatch. "
            f"Missing={missing_ids[:10]}, extra={extra_ids[:10]}"
        )

    aligned = (
        expected_manifest[
            ["submission_id", "video_id"]
        ]
        .astype(str)
        .merge(
            frame.drop(columns=["video_id"]),
            on="submission_id",
            how="left",
            validate="one_to_one",
        )
    )

    if aligned["status"].ne("ok").any():
        failures = aligned.loc[
            aligned["status"].ne("ok"),
            ["submission_id", "status", "rationale"],
        ]
        raise RuntimeError(
            f"{expert_name} contains failed predictions:\n"
            f"{failures.head(20).to_string(index=False)}"
        )

    return aligned[STANDARD_EXPERT_COLUMNS]


# ---------------------------------------------------------------------
# Resolve the private manifest produced earlier in this notebook
# ---------------------------------------------------------------------
def resolve_private_expert_manifest() -> pd.DataFrame:
    if "private_rows_ordered" in globals():
        manifest = private_rows_ordered.copy()
    elif "private_df" in globals():
        manifest = (
            private_df
            .sort_values("private_order")
            .reset_index(drop=True)
            .copy()
        )
    else:
        candidates = [
            OUTPUT_ROOT / "current_private_manifest.csv",
            PROJECT_ROOT / "abaw11_append_results"
            / "current_private_manifest.csv",
        ]
        candidates.extend(
            OUTPUT_ROOT.rglob("current_private_manifest.csv")
        )
        candidates = [
            Path(path)
            for path in candidates
            if Path(path).exists()
        ]
        if not candidates:
            raise FileNotFoundError(
                "No current_private_manifest.csv was found. "
                "Run the current-dataset reconstruction section first."
            )
        manifest = (
            pd.read_csv(max(
                candidates,
                key=lambda path: path.stat().st_mtime,
            ))
            .sort_values("private_order")
            .reset_index(drop=True)
        )

    required = {
        "private_order",
        "submission_id",
        "video_id",
        "video_path",
    }
    missing = required - set(manifest.columns)
    if missing:
        raise ValueError(
            f"Private manifest is missing columns: {sorted(missing)}"
        )

    manifest = (
        manifest
        .sort_values("private_order")
        .reset_index(drop=True)
        .copy()
    )
    manifest["submission_id"] = manifest["submission_id"].astype(str)
    manifest["video_id"] = manifest["video_id"].astype(str)
    manifest["video_path"] = manifest["video_path"].astype(str)

    if manifest["submission_id"].duplicated().any():
        raise ValueError("Duplicate submission IDs in private manifest.")

    if len(manifest) != 152:
        raise ValueError(
            f"Expected 152 private videos, found {len(manifest)}."
        )

    return manifest


def load_transcript_lookup(
    table_path: Path,
) -> dict[str, str]:
    table_path = Path(table_path)

    if not table_path.exists():
        print(
            "WARNING: transcript table not found; GPT-5.6 Sol will use "
            "sampled frames only:",
            table_path,
        )
        return {}

    suffix = table_path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        table = pd.read_excel(table_path)
    elif suffix == ".csv":
        table = pd.read_csv(table_path)
    else:
        raise ValueError(
            f"Unsupported transcript table: {table_path}"
        )

    if table.empty:
        return {}

    lower_to_original = {
        str(column).strip().lower(): column
        for column in table.columns
    }

    id_column = next(
        (
            lower_to_original[name]
            for name in [
                "submission_id",
                "video_id",
                "video",
                "filename",
                "file",
                "name",
            ]
            if name in lower_to_original
        ),
        table.columns[0],
    )

    text_column = next(
        (
            lower_to_original[name]
            for name in [
                "subtitle",
                "subtitles",
                "transcript",
                "text",
                "utterance",
                "caption",
            ]
            if name in lower_to_original
        ),
        table.columns[1] if len(table.columns) > 1 else None,
    )

    if text_column is None:
        raise ValueError(
            "Transcript table must contain at least two columns."
        )

    lookup: dict[str, str] = {}

    for _, row in table.iterrows():
        raw_id = row[id_column]
        raw_text = row[text_column]

        if pd.isna(raw_id):
            continue

        text = "" if pd.isna(raw_text) else str(raw_text).strip()
        lookup[normalize_item_id(raw_id)] = text
        lookup[normalize_stem_id(raw_id)] = text

    return lookup


def build_expert_manifest() -> pd.DataFrame:
    manifest = resolve_private_expert_manifest()
    transcript_lookup = load_transcript_lookup(
        TRANSCRIPT_TABLE_PATH
    )

    manifest["audio_path"] = manifest["video_path"].map(
        lambda value: str(
            AFFECTGPT_AUDIO_ROOT
            / f"{Path(value).stem}.wav"
        )
    )
    manifest["subtitle"] = manifest.apply(
        lambda row: transcript_lookup.get(
            normalize_item_id(row["submission_id"]),
            transcript_lookup.get(
                normalize_stem_id(row["submission_id"]),
                transcript_lookup.get(
                    normalize_item_id(row["video_id"]),
                    transcript_lookup.get(
                        normalize_stem_id(row["video_id"]),
                        "",
                    ),
                ),
            ),
        ),
        axis=1,
    )

    manifest.to_csv(
        EXPERT_MANIFEST_PATH,
        index=False,
    )
    return manifest


class PredictionExpert(ABC):
    """Common contract for independent binary-classification experts."""

    name: str

    @abstractmethod
    def predict_manifest(
        self,
        manifest: pd.DataFrame,
        force: bool = False,
    ) -> pd.DataFrame:
        raise NotImplementedError


expert_manifest = build_expert_manifest()

print("Expert manifest:", EXPERT_MANIFEST_PATH)
print("Rows:", len(expert_manifest))
display(
    expert_manifest[
        [
            "private_order",
            "submission_id",
            "video_path",
            "audio_path",
            "subtitle",
        ]
    ].head(5)
)

In [ ]:
class AffectGPTExpert(PredictionExpert):
    """
    Adapter for an external AffectGPT inference process.

    The external command receives:
      --manifest <CSV>
      --output <CSV>

    Its output CSV should contain one row per video and preferably:
      submission_id
      label                    # 0 or 1
      probability_class_1      # probability of class 1
      confidence               # optional, [0, 1]
      rationale                # optional
      response                 # optional raw AffectGPT text

    Flexible aliases such as video_id/name, prediction/class, probability/prob,
    and reason/explanation are accepted. When only response text is available,
    a conservative text parser is used; explicit numeric output is strongly
    recommended.
    """

    name = "affectgpt"

    def __init__(
        self,
        command_template: str,
        python_path: Path,
        runner_path: Path,
        output_path: Path,
        timeout_seconds: Optional[int] = None,
    ) -> None:
        self.command_template = str(command_template)
        self.python_path = Path(python_path)
        self.runner_path = Path(runner_path)
        self.output_path = Path(output_path)
        self.timeout_seconds = timeout_seconds

    def _run_external(
        self,
        manifest_path: Path,
    ) -> None:
        if not self.python_path.exists():
            raise FileNotFoundError(
                f"AffectGPT Python not found: {self.python_path}"
            )
        if not self.runner_path.exists():
            raise FileNotFoundError(
                f"AffectGPT runner not found: {self.runner_path}"
            )

        replacements = {
            "python": shlex.quote(str(self.python_path)),
            "runner": shlex.quote(str(self.runner_path)),
            "manifest": shlex.quote(str(manifest_path)),
            "output": shlex.quote(str(self.output_path)),
        }
        command_text = self.command_template.format(**replacements)
        command = shlex.split(command_text)

        self.output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        print("Running AffectGPT expert:")
        print(" ".join(shlex.quote(token) for token in command))

        subprocess.run(
            command,
            check=True,
            timeout=self.timeout_seconds,
            cwd=str(self.runner_path.parent),
        )

        if not self.output_path.exists():
            raise FileNotFoundError(
                "AffectGPT command finished but did not create: "
                f"{self.output_path}"
            )

    @staticmethod
    def _find_column(
        frame: pd.DataFrame,
        aliases: Iterable[str],
    ) -> Optional[str]:
        lookup = {
            str(column).strip().lower(): column
            for column in frame.columns
        }
        for alias in aliases:
            if alias in lookup:
                return lookup[alias]
        return None

    def _standardize_output(
        self,
        raw: pd.DataFrame,
        manifest: pd.DataFrame,
    ) -> pd.DataFrame:
        id_column = self._find_column(
            raw,
            [
                "submission_id",
                "video_id",
                "video",
                "filename",
                "file",
                "name",
            ],
        )
        label_column = self._find_column(
            raw,
            [
                "label",
                "prediction",
                "pred",
                "class",
                "binary_prediction",
            ],
        )
        probability_column = self._find_column(
            raw,
            [
                "probability_class_1",
                "probability",
                "prob",
                "score",
                "p1",
            ],
        )
        confidence_column = self._find_column(
            raw,
            ["confidence", "certainty"],
        )
        rationale_column = self._find_column(
            raw,
            [
                "rationale",
                "reason",
                "explanation",
                "analysis",
            ],
        )
        response_column = self._find_column(
            raw,
            [
                "response",
                "output",
                "answer",
                "generated_text",
                "caption",
            ],
        )

        if id_column is None:
            if len(raw) != len(manifest):
                raise ValueError(
                    "AffectGPT output has no ID column and row count "
                    "does not match the manifest."
                )
            raw = raw.copy()
            raw["__submission_id"] = (
                manifest["submission_id"].astype(str).to_numpy()
            )
            id_column = "__submission_id"

        records = []

        manifest_by_normalized_id = {}
        for _, row in manifest.iterrows():
            for value in [
                row["submission_id"],
                row["video_id"],
                row["video_path"],
            ]:
                manifest_by_normalized_id[
                    normalize_item_id(value)
                ] = str(row["submission_id"])
                manifest_by_normalized_id[
                    normalize_stem_id(value)
                ] = str(row["submission_id"])

        for _, row in raw.iterrows():
            raw_id = row[id_column]
            normalized_candidates = [
                normalize_item_id(raw_id),
                normalize_stem_id(raw_id),
            ]
            submission_id = next(
                (
                    manifest_by_normalized_id[key]
                    for key in normalized_candidates
                    if key in manifest_by_normalized_id
                ),
                str(raw_id),
            )

            response_text = (
                ""
                if response_column is None
                or pd.isna(row[response_column])
                else str(row[response_column]).strip()
            )

            if label_column is not None and not pd.isna(row[label_column]):
                label = coerce_binary_label(row[label_column])
                if (
                    probability_column is not None
                    and not pd.isna(row[probability_column])
                ):
                    probability = clip_probability(
                        row[probability_column]
                    )
                else:
                    probability = 0.75 if label == 1 else 0.25
            else:
                if not response_text:
                    raise ValueError(
                        f"No label or textual response for {raw_id!r}."
                    )
                label, probability = infer_binary_from_text(
                    response_text
                )

            if confidence_column is not None and not pd.isna(
                row[confidence_column]
            ):
                confidence = clip_probability(
                    row[confidence_column]
                )
            else:
                confidence = probability_to_confidence(
                    probability
                )

            rationale = (
                response_text
                if rationale_column is None
                or pd.isna(row[rationale_column])
                else str(row[rationale_column]).strip()
            )

            records.append({
                "submission_id": submission_id,
                "video_id": submission_id,
                "label": label,
                "probability_class_1": probability,
                "confidence": confidence,
                "rationale": rationale,
                "status": "ok",
                "expert": self.name,
            })

        return validate_expert_frame(
            pd.DataFrame(records),
            manifest,
            self.name,
        )

    def predict_manifest(
        self,
        manifest: pd.DataFrame,
        force: bool = False,
    ) -> pd.DataFrame:
        if force or not self.output_path.exists():
            manifest.to_csv(
                EXPERT_MANIFEST_PATH,
                index=False,
            )
            self._run_external(
                EXPERT_MANIFEST_PATH
            )

        raw = pd.read_csv(self.output_path)
        standardized = self._standardize_output(
            raw,
            manifest,
        )
        standardized.to_csv(
            self.output_path,
            index=False,
        )
        return standardized


affectgpt_expert = AffectGPTExpert(
    command_template=AFFECTGPT_COMMAND_TEMPLATE,
    python_path=AFFECTGPT_PYTHON,
    runner_path=AFFECTGPT_RUNNER,
    output_path=AFFECTGPT_OUTPUT_PATH,
)

if RUN_AFFECTGPT_EXPERT:
    affectgpt_predictions = (
        affectgpt_expert.predict_manifest(
            expert_manifest,
            force=FORCE_RERUN_EXPERTS,
        )
    )
    display(affectgpt_predictions.head())
    print(
        "AffectGPT counts:",
        affectgpt_predictions["label"]
        .value_counts()
        .sort_index()
        .to_dict(),
    )
else:
    print(
        "AffectGPT expert is configured but not executed. "
        "Set RUN_AFFECTGPT_EXPERT=1 after adapting "
        "AFFECTGPT_COMMAND_TEMPLATE to the local runner."
    )

In [ ]:
class GPT56SolSemanticExpert(PredictionExpert):
    """
    Independent semantic expert using GPT-5.6 Sol.

    GPT-5.6 Sol accepts text and image inputs, not raw video/audio. This
    adapter therefore samples frames uniformly from each video and adds the
    transcript/subtitle text. It does not receive CALM-AH, anchor, or
    AffectGPT predictions, preserving expert independence.
    """

    name = "gpt56_sol"

    def __init__(
        self,
        model: str,
        output_path: Path,
        cache_path: Path,
        frame_count: int = 6,
        reasoning_effort: str = "medium",
        max_output_tokens: int = 400,
        max_retries: int = 3,
        retry_seconds: float = 3.0,
    ) -> None:
        self.model = str(model)
        self.output_path = Path(output_path)
        self.cache_path = Path(cache_path)
        self.frame_count = int(frame_count)
        self.reasoning_effort = str(reasoning_effort)
        self.max_output_tokens = int(max_output_tokens)
        self.max_retries = int(max_retries)
        self.retry_seconds = float(retry_seconds)
        self._client = None

    @property
    def client(self):
        if self._client is None:
            try:
                from openai import OpenAI
            except ImportError as exc:
                raise ImportError(
                    "Install the OpenAI SDK first: pip install -U openai"
                ) from exc

            if not os.getenv("OPENAI_API_KEY"):
                raise EnvironmentError(
                    "OPENAI_API_KEY is not set."
                )

            self._client = OpenAI()

        return self._client

    @staticmethod
    def _extract_json(text: str) -> dict[str, Any]:
        text = str(text).strip()
        text = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            text,
            flags=re.IGNORECASE | re.DOTALL,
        )

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            match = re.search(
                r"\{.*\}",
                text,
                flags=re.DOTALL,
            )
            if not match:
                raise
            return json.loads(match.group(0))

    def _sample_frame_data_urls(
        self,
        video_path: Path,
    ) -> list[str]:
        try:
            import cv2
        except ImportError as exc:
            raise ImportError(
                "OpenCV is required to sample video frames: "
                "pip install opencv-python"
            ) from exc

        video_path = Path(video_path)
        capture = cv2.VideoCapture(str(video_path))

        if not capture.isOpened():
            raise RuntimeError(
                f"Cannot open video: {video_path}"
            )

        frame_total = int(
            capture.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        if frame_total <= 0:
            capture.release()
            raise RuntimeError(
                f"Video has no readable frames: {video_path}"
            )

        positions = np.linspace(
            0,
            frame_total - 1,
            num=min(self.frame_count, frame_total),
            dtype=int,
        )

        data_urls = []

        for position in positions:
            capture.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(position),
            )
            success, frame = capture.read()

            if not success:
                continue

            success, encoded = cv2.imencode(
                ".jpg",
                frame,
                [int(cv2.IMWRITE_JPEG_QUALITY), 85],
            )
            if not success:
                continue

            payload = base64.b64encode(
                encoded.tobytes()
            ).decode("ascii")
            data_urls.append(
                f"data:image/jpeg;base64,{payload}"
            )

        capture.release()

        if not data_urls:
            raise RuntimeError(
                f"No frames could be sampled from: {video_path}"
            )

        return data_urls

    def _load_cache(self) -> dict[str, dict[str, Any]]:
        if not self.cache_path.exists():
            return {}

        cache = {}

        with self.cache_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            for line in handle:
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                cache[str(record["submission_id"])] = record

        return cache

    def _append_cache(
        self,
        record: Mapping[str, Any],
    ) -> None:
        self.cache_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        with self.cache_path.open(
            "a",
            encoding="utf-8",
        ) as handle:
            handle.write(
                json.dumps(
                    dict(record),
                    ensure_ascii=False,
                )
                + "\n"
            )

    def _build_request_content(
        self,
        row: pd.Series,
    ) -> list[dict[str, Any]]:
        transcript = str(
            row.get("subtitle", "")
        ).strip()

        prompt = f"""
Task:
{TASK_DEFINITION}

Class definitions:
- 1: {POSITIVE_CLASS_NAME}
- 0: {NEGATIVE_CLASS_NAME}

Evidence:
- Video ID: {row['submission_id']}
- Transcript/subtitle:
{transcript if transcript else '[No transcript available]'}

Inspect the sampled frames in temporal order. Consider uncertainty,
self-correction, pauses implied by context, conflicting affect, reluctance,
and whether the visible behaviour supports hesitation/ambivalence. Do not
infer private traits or use information outside this clip.

Return JSON only:
{{
  "label": 0 or 1,
  "probability_class_1": number from 0 to 1,
  "confidence": number from 0 to 1,
  "rationale": "brief observable evidence",
  "evidence": ["brief cue 1", "brief cue 2"]
}}
""".strip()

        content: list[dict[str, Any]] = [
            {
                "type": "input_text",
                "text": prompt,
            }
        ]

        for frame_url in self._sample_frame_data_urls(
            Path(row["video_path"])
        ):
            content.append({
                "type": "input_image",
                "image_url": frame_url,
                "detail": "auto",
            })

        return content

    def _predict_one(
        self,
        row: pd.Series,
    ) -> dict[str, Any]:
        last_error: Optional[Exception] = None

        for attempt in range(
            1,
            self.max_retries + 1,
        ):
            try:
                response = self.client.responses.create(
                    model=self.model,
                    reasoning={
                        "effort": self.reasoning_effort,
                    },
                    input=[
                        {
                            "role": "system",
                            "content": [
                                {
                                    "type": "input_text",
                                    "text": (
                                        "You are an independent, conservative "
                                        "binary semantic classification expert. "
                                        "Return valid JSON only."
                                    ),
                                }
                            ],
                        },
                        {
                            "role": "user",
                            "content": self._build_request_content(
                                row
                            ),
                        },
                    ],
                    max_output_tokens=self.max_output_tokens,
                )

                parsed = self._extract_json(
                    response.output_text
                )

                label = coerce_binary_label(
                    parsed["label"]
                )
                probability = clip_probability(
                    parsed["probability_class_1"]
                )
                confidence = clip_probability(
                    parsed.get(
                        "confidence",
                        probability_to_confidence(
                            probability
                        ),
                    )
                )

                return {
                    "submission_id": str(
                        row["submission_id"]
                    ),
                    "video_id": str(row["video_id"]),
                    "label": label,
                    "probability_class_1": probability,
                    "confidence": confidence,
                    "rationale": str(
                        parsed.get("rationale", "")
                    ).strip(),
                    "evidence": parsed.get(
                        "evidence",
                        [],
                    ),
                    "status": "ok",
                    "expert": self.name,
                    "model": self.model,
                }

            except Exception as exc:
                last_error = exc
                if attempt < self.max_retries:
                    time.sleep(
                        self.retry_seconds * attempt
                    )

        return {
            "submission_id": str(
                row["submission_id"]
            ),
            "video_id": str(row["video_id"]),
            "label": 0,
            "probability_class_1": 0.5,
            "confidence": 0.0,
            "rationale": (
                f"ERROR after {self.max_retries} attempts: "
                f"{type(last_error).__name__}: {last_error}"
            ),
            "evidence": [],
            "status": "error",
            "expert": self.name,
            "model": self.model,
        }

    def predict_manifest(
        self,
        manifest: pd.DataFrame,
        force: bool = False,
    ) -> pd.DataFrame:
        if force:
            if self.cache_path.exists():
                self.cache_path.unlink()
            if self.output_path.exists():
                self.output_path.unlink()

        cache = self._load_cache()
        records = []

        for index, row in manifest.iterrows():
            submission_id = str(
                row["submission_id"]
            )

            cached = cache.get(submission_id)

            if (
                cached is not None
                and cached.get("status") == "ok"
            ):
                record = cached
            else:
                record = self._predict_one(row)
                self._append_cache(record)

            records.append(record)

            if (index + 1) % 10 == 0 or index == 0:
                print(
                    f"GPT-5.6 Sol: {index + 1}/{len(manifest)}"
                )

        frame = pd.DataFrame(records)
        standardized = validate_expert_frame(
            frame,
            manifest,
            self.name,
        )
        standardized.to_csv(
            self.output_path,
            index=False,
        )
        return standardized


gpt56_semantic_expert = GPT56SolSemanticExpert(
    model=GPT56_MODEL,
    output_path=GPT56_OUTPUT_PATH,
    cache_path=GPT56_CACHE_PATH,
    frame_count=GPT56_FRAME_COUNT,
    reasoning_effort=GPT56_REASONING_EFFORT,
    max_output_tokens=GPT56_MAX_OUTPUT_TOKENS,
    max_retries=GPT56_MAX_RETRIES,
    retry_seconds=GPT56_RETRY_SECONDS,
)

if RUN_GPT56_EXPERT:
    gpt56_predictions = (
        gpt56_semantic_expert.predict_manifest(
            expert_manifest,
            force=FORCE_RERUN_EXPERTS,
        )
    )
    display(gpt56_predictions.head())
    print(
        "GPT-5.6 Sol counts:",
        gpt56_predictions["label"]
        .value_counts()
        .sort_index()
        .to_dict(),
    )
else:
    print(
        "GPT-5.6 Sol expert is configured but not executed. "
        "Set OPENAI_API_KEY and RUN_GPT56_EXPERT=1."
    )

In [ ]:
def _source_frame_from_arrays(
    manifest: pd.DataFrame,
    labels: Any,
    probabilities: Optional[Any],
    source_name: str,
) -> pd.DataFrame:
    labels = np.asarray(labels).reshape(-1)

    if len(labels) != len(manifest):
        raise ValueError(
            f"{source_name}: expected {len(manifest)} labels, "
            f"found {len(labels)}."
        )

    labels = np.asarray(
        [coerce_binary_label(value) for value in labels],
        dtype=int,
    )

    if probabilities is None:
        probabilities = np.where(
            labels == 1,
            0.75,
            0.25,
        )
    else:
        probabilities = np.asarray(
            probabilities,
            dtype=float,
        ).reshape(-1)

    if len(probabilities) != len(manifest):
        raise ValueError(
            f"{source_name}: probability length mismatch."
        )

    probabilities = np.clip(
        probabilities,
        0.0,
        1.0,
    )

    return pd.DataFrame({
        "submission_id": (
            manifest["submission_id"].astype(str)
        ),
        "label": labels,
        "probability_class_1": probabilities,
        "confidence": np.abs(
            probabilities - 0.5
        ) * 2.0,
        "source": source_name,
    })


def _parse_submission_file(
    path: Path,
    manifest: pd.DataFrame,
    source_name: str,
) -> pd.DataFrame:
    path = Path(path)
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue

            match = re.search(
                r"(.+?\.(?:mp4|avi|mov|mkv|webm|m4v))"
                r"[\s,;\t]+([01](?:\.0)?)\s*$",
                line,
                flags=re.IGNORECASE,
            )
            if not match:
                continue

            rows.append({
                "raw_id": match.group(1),
                "label": int(float(match.group(2))),
            })

    if not rows:
        raise ValueError(
            f"No predictions parsed from {path}"
        )

    parsed = pd.DataFrame(rows)
    parsed["normalized_id"] = parsed["raw_id"].map(
        normalize_item_id
    )

    manifest_lookup = {}
    for _, row in manifest.iterrows():
        for key in [
            row["submission_id"],
            row["video_id"],
            row["video_path"],
        ]:
            manifest_lookup[
                normalize_item_id(key)
            ] = str(row["submission_id"])

    parsed["submission_id"] = parsed["normalized_id"].map(
        manifest_lookup
    )

    if parsed["submission_id"].isna().any():
        bad = parsed.loc[
            parsed["submission_id"].isna(),
            "raw_id",
        ].tolist()
        raise ValueError(
            f"Unmatched IDs in {path}: {bad[:10]}"
        )

    aligned = (
        manifest[["submission_id"]]
        .astype(str)
        .merge(
            parsed[["submission_id", "label"]],
            on="submission_id",
            how="left",
            validate="one_to_one",
        )
    )

    if aligned["label"].isna().any():
        raise ValueError(
            f"{source_name} submission file is incomplete."
        )

    return _source_frame_from_arrays(
        manifest,
        aligned["label"].to_numpy(),
        probabilities=None,
        source_name=source_name,
    )


def _find_newest_file(
    filename: str,
) -> Optional[Path]:
    search_roots = [
        EXPERT_OUTPUT_ROOT,
        OUTPUT_ROOT,
        PROJECT_ROOT / "abaw11_append_results",
    ]
    matches = []

    for root in search_roots:
        if root.exists():
            matches.extend(
                path
                for path in root.rglob(filename)
                if path.is_file()
            )

    if not matches:
        return None

    return max(
        matches,
        key=lambda path: path.stat().st_mtime,
    )


def resolve_calm_ah_source(
    manifest: pd.DataFrame,
) -> pd.DataFrame:
    # The multi-seed result is the preferred "best" source in the current
    # notebook. The locked retrained ensemble is the deterministic fallback.
    candidates = [
        (
            "MULTI_PRIVATE_PRED",
            "MULTI_PRIVATE_SCORE",
        ),
        (
            "private_prediction",
            "private_score",
        ),
    ]

    for label_name, score_name in candidates:
        if label_name in globals():
            return _source_frame_from_arrays(
                manifest,
                globals()[label_name],
                globals().get(score_name),
                "calm_ah",
            )

    file_candidates = [
        _find_newest_file(
            "trial-mlp-multiseed-average.txt"
        ),
        _find_newest_file(
            "trial-retrained-regularized-weights.txt"
        ),
        _find_newest_file(
            "trial-retrained-base-models.txt"
        ),
    ]

    for path in file_candidates:
        if path is not None:
            print("CALM-AH fallback file:", path)
            return _parse_submission_file(
                path,
                manifest,
                "calm_ah",
            )

    raise RuntimeError(
        "Cannot resolve the CALM-AH best prediction. "
        "Run the locked/multi-seed private inference first."
    )


def resolve_anchor_source(
    manifest: pd.DataFrame,
) -> pd.DataFrame:
    if "private_released_prediction" in globals():
        return _source_frame_from_arrays(
            manifest,
            globals()["private_released_prediction"],
            globals().get("private_released_score"),
            "anchor",
        )

    anchor_path = _find_newest_file(
        "trial-retrained-released-weights.txt"
    )

    if anchor_path is None:
        raise RuntimeError(
            "Cannot resolve the released-weight anchor prediction. "
            "Run Section 13 RESTORE first."
        )

    print("Anchor fallback file:", anchor_path)
    return _parse_submission_file(
        anchor_path,
        manifest,
        "anchor",
    )


def load_saved_expert_source(
    path: Path,
    manifest: pd.DataFrame,
    source_name: str,
) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"{source_name} predictions not found: {path}"
        )

    raw = pd.read_csv(path)
    validated = validate_expert_frame(
        raw,
        manifest,
        source_name,
    )

    return pd.DataFrame({
        "submission_id": (
            validated["submission_id"].astype(str)
        ),
        "label": validated["label"].astype(int),
        "probability_class_1": (
            validated["probability_class_1"].astype(float)
        ),
        "confidence": (
            validated["confidence"].astype(float)
        ),
        "source": source_name,
    })


class RoleAwareDecisionAgent:
    """
    Orchestrates four non-equivalent prediction sources.

    Decision order:
      1. Apply CALM-AH veto when primary confidence is high.
      2. If veto is inactive, allow a unanimous, sufficiently confident
         three-expert secondary coalition to override CALM-AH.
      3. Otherwise use confidence-adjusted weighted voting.
      4. Return CALM-AH on an exact weighted tie.
    """

    def __init__(
        self,
        policy: FusionPolicy,
    ) -> None:
        self.policy = policy

    def _validate_policy(
        self,
        source_names: set[str],
    ) -> None:
        expected = set(
            self.policy.source_weights
        )

        if self.policy.require_all_four_sources:
            missing = expected - source_names
            if missing:
                raise ValueError(
                    f"Missing required sources: {sorted(missing)}"
                )

        if self.policy.primary_source not in source_names:
            raise ValueError(
                "Primary source is missing: "
                f"{self.policy.primary_source}"
            )

        weights = np.asarray(
            list(self.policy.source_weights.values()),
            dtype=float,
        )

        if np.any(weights < 0) or weights.sum() <= 0:
            raise ValueError(
                "Fusion weights must be non-negative "
                "and have a positive sum."
            )

    def decide(
        self,
        manifest: pd.DataFrame,
        sources: Mapping[str, pd.DataFrame],
    ) -> pd.DataFrame:
        self._validate_policy(set(sources))

        audit = manifest[
            [
                "private_order",
                "submission_id",
                "video_id",
                "video_path",
            ]
        ].copy()
        audit["submission_id"] = (
            audit["submission_id"].astype(str)
        )

        for source_name, frame in sources.items():
            frame = frame.copy()
            frame["submission_id"] = (
                frame["submission_id"].astype(str)
            )

            renamed = frame[
                [
                    "submission_id",
                    "label",
                    "probability_class_1",
                    "confidence",
                ]
            ].rename(
                columns={
                    "label": f"{source_name}_label",
                    "probability_class_1":
                        f"{source_name}_probability",
                    "confidence":
                        f"{source_name}_confidence",
                }
            )

            audit = audit.merge(
                renamed,
                on="submission_id",
                how="left",
                validate="one_to_one",
            )

        source_names = list(
            self.policy.source_weights
        )
        secondary_names = [
            name
            for name in source_names
            if name != self.policy.primary_source
        ]

        decisions = []

        for _, row in audit.iterrows():
            primary = self.policy.primary_source
            primary_label = int(
                row[f"{primary}_label"]
            )
            primary_confidence = float(
                row[f"{primary}_confidence"]
            )

            if (
                primary_confidence
                >= self.policy.primary_veto_confidence
            ):
                final_label = primary_label
                final_score = float(
                    row[f"{primary}_probability"]
                )
                decision_rule = "calm_ah_high_confidence_veto"
                override_sources = []

            else:
                secondary_votes = []
                for name in secondary_names:
                    label = int(row[f"{name}_label"])
                    confidence = float(
                        row[f"{name}_confidence"]
                    )
                    secondary_votes.append(
                        (name, label, confidence)
                    )

                override_sources = [
                    name
                    for name, label, confidence
                    in secondary_votes
                    if (
                        label != primary_label
                        and confidence
                        >= self.policy
                        .secondary_override_min_confidence
                    )
                ]

                override_labels = {
                    label
                    for _, label, confidence
                    in secondary_votes
                    if (
                        confidence
                        >= self.policy
                        .secondary_override_min_confidence
                    )
                }

                unanimous_override = (
                    len(override_sources)
                    >= self.policy
                    .secondary_override_required
                    and len(override_labels) == 1
                    and next(iter(override_labels))
                    != primary_label
                )

                if unanimous_override:
                    final_label = next(
                        iter(override_labels)
                    )
                    final_score = float(
                        np.mean([
                            row[f"{name}_probability"]
                            for name in override_sources
                        ])
                    )
                    decision_rule = (
                        "unanimous_confident_secondary_override"
                    )
                else:
                    weighted_sum = 0.0
                    effective_weight_sum = 0.0

                    for name, base_weight in (
                        self.policy.source_weights.items()
                    ):
                        probability = float(
                            row[f"{name}_probability"]
                        )
                        confidence = float(
                            row[f"{name}_confidence"]
                        )

                        # A low-confidence source retains only half of its
                        # base weight; a fully confident source retains all.
                        effective_weight = float(
                            base_weight
                            * (0.5 + 0.5 * confidence)
                        )

                        weighted_sum += (
                            effective_weight * probability
                        )
                        effective_weight_sum += (
                            effective_weight
                        )

                    final_score = (
                        weighted_sum
                        / effective_weight_sum
                    )

                    if np.isclose(
                        final_score,
                        self.policy.final_threshold,
                        atol=1e-12,
                    ):
                        final_label = primary_label
                        decision_rule = (
                            "weighted_tie_primary_tiebreak"
                        )
                    else:
                        final_label = int(
                            final_score
                            >= self.policy.final_threshold
                        )
                        decision_rule = (
                            "confidence_adjusted_weighted_vote"
                        )

            decisions.append({
                "final_probability_class_1":
                    float(final_score),
                "final_prediction":
                    int(final_label),
                "decision_rule":
                    decision_rule,
                "primary_veto_active":
                    bool(
                        decision_rule
                        == "calm_ah_high_confidence_veto"
                    ),
                "override_sources":
                    "|".join(override_sources),
            })

        decision_frame = pd.DataFrame(decisions)
        return pd.concat(
            [
                audit.reset_index(drop=True),
                decision_frame,
            ],
            axis=1,
        )


decision_agent = RoleAwareDecisionAgent(
    FUSION_POLICY
)

print("RoleAwareDecisionAgent ready.")

In [ ]:
if RUN_FINAL_FUSION:
    calm_ah_source = resolve_calm_ah_source(
        expert_manifest
    )
    anchor_source = resolve_anchor_source(
        expert_manifest
    )
    affectgpt_source = load_saved_expert_source(
        AFFECTGPT_OUTPUT_PATH,
        expert_manifest,
        "affectgpt",
    )
    gpt56_source = load_saved_expert_source(
        GPT56_OUTPUT_PATH,
        expert_manifest,
        "gpt56_sol",
    )

    four_source_audit = decision_agent.decide(
        expert_manifest,
        {
            "calm_ah": calm_ah_source,
            "anchor": anchor_source,
            "affectgpt": affectgpt_source,
            "gpt56_sol": gpt56_source,
        },
    )

    four_source_audit.to_csv(
        FOUR_SOURCE_AUDIT_PATH,
        index=False,
    )

    with FOUR_SOURCE_SUBMISSION_PATH.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        for submission_id, prediction in zip(
            four_source_audit[
                "submission_id"
            ].astype(str),
            four_source_audit[
                "final_prediction"
            ].astype(int),
        ):
            handle.write(
                f"{submission_id},{prediction}\n"
            )

    submission_lines = [
        line
        for line in FOUR_SOURCE_SUBMISSION_PATH
        .read_text(encoding="utf-8")
        .splitlines()
        if line.strip()
    ]

    assert len(submission_lines) == len(
        expert_manifest
    ) == 152

    assert four_source_audit[
        "submission_id"
    ].tolist() == expert_manifest[
        "submission_id"
    ].astype(str).tolist()

    assert set(
        four_source_audit["final_prediction"]
    ).issubset({0, 1})

    source_comparison = pd.DataFrame({
        "source": [
            "calm_ah",
            "anchor",
            "affectgpt",
            "gpt56_sol",
            "final",
        ],
        "predicted_0": [
            int((calm_ah_source["label"] == 0).sum()),
            int((anchor_source["label"] == 0).sum()),
            int((affectgpt_source["label"] == 0).sum()),
            int((gpt56_source["label"] == 0).sum()),
            int((
                four_source_audit[
                    "final_prediction"
                ] == 0
            ).sum()),
        ],
        "predicted_1": [
            int((calm_ah_source["label"] == 1).sum()),
            int((anchor_source["label"] == 1).sum()),
            int((affectgpt_source["label"] == 1).sum()),
            int((gpt56_source["label"] == 1).sum()),
            int((
                four_source_audit[
                    "final_prediction"
                ] == 1
            ).sum()),
        ],
    })

    disagreement_summary = {
        "final_changed_from_calm_ah": int(
            (
                four_source_audit[
                    "final_prediction"
                ]
                != four_source_audit[
                    "calm_ah_label"
                ]
            ).sum()
        ),
        "primary_veto_count": int(
            four_source_audit[
                "primary_veto_active"
            ].sum()
        ),
        "secondary_override_count": int(
            (
                four_source_audit[
                    "decision_rule"
                ]
                == "unanimous_confident_secondary_override"
            ).sum()
        ),
        "weighted_vote_count": int(
            four_source_audit[
                "decision_rule"
            ].str.contains(
                "weighted",
                regex=False,
            ).sum()
        ),
    }

    print("Final submission:", FOUR_SOURCE_SUBMISSION_PATH)
    print("Decision audit:", FOUR_SOURCE_AUDIT_PATH)
    print("Decision summary:", disagreement_summary)
    display(source_comparison)
    display(
        four_source_audit[
            [
                "submission_id",
                "calm_ah_label",
                "calm_ah_confidence",
                "anchor_label",
                "affectgpt_label",
                "affectgpt_confidence",
                "gpt56_sol_label",
                "gpt56_sol_confidence",
                "final_prediction",
                "decision_rule",
            ]
        ].head(20)
    )
else:
    print(
        "Final four-source fusion is disabled. Complete both expert "
        "prediction files, inspect them, then set RUN_FINAL_FUSION=1."
    )